<a id="nb05-toc"></a>

# Notebook 05 — Hotspot & Cold Spot Analysis
## Surface Urban Thermal Environment Assessment — Nigerian Cities

This notebook implements a six-step analytical pipeline to identify, delineate, validate,
enrich, visualise and aggregate statistically significant Urban Heat Island (UHI) hotspots
and cold spots across 56 Nigerian cities. It is the core analytical stage of the broader
Surface LST Modelling workflow and feeds downstream notebooks (NB06 — comparative
statistics, NB07 — reporting).

**Version:** `NB05_v5_SMOD_r5`
**Upstream inputs:**
- `NB01_v5_SMOD` → zone geometries (Urban Core, Background Reference Area, City-Region)
  for all 56 ABOVE_300K cities
- `NB04` → median SEVIRI LST rasters (day / night, per city, ESRI:102022 @ 1 km) + SUHII
  lookup

**Spatial reference:**
- All spatial operations: **ESRI:102022** (Africa Albers Equal Area Conic, metres)
- Storage / export: **EPSG:4326** (WGS84 geographic, degrees)

**Cache strategy:** heavy computations (Gi★ masks §5, KDE polygons §6, enriched GDF §7)
are cached as `.npz` / `.geojson` files keyed by `SOURCE_VERSION` + geometry hash.
Set `FORCE_RERUN = True` to bypass all caches.

---

## Table of Contents

| § | Title | Key output |
|---|-------|-----------|
| [§1](#nb05-s1) | Setup & Configuration | `SOURCE_VERSION`, cache helpers |
| [§2a](#nb05-s2a) | Data Loading — AOI + Cluster25 | zone geometries, cluster lookup |
| [§2b](#nb05-s2b) | SEVIRI Raster Inventory | per-city LST raster paths |
| [§2c](#nb05-s2c) | Stakeholder & Population Data | GPS points (ABOVE_300K filter), GHS-POP 100 m cache |
| [§3](#nb05-s3) | P90 / P10 Thresholds | city-specific percentile cutoffs |
| [§4](#nb05-s4) | Getis-Ord Gi★ + BH-FDR | spatial autocorrelation significance |
| [§5](#nb05-s5) | Validated Hot & Cold Spot Masks | NPZ mask cache |
| [§5b](#nb05-s5b) | Validated Pixel Counts per City | BH-flagged summary table |
| [§6](#nb05-s6) | Two-Tier KDE Delineation | KDE polygon records |
| [§6b](#nb05-s6b) | Contour Completeness Diagnostic | tier classification table |
| [§6c](#nb05-s6c) | Comprehensive Spot Summary | `spot_comprehensive_summary.csv` |
| [§7](#nb05-s7) | Per-spot layer and enrichment | `gdf` (one row per spot) · `all_cities_spots.geojson` |
| [§8](#nb05-s8) | Distribution assessment | not performed in this notebook (see the note) |
| [§9](#nb05-s9) | Stakeholder hit rates and buffer sensitivity | `HITRATE_NB05` · charts · national / cluster / city tables · no file |
| [§10](#nb05-s10) | NB05 working GeoPackage, per contour level | `master_thermal_spots.gpkg` (not the published dataset) |
| [§11](#nb05-s11) | Visualisations (a–f) | SVG / PDF figures |
| [§11a2](#nb05-s11a2) | Publication A4 Grid | 56-city grid figure |
| [§12](#nb05-s12) | Aggregation Hierarchy & Tables | city / cluster / national CSVs |
| [§12b](#nb05-s12b) | Descriptive Statistical Charts | bar / box charts |
| [§13](#nb05-s13) | Summary and carry-forward to NB06 | completion table · NB06 inputs |
| [§14](#nb05-s14) | **Build the dataset** (last section) | [14a](#nb05-s14a) build · [14b](#nb05-s14b) summary of the whole dataset |


<a id="nb05-s1"></a>

## §1 — Setup & Configuration

### Design rationale

Three configuration switches govern the entire notebook run:

| Variable | Default | Effect |
|----------|---------|--------|
| `SOURCE_VERSION` | `NB05_v5_SMOD_r5` | Cache key — bump when methodology changes to force full rebuild |
| `FORCE_RERUN` | `False` | If `True`, bypasses every cache and recomputes from scratch |
| `TIER2_BUFFER_M` | `500` m | Radius of Tier 2 KDE spot geometry (500 m circle per SEVIRI pixel) |

### CRS choices

**ESRI:102022 (Africa Albers Equal Area Conic)** is used for all spatial operations (buffering, area calculation, KDE) because equal-area projection ensures that polygon areas in km² are directly comparable across the country. Nigeria spans roughly 3°E–15°E, 4°N–14°N; Albers distortion in this zone is < 0.1%.

**EPSG:4326 (WGS84)** is used only for file storage and map rendering where geographic coordinates are required.

### Cache helpers

`cache_valid(path, version, geom_hash)` returns `True` if a cached output file exists and was generated with the same `SOURCE_VERSION` and AOI geometry hash. This prevents stale results when zone geometries are updated upstream (NB01) without requiring a full rerun of every cell.

> Source version `NB05_v5_SMOD`; every spatial operation runs in ESRI:102022.
> `FORCE_RERUN = True` bypasses all caches and recomputes from scratch.

[↑ Table of Contents](#nb05-toc)


In [ ]:
import os, re, sys, json, hashlib, importlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import xy as rio_xy
from rasterio.mask import mask as rio_mask
from rasterio.transform import Affine as RioAffine
from rasterio.features import shapes as rio_shapes
import scipy.stats
from scipy.stats import gaussian_kde
from skimage import measure as sk_measure
from shapely.geometry import shape, Point, Polygon, MultiPolygon, mapping
from shapely.ops import unary_union
import fiona
import libpysal
from esda.getisord import G_Local
from statsmodels.stats.multitest import multipletests
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

ROOT = Path(os.getcwd())
if ROOT.name != 'stake_lst_modelling':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import config
from config import CLUSTER25_COLORS
importlib.reload(config)

RESULTS_DIR = ROOT / 'Results' / '05_hotspot_analysis'
AOI_DIR     = ROOT / 'Results' / '01_aoi_delineation'
SEVIRI_DIR  = ROOT / 'Results' / '04_temporal_analysis_daily' / 'seviri'
STAKE_DIR   = ROOT / 'data' / 'inputs' / 'stakeholder_survey'
FIGS_DIR    = RESULTS_DIR / 'figures'

for d in [RESULTS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Version ────────────────────────────────────────────────────────────────────
SOURCE_VERSION   = 'NB05_v5_SMOD_r5'
FORCE_RERUN      = False   # Set True to bypass ALL caches (Gi★, KDE, GDF)
CLEAR_GDF_CACHE  = False   # Set True to delete only the GDF/master-GDF cache
                           # (cheaper than FORCE_RERUN — keeps Gi★ & KDE caches)



# ── Clear GDF cache if requested ─────────────────────────────────────────────
if CLEAR_GDF_CACHE or FORCE_RERUN:
    import shutil as _shutil
    _gdf_files = [
        RESULTS_DIR / 'all_cities_spots.geojson',
        RESULTS_DIR / 'all_cities_spots.csv',
        RESULTS_DIR / 'all_cities_spots.meta.json',
        RESULTS_DIR / 'master_thermal_spots.gpkg',
    ]
    _deleted = []
    for _gf in _gdf_files:
        if _gf.exists():
            _gf.unlink()
            _deleted.append(_gf.name)
    if _deleted:
        print(f'GDF cache cleared: {_deleted}')
    else:
        print('GDF cache: nothing to clear (already empty)')

# ── Read params from config (with sensible defaults if not yet present) ────────
P_HOT    = getattr(config, 'HOTSPOT_PERCENTILE_HOT',  90)
P_COLD   = getattr(config, 'HOTSPOT_PERCENTILE_COLD', 10)
FDR_ALPHA        = getattr(config, 'HOTSPOT_FDR_ALPHA',   0.05)
KDE_WEIGHTED     = getattr(config, 'KDE_WEIGHTED',         True)
KDE_CONTOURS     = getattr(config, 'KDE_CONTOUR_LEVELS',   [0.50, 0.75, 0.90])
MIN_PIXELS_TIER1 = getattr(config, 'MIN_PIXELS_TIER1',     10)
TIER2_BUFFER_M   = getattr(config, 'KDE_TIER2_BUFFER_M',   500)
DP_TOL_M         = getattr(config, 'KDE_DP_TOLERANCE_M',   50)
# Stakeholder buffer — controlled via config.STAKEHOLDER_VALIDATION_BUFFER_M
STAKE_BUFFER_M   = getattr(config, 'STAKEHOLDER_VALIDATION_BUFFER_M', 1_000)
STAKE_BUFFER_OPTIONS = getattr(config, 'STAKEHOLDER_VALIDATION_BUFFER_OPTIONS_M',
                                list(range(500, 10_500, 500)))
PERIODS = ['day', 'night']

# ── Cache helpers ──────────────────────────────────────────────────────────────
def _geom_hash(wkt: str) -> str:
    return hashlib.sha256(wkt.encode()).hexdigest()[:16]

def cache_valid(out_path: Path, geom_hash: str) -> bool:
    if FORCE_RERUN or not out_path.exists():
        return False
    meta_path = out_path.with_suffix('').with_suffix('.meta.json') \
                if out_path.suffix == '.npz' else out_path.with_suffix('.meta.json')
    try:
        m = json.loads(meta_path.read_text())
        return m.get('source_version') == SOURCE_VERSION and m.get('geom_hash') == geom_hash
    except Exception:
        return False

def save_meta(out_path: Path, geom_hash: str, extra: dict = None):
    meta_path = out_path.with_suffix('').with_suffix('.meta.json') \
                if out_path.suffix == '.npz' else out_path.with_suffix('.meta.json')
    m = {'source_version': SOURCE_VERSION, 'geom_hash': geom_hash}
    if extra:
        m.update(extra)
    meta_path.write_text(json.dumps(m, indent=2))

print(f"ROOT              : {ROOT}")
print(f"SOURCE_VERSION    : {SOURCE_VERSION}")
print(f"P_HOT/P_COLD      : {P_HOT}/{P_COLD}  |  FDR α={FDR_ALPHA}")
print(f"KDE weighted      : {KDE_WEIGHTED}  |  Contours: {KDE_CONTOURS}")
print(f"Stake buffer      : {STAKE_BUFFER_M} m  (options: {STAKE_BUFFER_OPTIONS[0]}–{STAKE_BUFFER_OPTIONS[-1]} m)")
print(f"STAKE_DIR exists  : {STAKE_DIR.exists()}  ({STAKE_DIR.relative_to(ROOT)})")


In [ ]:
# ── Load SUHII computed in NB04 §10d ─────────────────────────────────────────
# File: Results/04_temporal_analysis_daily/seviri/City_SEVIRI_3zone_SUHII.csv
# Columns: city, period, start_date, end_date, uc_lst_mean, bra_lst_mean,
#          cr_lst_mean, suhii, tier, note
#
# Fallback candidates listed in order of priority — first match wins.
_suhii_candidates = [
    SEVIRI_DIR / 'City_SEVIRI_3zone_SUHII.csv',                          # primary (NB04 §10d output)
    SEVIRI_DIR / 'city_seviri_3zone_suhii.csv',                          # lowercase variant
    ROOT / 'Results' / '04_temporal_analysis_daily' / 'City_SEVIRI_3zone_SUHII.csv',
    SEVIRI_DIR / 'suhii_city_summary.csv',                               # legacy name
]
suhii_df = None
for _p in _suhii_candidates:
    if _p.exists():
        suhii_df = pd.read_csv(_p)
        print(f"SUHII loaded from: {_p.relative_to(ROOT)}")
        print(f"  Shape: {suhii_df.shape}  |  Columns: {list(suhii_df.columns)}")
        print(suhii_df.head(3).to_string(index=False))
        break
if suhii_df is None:
    print("⚠ SUHII CSV not found — suhii column will be NaN in §7.")
    print("  Expected: Results/04_temporal_analysis_daily/seviri/City_SEVIRI_3zone_SUHII.csv")
    print("  Re-run NB04 §10d to generate it.")

# Build lookup: city → period → suhii value
suhii_lookup = {}
if suhii_df is not None:
    for _, row in suhii_df.iterrows():
        c = str(row.get('city', row.get('city_name', ''))).strip()
        p = str(row.get('period', 'day')).strip().lower()
        v = float(row.get('suhii', row.get('SUHII', np.nan)))
        suhii_lookup.setdefault(c, {})[p] = v
    print(f"\nSUHII lookup: {len(suhii_lookup)} cities  ×  {set(p for d in suhii_lookup.values() for p in d)} periods")

<a id="nb05-s2a"></a>

## §2a — Data Loading: AOI + Cluster25

### Study area

The analysis covers **56 Nigerian cities** spanning all six geopolitical zones and a wide
range of climatic contexts — from the Sahel (Köppen BSh, cluster C1) to humid forest
(Köppen Af/Am, cluster C12+). This diversity is the primary motivation for city-specific
thresholds (§3) rather than a single national cutoff.

### Zone definitions

Three nested spatial zones are loaded for each city from NB01 outputs:

| Zone | Abbreviation | Role in this notebook |
|------|-------------|----------------------|
| Urban Core | UC | Primary analysis boundary; all LST sampling, Gi★ and KDE operations are restricted to UC pixels |
| Background Reference Area | BRA | SUHII reference ring — the peri-urban buffer whose mean LST provides the background temperature against which SUHII is measured (UC_LST − BRA_LST); also used to classify spots near the UC/BRA boundary |
| City-Region | CR | Broader administrative boundary used as the spatial extent for P90/P10 threshold derivation and zone area metrics |

### Cluster25

Cities are grouped into **25 physiographic landscape clusters** derived from k-means on
terrain, vegetation, and climate variables (details in NB02). Cluster membership drives
colour schemes throughout §11–§12 and is used as the primary stratification variable in
aggregation tables (§12).

> **Why UC and BRA?**
> - **UC** — tight spatial boundary concentrating the urban heat signal; avoids diluting the KDE with peri-urban pixels.
> - **BRA** — Background Reference Area: the suburban ring used as the thermal baseline for SUHII computation.  Spots whose centroid falls in the BRA rather than the UC are flagged (`zone = 'bra'`) and retained — they represent genuine urban-fringe heat anomalies.

### Computational role of BRA in NB05

| Usage | Where | What |
|-------|-------|------|
| Zone classification | §6 (`classify_zone`) | Tags each KDE polygon as `urban_core`, `bra`, or `both` based on centroid position |
| Area metric | §7 GDF | `bra_area_km2` column — BRA polygon area in km² per city |
| SUHII reference | §7 GDF (NB04 input) | `suhii` = UC mean LST − BRA mean LST, computed in NB04 |
| Export | §10 GeoPackage | `bra` zone polygon included in `city_zones` layer |

[↑ Table of Contents](#nb05-toc)


In [ ]:
cluster_df      = pd.read_csv(AOI_DIR / 'city_cluster25_lookup.csv')
city_to_cluster = dict(zip(cluster_df['city_name'], cluster_df['Cluster25']))

# ── Normalisation helper ──────────────────────────────────────────────────────
# AOI folders sometimes use underscores (Ado_Ekiti, Port_Harcourt) while the
# cluster CSV, SEVIRI folders, and SUHII CSV use spaces (Ado Ekiti, Port Harcourt).
# We normalise all city keys to their display form (spaces, hyphens preserved).
def _display(folder_name: str) -> str:
    return folder_name.replace('_', ' ')

# ── Discover cities from AOI folder structure ─────────────────────────────────
# Pattern: Results/01_aoi_delineation/{city_folder}/{city_slug}_urban_core_102022_v5.geojson
uc_files = sorted(AOI_DIR.glob('*/*_urban_core_102022_v5.geojson'))
# Use display names as canonical keys throughout NB05
cities   = [_display(f.parent.name) for f in uc_files]
# Keep a map: display name → actual folder name (for file path construction)
_folder  = {_display(f.parent.name): f.parent.name for f in uc_files}
print(f"Cities with zone GeoJSONs: {len(cities)}")

def _zone_path(city_display: str, zone: str, crs: str = '102022') -> Path:
    """Return zone GeoJSON path, handling underscore vs space folder names."""
    folder = _folder.get(city_display, city_display)
    slug   = folder.replace(' ', '_')   # file stem always uses underscores
    for suffix in ('_v5', ''):
        p = AOI_DIR / folder / f'{slug}_{zone}_{crs}{suffix}.geojson'
        if p.exists():
            return p
    return None

zones_dict = {}
for city in cities:
    try:
        uc_p  = _zone_path(city, 'urban_core')
        bra_p = _zone_path(city, 'bra')
        cr_p  = _zone_path(city, 'city_region')
        if not (uc_p and bra_p and cr_p):
            print(f"  ⚠ {city}: missing zone file(s)  (uc={uc_p}, bra={bra_p}, cr={cr_p})")
            continue
        uc  = gpd.read_file(uc_p).geometry.iloc[0]
        bra = gpd.read_file(bra_p).geometry.iloc[0]
        cr  = gpd.read_file(cr_p).geometry.iloc[0]
        zones_dict[city] = {'uc': uc, 'bra': bra, 'cr': cr,
                            'hash': _geom_hash(uc.wkt)}
    except Exception as e:
        print(f"  ⚠ {city}: {e}")

print(f"Zone geometries loaded: {len(zones_dict)}")
# Spot-check the known problem cities
for c in ['Port Harcourt', 'Ado Ekiti', 'Benin City', 'Ijebu-Ode']:
    status = '✓' if c in zones_dict else '✗ MISSING'
    print(f"  {status}  {c}")

<a id="nb05-s2b"></a>

## §2b — SEVIRI Raster Inventory

### Sensor background

SEVIRI (Spinning Enhanced Visible and InfraRed Imager) aboard Meteosat provides Land Surface Temperature retrievals every 15 minutes at a native spatial resolution of ~3 km at nadir over West Africa. NB04 pre-processed the archive into per-city median composites at 1 km (reprojected ESRI:102022) for two diurnal periods:

- **Day** — LST retrievals between 10:00–14:00 UTC (peak solar heating)
- **Night** — LST retrievals between 00:00–04:00 UTC (minimal solar input, captures nocturnal UHI)

### Temporal composite

Median composites are used rather than means to reduce sensitivity to cloud contamination and anomalous hot/cold events. The result is a single representative 1 km LST raster per city × period capturing the climatological thermal structure.

### Raster naming convention

`{city}_{period}_{start}_{end}_median_102022_1km_clipped.tif`

**Pattern:** `{city}_SEVIRI_{period}_{start}_{end}_median_102022_1km_clipped.tif`
**CRS:** Already ESRI:102022 @ 1 km pixel spacing.

[↑ Table of Contents](#nb05-toc)


In [ ]:
seviri_inv = {}
for city in cities:
    city_dir = SEVIRI_DIR / city
    if not city_dir.exists():
        continue
    for period in PERIODS:
        tifs = sorted(city_dir.glob(f'{city}_SEVIRI_{period}_*_median_102022_1km_clipped.tif'))
        if tifs:
            seviri_inv.setdefault(city, {})[period] = tifs[-1]

cities_with_sev = sorted(seviri_inv.keys())
print(f"Cities with SEVIRI data: {len(cities_with_sev)}")
for c in cities_with_sev[:3]:
    for p, f in seviri_inv[c].items():
        print(f"  {c} [{p}]: {f.name}")


<a id="nb05-s2c"></a>

## §2c — Stakeholder & Population Data

### Stakeholder survey (§9 validation)

Field GPS coordinates were collected from local informants in each city, identifying locations
perceived as heat/cold anomalies. This dataset serves as **independent spatial validation** —
if our model-derived hotspots are meaningful, a high proportion of stakeholder GPS points
should fall within a reasonable buffer of a computed hotspot polygon.

#### ABOVE_300K city filter

GPS records are filtered to respondents whose reported workshop city (`matched_city_y`)
belongs to the 56 ABOVE_300K study cities. Points associated with cities outside the
analysis extent are dropped and a drop-count is printed. This ensures that hit-rate
statistics (§9) are not diluted by off-study-area records.

> **Note — the hit rate uses the cohort, and the released coordinates.**
> The filter described above keys on `matched_city_y`, so the raw point set here is 97.
> §9 no longer uses it. `hitrate_sensitivity` is derived during the build by
> `scripts/compute_hitrate.py` from the two released layers — `spots` and the 94
> cohort points in `stakeholder_spots_gps` (55 hot / 39 cold), whose coordinates are
> rounded to three decimal degrees — so the published hit rates can be reproduced
> from the dataset alone. §9 below reads that layer and displays it.


#### Deduplication

Unique combinations of `(pid, feature_spot, matched_city_y, lat_spot, long_spot)` are
retained. A respondent who identified both `hotspot1` and `hotspot2` keeps both records
(different locations); duplicate rows from the fan-out structure of the master CSV
(one row per city × feature) are collapsed.

#### Day + Night validation

Hit-rate validation is computed separately for **Day** and **Night** spot polygons,
matching the diurnal structure of the SEVIRI analysis. Results are saved as
`stakeholder_hitrate_{buf}m_day.csv` and `stakeholder_hitrate_{buf}m_night.csv`.

The hit-rate is computed at multiple buffer distances (500 m – 10 km in 500 m steps)
to produce a sensitivity curve per period, revealing whether model/stakeholder agreement
is tighter in daytime or nocturnal spot locations.

### Population data (§6 + §11d)

**Source:** JRC/GHSL Global Human Settlement Population (GHS-POP) P2023A, epoch selected by
`config.get_closest_ghsl_year(PROJECT_STUDY_YEAR)` — 2024 resolves to **2025**.

Population is **summed on GHS-POP's own 100 m grid**, server-side, one polygon at a time; it is
*not* aggregated to the 1 km SEVIRI grid and no raster is downloaded. Values are read from
`Results/05_hotspot_analysis/population_100m/` (`contours/` for spots, `zones/` for UC/BRA/CR),
written by `scripts/extract_population_100m.py`.

> **Why not 1 km.** Matching the SEVIRI pixel size was the earlier approach and it was wrong:
> the 1 km raster had been resampled from 100 m without an additive reducer, so each 1 km cell
> held a single 100 m value instead of the sum of the ~100 beneath it. Population came out about
> **100× too low** (Jos city region 15,970 against a true 1,550,542). Re-doing the 1 km
> aggregation correctly did not rescue it either — at 1 km the pixel-centre rule left only 3 of
> 10 Jos polygons within 5% of the truth. See the README beside the cache.

> **Denominator.** Use the **city region** for exposure shares. Spots do not respect the
> urban-core boundary: of the 185 L50 footprints only 16 lie wholly inside the UC, 75 wholly
> inside the BRA, 36 straddle the two and 8 sit mostly in neither.

Population data is used in two ways:
1. **Per-spot population exposure** — summing population within each KDE contour polygon
   (L50 / L75 / L90) to quantify the number of residents exposed to statistically
   significant heat/cold stress.
2. **Half-donut charts (§11d)** — Ring 3 of each donut encodes population within each
   contour zone, giving an immediate visual comparison of exposure intensity across
   cities and clusters.

[↑ Table of Contents](#nb05-toc)


In [ ]:
# ── Stakeholder ──────────────────────────────────────────────────────────────
# Source: data/inputs/stakeholder_survey/master_composite.csv  (same file as NB02)
#
# Row structure: each row = one (PID x matched_city_x x feature_spot) combination.
# A single PID fans out into many rows because matched_city_x repeats for every
# registered city AND feature_spot repeats for each identified feature.
#
# Column semantics (confirmed from NB02):
#   matched_city_x     -- city where respondent registered (multiple per PID)
#   matched_city_y     -- city where respondent ATTENDED the workshop
#   survey status      -- Yes = attended; No = registered only
#   feature_spot       -- hotspot1/2, coolspot1/2, veg_specie_1/2
#   lat_spot/long_spot -- GPS of that specific feature
#   61_temp_increase_10yrs   -- Q61 warming perception
#   60_diurnal_severity_heat -- Q60 diurnal heat severity
#
# Deduplication rule for S9 GPS validation:
#   unique (pid, feature_spot, matched_city_y, lat_spot, long_spot)
#   A PID that marked hotspot1 AND hotspot2 keeps both (different locations).

stake_files = sorted(STAKE_DIR.glob("*.csv"))
if stake_files:
    stake_df    = pd.read_csv(stake_files[0])
    workshop_df = stake_df[stake_df["survey status"] == "Yes"].copy()

    _has_gps   = workshop_df.dropna(subset=["lat_spot", "long_spot", "matched_city_y"]).copy()
    _dedup_key = ["pid", "feature_spot", "matched_city_y", "lat_spot", "long_spot"]
    _gps_deduped = _has_gps.drop_duplicates(subset=_dedup_key).copy()

    hotspot_gps_df  = _gps_deduped[_gps_deduped["feature_spot"].isin(["hotspot1","hotspot2"])].copy()
    coldspot_gps_df = _gps_deduped[_gps_deduped["feature_spot"].isin(["coolspot1","coolspot2"])].copy()
    # Filter to ABOVE_300K study cities only (workshop attendees may have registered
    # in cities outside the analysis extent; those GPS points are excluded)
    _before_hot  = len(hotspot_gps_df)
    _before_cold = len(coldspot_gps_df)
    hotspot_gps_df  = hotspot_gps_df[hotspot_gps_df["matched_city_y"].isin(cities)].copy()
    coldspot_gps_df = coldspot_gps_df[coldspot_gps_df["matched_city_y"].isin(cities)].copy()
    print(f"    → ABOVE_300K study-city filter: "
          f"hot {_before_hot}→{len(hotspot_gps_df)} "
          f"(dropped {_before_hot - len(hotspot_gps_df)}), "
          f"cold {_before_cold}→{len(coldspot_gps_df)} "
          f"(dropped {_before_cold - len(coldspot_gps_df)})")

    print(f"Stakeholder CSV : {stake_files[0].name}  ({len(stake_df)} rows total)")
    print(f"  Workshop attendees (survey status=Yes) : "
          f"{workshop_df['pid'].nunique()} unique PIDs  ({len(workshop_df)} rows)")
    print()
    print("  GPS validation sets (after dedup on pid x feature x city x lat/lon):")
    print(f"    Hotspot  GPS : {len(hotspot_gps_df):3d} points "
          f"({hotspot_gps_df['pid'].nunique()} PIDs, "
          f"{hotspot_gps_df['matched_city_y'].nunique()} cities)  "
          f"[hotspot1={(hotspot_gps_df['feature_spot']=='hotspot1').sum()}, "
          f"hotspot2={(hotspot_gps_df['feature_spot']=='hotspot2').sum()}]")
    print(f"    Coldspot GPS : {len(coldspot_gps_df):3d} points "
          f"({coldspot_gps_df['pid'].nunique()} PIDs, "
          f"{coldspot_gps_df['matched_city_y'].nunique()} cities)  "
          f"[coolspot1={(coldspot_gps_df['feature_spot']=='coolspot1').sum()}, "
          f"coolspot2={(coldspot_gps_df['feature_spot']=='coolspot2').sum()}]")
    print()
    print(f"  Q61 warming values : {stake_df['61_temp_increase_10yrs'].dropna().unique().tolist()}")
    print(f"  Q60 diurnal values : {stake_df['60_diurnal_severity_heat'].dropna().unique().tolist()}")
else:
    stake_df = workshop_df = hotspot_gps_df = coldspot_gps_df = pd.DataFrame()
    print(f"No stakeholder CSV found in {STAKE_DIR.relative_to(ROOT)}")
    print("  S9 validation and S11f perception charts will be skipped")

# ── Population: GHS-POP summed at its native 100 m grid ──────────────────────
# Read from the cache written by scripts/extract_population_100m.py, keyed by
# contour_id ({spot_id}_L50 / _L75 / _L90) and by {city_slug}_{zone}.
#
# The previous approach summed a 1 km raster that had been resampled from 100 m
# without an additive reducer, so each 1 km cell held ONE 100 m cell value instead
# of the sum of the ~100 beneath it. That understated population by about 100x
# (Jos city region: 15,970 against 1,550,542; Lagos urban core 102x short of its
# WUP population). Re-extracting at 1 km with a correct sum was tested and rejected
# too: the pixel-centre rule left only 3 of 10 Jos polygons within 5% of truth.
# See Results/05_hotspot_analysis/population_100m/README.md.
POP100M_DIR = RESULTS_DIR / 'population_100m'

# Scope to the CURRENT run. The cache persists between runs, so if config.CITY_SELECTION_MODE
# narrows and NB01 is re-run, rows for retired cities are still on disk; they are ignored here
# rather than silently read back in.
_active_cities = set(
    pd.read_csv(ROOT / 'Results' / '01_aoi_delineation' / 'study_cities_final.csv')
      .query("status == 'included'")['city_name'].astype(str))

def _load_pop_cache(sub):
    files = sorted((POP100M_DIR / sub).glob('*.csv'))
    if not files:
        raise FileNotFoundError(
            f"No {sub} population cache in {(POP100M_DIR / sub).relative_to(ROOT)}. "
            f"Run: python scripts/extract_population_100m.py --kind {sub}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    n_all = len(df)
    df = df[df['city'].astype(str).isin(_active_cities)]
    if len(df) < n_all:
        print(f"  {sub}: ignored {n_all - len(df)} row(s) from cities outside the "
              f"current selection")
    return df

_con_pop = _load_pop_cache('contours')   # KDE contours -- the exposure numbers
_zon_pop = _load_pop_cache('zones')      # UC / BRA / CR -- denominators
POP100M  = _con_pop.set_index('key')['population_100m'].astype(float).to_dict()
POPZONE  = _zon_pop.set_index('key')['population_100m'].astype(float).to_dict()
print(f"Population: {len(POP100M):,} contours + {len(POPZONE):,} zones "
      f"(GHS-POP {int(_con_pop['ghs_epoch'].iloc[0])}, native 100 m, "
      f"{len(_active_cities)} cities in scope)")

<a id="nb05-s3"></a>

## §3 — P90 / P10 Thresholds

### Rationale for percentile-based thresholds

Absolute temperature cutoffs (e.g., LST > 40 °C) would systematically miss hot spots in cooler northern cities while flagging nearly the entire urban area in southern cities during the hot season.  City-specific percentile thresholds normalise for background climate, isolating the *relative* thermal anomaly — pixels that are anomalously hot (or cold) compared to the broader city-region context.

### Threshold definition

$$P_{q}(c,\,p) = \operatorname{quantile}_{q}\!\bigl\{\,x_i \mid i \in \Omega_c^{\mathrm{CR}},\; x_i \text{ valid}\,\bigr\}$$

| Symbol | Meaning |
|--------|---------|
| $P_{q}(c,p)$ | City-specific $q$-th percentile LST threshold for city $c$, period $p$ |
| $x_i$ | LST value (°C) at pixel $i$ |
| $\Omega_c^{\mathrm{CR}}$ | Set of all valid pixels within the City-Region boundary of city $c$ |
| $q$ | Percentile level: **90** for hot threshold, **10** for cold threshold |
| $p$ | Diurnal period: `day` or `night` |

**Hot pixel criterion:** $x_i \ge P_{90}(c,p)$  
**Cold pixel criterion:** $x_i \le P_{10}(c,p)$

| Threshold | Spot type | Criterion |
|-----------|-----------|----------|
| $P_{90}$ (90th percentile) | Hot | LST $\ge P_{90}$ within City-Region |
| $P_{10}$ (10th percentile) | Cold | LST $\le P_{10}$ within City-Region |

### Why City-Region (not Urban Core) for threshold derivation?

Using the City-Region as the reference population ensures the threshold captures the full thermal gradient from rural to urban, which is where the UHI signal manifests. A UC-only threshold would be biased upward by the dense urban heat mass, artificially raising the bar for hot-spot qualification.

### Day vs Night thresholds

Thresholds are derived separately for Day and Night composites because the diurnal thermal cycle produces fundamentally different distributions.  Daytime P90 is driven by solar heating of impervious surfaces; night-time P90 reflects stored heat release, producing a distinct spatial pattern that must be characterised independently.

> **Note:** These P90/P10 masks are a *necessary but not sufficient* condition for hotspot status.  The Gi★ spatial clustering criterion (§4) and KDE delineation (§6) provide additional filters.

[↑ Table of Contents](#nb05-toc)


In [ ]:
def load_seviri_arr(tif_path):
    # SEVIRI median composites from NB04 are stored in Celsius (approx 20-60 C).
    # nodata is NaN (set by NB04 export). No Kelvin guard needed.
    with rasterio.open(tif_path) as src:
        arr = src.read(1).astype(np.float32)
        nd  = src.nodata
        tf  = src.transform
        crs = src.crs
    if nd is not None:
        arr[arr == nd] = np.nan
    # Mask only physically impossible Celsius values (fill values, not valid LST)
    arr[(arr < -50) | (arr > 100)] = np.nan
    return arr, tf, crs

thresholds = {}
for city in cities_with_sev:
    thresholds[city] = {}
    for period, tif in seviri_inv[city].items():
        arr, *_ = load_seviri_arr(tif)
        valid = arr[np.isfinite(arr)]
        thresholds[city][period] = {
            'p90': float(np.percentile(valid, P_HOT))  if valid.size > 0 else np.nan,
            'p10': float(np.percentile(valid, P_COLD)) if valid.size > 0 else np.nan,
            'n':   int(valid.size),
        }

# thr_df summarises thresholds across all 56 cities x 2 periods (112 rows).
# p90/p10 = per-city threshold values (single scalar each); n = valid pixel count.
# describe() shows how thresholds vary across cities -- std is cross-city spread.
thr_df = pd.DataFrame([{'city': c, 'period': p, **v}
                        for c, pd_ in thresholds.items() for p, v in pd_.items()])
print(f"Thresholds computed for {len(thr_df)} city-period pairs")
print(thr_df[['p90','p10','n']].describe().round(2))
print()
print("Sample (first 8 rows):")
print(thr_df.head(8).to_string(index=False))

<a id="nb05-s4"></a>

## §4 — Getis-Ord Gi★ + Benjamini-Hochberg FDR Correction

### Why Getis-Ord Gi★?

The Gi★ statistic tests whether a pixel's value — and the values of its neighbours — together constitute a statistically significant spatial cluster, rather than testing each pixel in isolation.  This is critical for remote-sensing data where spatial autocorrelation is strong: adjacent LST pixels are correlated by heat-diffusion physics, so conventional i.i.d. significance tests would massively inflate false-positive rates.

### Gi★ statistic

$$G_i^{*} = \frac{\displaystyle\sum_{j=1}^{n} w_{ij}\,x_j \;-\; \bar{x}\sum_{j=1}^{n} w_{ij}}{S\,\sqrt{\dfrac{n\sum_{j} w_{ij}^{2} - \Bigl(\sum_{j} w_{ij}\Bigr)^{2}}{n-1}}}$$

where the global mean and standard deviation are:

$$\bar{x} = \frac{1}{n}\sum_{j=1}^{n} x_j\qquad S = \sqrt{\frac{1}{n}\sum_{j=1}^{n} x_j^2 - \bar{x}^{\,2}}$$

| Symbol | Meaning |
|--------|---------|
| $G_i^{*}$ | Local Gi★ statistic at pixel $i$; asymptotically distributed as a standard normal Z-score |
| $x_j$ | LST value at pixel $j$ |
| $w_{ij}$ | Spatial weight (Queen contiguity: 1 if $j$ is a neighbour of $i$, 0 otherwise; row-standardised) |
| $\bar{x}$ | Global mean LST across all $n$ valid pixels in the city |
| $S$ | Global standard deviation of LST |
| $n$ | Total number of valid pixels in the Urban Core |

A **positive** $G_i^{*}$ with $|Z| \ge 1.96$ indicates a statistically significant hot cluster; a **negative** value with $|Z| \ge 1.96$ indicates a cold cluster.

### Queen contiguity weights

Queen contiguity (8-neighbours) was chosen over Rook (4-neighbours) because diagonal neighbours share a corner and still experience physically meaningful thermal influence at 1 km pixel spacing.  The weight matrix is row-standardised (`w.transform = 'R'`) so Gi★ Z-scores are comparable across cities with different pixel counts.

### Significance threshold: Z ≥ ±1.96

The Z ≥ 1.96 threshold (two-tailed 95% confidence) is the established convention in spatial hotspot literature (ArcGIS Spatial Statistics, GeoDa).  With $n \approx$ 1,000–9,000 pixels per city, the raw Z > 1.96 criterion is used directly for all downstream spatial analysis (§6+).

### Benjamini-Hochberg FDR correction (stored, not applied)

> **Why I store BH-FDR but do not apply it.** BH assumes independent tests, but raster pixels are spatially autocorrelated: BH sets $\alpha/n \approx 0.00005$ (Z > 4) and misses real clusters in 50 of 56 cities. I use Z ≥ 1.96 for all spatial analysis from §6 on, and store the BH flag (`bh_significant`) to mark the clusters that survive the stricter correction (★ in §5b) for sensitivity reporting.

[↑ Table of Contents](#nb05-toc)


In [ ]:
from scipy.stats import norm as sp_norm
from libpysal.weights import W as libW

# BH-FDR is stored, not applied (see the §4 note):
# BH assumes independent tests; raster pixels are spatially autocorrelated —
# with n ≈ 1 000–9 000 pixels the BH threshold (α/n ≈ 0.000 05) requires Z > 4
# and misses real clusters in 50/56 cities (standard GIS practice: ArcGIS
# Getis-Ord, GeoDa both use |Z| ≥ 1.96 directly without FDR correction).
# RESOLUTION: Z ≥ 1.96 is used for all downstream spatial analysis (§6+).
# BH flags are computed and stored to identify "well-established" clusters
# where the stricter correction also finds signal; these are reported in the
# §5b summary table (cities marked with ★).
GISTAR_Z_HOT  =  1.96   # lower bound for hot Gi* significance (downstream §6+)
GISTAR_Z_COLD = -1.96   # upper bound for cold Gi* significance (downstream §6+)

def compute_gistar(arr: np.ndarray):
    """Return (z_grid, sig_hot_z, sig_cold_z, sig_hot_bh, sig_cold_bh).

    sig_*_z  : Z >= ±1.96 — used by all downstream steps (§6+)
    sig_*_bh : additionally survives BH-FDR at alpha=FDR_ALPHA — stored for reporting
    """
    valid_mask = np.isfinite(arr)
    ys, xs = np.where(valid_mask)
    vals = arr[ys, xs].astype(float)
    n_pix = len(vals)

    z_grid      = np.full_like(arr, np.nan, dtype=float)
    sig_hot_z   = np.zeros_like(arr, dtype=bool)
    sig_cold_z  = np.zeros_like(arr, dtype=bool)
    sig_hot_bh  = np.zeros_like(arr, dtype=bool)
    sig_cold_bh = np.zeros_like(arr, dtype=bool)

    if n_pix < 4:
        return z_grid, sig_hot_z, sig_cold_z, sig_hot_bh, sig_cold_bh

    # Map pixel (row,col) → compact grid indices for Queen weights
    row_off, col_off = ys.min(), xs.min()
    n_rows = ys.max() - row_off + 1
    n_cols = xs.max() - col_off + 1
    idx_map = np.full((n_rows, n_cols), -1, dtype=int)
    for k, (r, c) in enumerate(zip(ys - row_off, xs - col_off)):
        idx_map[r, c] = k

    adj = {k: [] for k in range(n_pix)}
    for k, (r, c) in enumerate(zip(ys - row_off, xs - col_off)):
        for dr, dc in [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < n_rows and 0 <= nc < n_cols and idx_map[nr, nc] >= 0:
                adj[k].append(idx_map[nr, nc])
    w = libW(adj)
    w.transform = 'R'

    g = G_Local(vals, w, star=True, permutations=0)
    z_vals = np.array(g.Zs)
    p_vals = 2 * (1 - sp_norm.cdf(np.abs(z_vals)))
    reject_bh, _, _, _ = multipletests(p_vals, alpha=FDR_ALPHA, method='fdr_bh')

    for k, (y, x) in enumerate(zip(ys, xs)):
        z = z_vals[k]
        z_grid[y, x] = z
        if np.isfinite(z):
            if z >= GISTAR_Z_HOT:
                sig_hot_z[y, x] = True
                if reject_bh[k]:
                    sig_hot_bh[y, x] = True
            elif z <= GISTAR_Z_COLD:
                sig_cold_z[y, x] = True
                if reject_bh[k]:
                    sig_cold_bh[y, x] = True

    return z_grid, sig_hot_z, sig_cold_z, sig_hot_bh, sig_cold_bh

print("compute_gistar() ready — returns (z_grid, sig_hot_z, sig_cold_z, sig_hot_bh, sig_cold_bh)")
print(f"  Z thresholds : hot >= {GISTAR_Z_HOT}  |  cold <= {GISTAR_Z_COLD}  (used by §6+)")
print(f"  BH-FDR alpha : {FDR_ALPHA} — stored in validated_hot_bh / validated_cold_bh (reporting only)")


<a id="nb05-s5"></a>

## §5 — Validated Hot & Cold Spot Masks

### Dual-criterion validation

A pixel is classified as a **validated hot spot** only if it simultaneously satisfies:

1. **Thermal intensity** — LST ≥ P90 (city-region 90th percentile, §3)
2. **Spatial clustering** — Gi★ Z-score ≥ 1.96 (statistically significant hot cluster, §4)

The conjunction of these two criteria eliminates two failure modes:
- Isolated high-LST pixels that are thermal noise rather than genuine spatial clusters (removed by Gi★ filter)
- Pixels at the edge of spatial clusters that have elevated neighbours but are not themselves extreme (removed by P90 filter)

Analogously, a **validated cold spot** requires LST ≤ P10 AND Gi★ Z ≤ −1.96.

### KDE delineation is a third filter

The validated pixel masks produced here are inputs to the KDE step (§6), which fits a continuous density surface and extracts 50/75/90% contour polygons. Only pixels inside the 50% KDE contour are considered "well-delineated" spots. Isolated significant pixels (1–9 pixels, Tier 2) are **never discarded**: each receives a 1 km buffer square that is treated as its L50 geometry. If the Tier-2 spot lies within 2 km of a Tier-1 KDE polygon its geometry is absorbed into that polygon (§6d), otherwise it remains as a standalone Tier-2 spot.

### Cache structure

Per-city results are stored as:
- `{city}_masks_{period}.npz` — binary arrays `(sig_hot_z, sig_cold_z, sig_hot_bh, sig_cold_bh, z_grid, p90, p10)`
- `{city}_masks_{period}.meta.json` — `{source_version, geom_hash, timestamp}`

Cache validation uses `source_version` + geometry hash to detect upstream changes (new zone geometries from NB01) without requiring a date-based expiry.

[↑ Table of Contents](#nb05-toc)


In [ ]:
hotspot_results = {}

for city in cities_with_sev:
    zi = zones_dict.get(city)
    if zi is None:
        continue
    gh = zi['hash']
    city_out = RESULTS_DIR / city
    city_out.mkdir(parents=True, exist_ok=True)
    hotspot_results[city] = {}

    for period in PERIODS:
        if period not in seviri_inv.get(city, {}):
            continue
        tif_path   = seviri_inv[city][period]
        cache_file = city_out / f'{city}_masks_{period}.npz'

        if cache_valid(cache_file, gh):
            loaded = np.load(cache_file, allow_pickle=True)
            hotspot_results[city][period] = {k: loaded[k] for k in loaded.files}
            r   = hotspot_results[city][period]
            vh  = int(r['validated_hot'].sum())
            vc  = int(r['validated_cold'].sum())
            vhb = int(r.get('validated_hot_bh',  np.zeros(1, bool)).sum())
            vcb = int(r.get('validated_cold_bh', np.zeros(1, bool)).sum())
            bh_tag = '  ★' if (vhb + vcb) > 0 else ''
            print(f"  ✓ cache  {city} [{period}]  "
                  f"z_hot={vh}  bh_hot={vhb}  |  z_cold={vc}  bh_cold={vcb}{bh_tag}")
            continue

        arr, tf, crs = load_seviri_arr(tif_path)
        thr = thresholds[city][period]
        p90, p10 = thr['p90'], thr['p10']

        hot_p90  = np.isfinite(arr) & (arr >= p90)
        cold_p10 = np.isfinite(arr) & (arr <= p10)
        z_grid, sig_hot_z, sig_cold_z, sig_hot_bh, sig_cold_bh = compute_gistar(arr)

        result = {
            'arr'  : arr,
            'p90'  : np.float32(p90),
            'p10'  : np.float32(p10),
            'hot_p90'  : hot_p90,
            'cold_p10' : cold_p10,
            'z_grid'   : z_grid,
            'sig_hot'  : sig_hot_z,    # Z-threshold — used by §6+
            'sig_cold' : sig_cold_z,
            'sig_hot_bh'  : sig_hot_bh,   # BH-FDR — reporting only
            'sig_cold_bh' : sig_cold_bh,
            'validated_hot'      : hot_p90  & sig_hot_z,    # downstream (§6+)
            'validated_cold'     : cold_p10 & sig_cold_z,
            'validated_hot_bh'   : hot_p90  & sig_hot_bh,   # reporting only
            'validated_cold_bh'  : cold_p10 & sig_cold_bh,
            'transform': np.array(tf),
        }
        hotspot_results[city][period] = result
        np.savez_compressed(cache_file, **result)
        save_meta(cache_file, gh)

        vh  = int(result['validated_hot'].sum())
        vc  = int(result['validated_cold'].sum())
        vhb = int(result['validated_hot_bh'].sum())
        vcb = int(result['validated_cold_bh'].sum())
        bh_tag = '  ★' if (vhb + vcb) > 0 else ''
        print(f"  computed {city} [{period}]  "
              f"z_hot={vh}  bh_hot={vhb}  |  z_cold={vc}  bh_cold={vcb}{bh_tag}")

print("\n§5 complete.")


<a id="nb05-s5b"></a>

## §5b — Validated Spot Pixels per City × Period

### Dual-criterion pixel classification

A pixel $i$ in city $c$, period $p$ is a **validated hot pixel** if and only if:

$$x_i \ge P_{90}(c,p) \quad \text{AND} \quad G_i^{*}(c,p) \ge +1.96$$

A **validated cold pixel** satisfies:

$$x_i \le P_{10}(c,p) \quad \text{AND} \quad G_i^{*}(c,p) \le -1.96$$

| Symbol | Meaning |
|--------|---------|
| $x_i$ | LST at pixel $i$ (°C) |
| $P_{90}(c,p)$ | City-Region 90th-percentile LST for city $c$, period $p$ (§3) |
| $P_{10}(c,p)$ | City-Region 10th-percentile LST |
| $G_i^{*}(c,p)$ | Gi★ Z-score at pixel $i$ (§4) |

### Reading the table

Each row is one city × period combination.  Columns show the count of pixels that pass **both** criteria simultaneously.

**★ markers** — cities where the stricter BH-FDR correction (§4) also finds significant clusters — the most robust hotspot signals in the dataset:
- **★D** — BH-FDR significant in the *Day* period
- **★N** — BH-FDR significant in the *Night* period

### Interpretation

Cities showing many day-validated pixels but few night-validated pixels typically have a **daytime radiant-heat** profile driven by impervious-surface solar absorption.  The reverse (strong nocturnal signal) indicates dense urban fabric with high thermal mass sustaining heat release overnight.

> Full area (km²) summary by cluster: see **§6c**.

[↑ Table of Contents](#nb05-toc)


In [ ]:
# ── §5b — Validated Spot Pixels per City × Period
import pandas as _pd5b

_rows = []
for city in cities_with_sev:
    cl = city_to_cluster.get(city, -1)
    row = {'cluster': int(cl) if cl != -1 else '?', 'city': city}
    for period in ('day', 'night'):
        res = hotspot_results.get(city, {}).get(period, {})
        row[f'{period}_hot_z']   = int(res.get('validated_hot',    np.zeros(1)).sum()) if res else 0
        row[f'{period}_cold_z']  = int(res.get('validated_cold',   np.zeros(1)).sum()) if res else 0
        row[f'{period}_hot_bh']  = int(res.get('validated_hot_bh', np.zeros(1)).sum()) if res else 0
        row[f'{period}_cold_bh'] = int(res.get('validated_cold_bh',np.zeros(1)).sum()) if res else 0
    _rows.append(row)

_tbl = _pd5b.DataFrame(_rows).sort_values(['cluster', 'city'])

# Build display labels: city + per-period BH markers
def _city_label(r):
    lbl = r['city']
    flags = []
    if r['day_hot_bh']   > 0: flags.append('★D-Hot')
    if r['night_hot_bh'] > 0: flags.append('★N-Hot')
    if r['day_cold_bh']  > 0: flags.append('★D-Cold')
    if r['night_cold_bh']> 0: flags.append('★N-Cold')
    return lbl + (' [' + ', '.join(flags) + ']' if flags else '')

_tbl['City (BH markers)'] = _tbl.apply(_city_label, axis=1)

_disp = _tbl[['cluster', 'City (BH markers)',
              'day_hot_z', 'day_cold_z', 'night_hot_z', 'night_cold_z']].copy()
_disp.columns = ['Cluster', 'City', 'Day Hot', 'Day Cold', 'Night Hot', 'Night Cold']

# Stub rows
_tot = {c: (_tbl[s] > 0).sum() for c, s in [
    ('Day Hot', 'day_hot_z'), ('Day Cold', 'day_cold_z'),
    ('Night Hot', 'night_hot_z'), ('Night Cold', 'night_cold_z')]}
_bh_tot = {c: (_tbl[s] > 0).sum() for c, s in [
    ('Day Hot', 'day_hot_bh'), ('Day Cold', 'day_cold_bh'),
    ('Night Hot', 'night_hot_bh'), ('Night Cold', 'night_cold_bh')]}

_sep = {c: '─'*5 for c in _disp.columns}
_stub_z  = {'Cluster': '', 'City': f'  Cities with Z-signal  (n={len(cities_with_sev)})', **_tot}
_stub_bh = {'Cluster': '', 'City': '  Cities with BH signal (★)', **_bh_tot}
_disp = _pd5b.concat([_disp,
                       _pd5b.DataFrame([_sep]),
                       _pd5b.DataFrame([_stub_z]),
                       _pd5b.DataFrame([_stub_bh])], ignore_index=True)

_W = {'Cluster': 9, 'City': 32, 'Day Hot': 9, 'Day Cold': 10, 'Night Hot': 11, 'Night Cold': 12}
print("§5b — Validated Spot Pixels  |  ★D=BH-Day  ★N=BH-Night  (Cluster → City order)")
_hdr = ''.join(f'{c:<{w}}' for c, w in _W.items())
print(_hdr); print('─' * sum(_W.values()))
for _, r in _disp.iterrows():
    print(''.join(f'{str(r[c]):<{w}}' for c, w in _W.items()))
print()
print("  Note: area (km²) breakdown by cluster in §6c table (after KDE delineation).")


<a id="nb05-s6"></a>

## §6 — Two-Tier KDE Delineation

### Why kernel density estimation?

Validated pixel masks (§5) give binary hot/cold flags on the SEVIRI 1 km grid.  KDE converts this discrete point cloud into a continuous density surface, enabling the extraction of smooth polygonal boundaries (contours) at multiple probability levels.  This produces the spatial footprint of each hotspot cluster as a polygon suitable for area measurement, population exposure calculation, and map visualisation.

### KDE formulation

$$\hat{f}(\mathbf{x}) = \frac{1}{n\,h^{2}}\sum_{i=1}^{n} K\!\left(\frac{\mathbf{x}-\mathbf{x}_i}{h}\right)$$

Scott's bandwidth rule:

$$h = n^{-1/6}\cdot\hat{\sigma}$$

| Symbol | Meaning |
|--------|---------|
| $\hat{f}(\mathbf{x})$ | Estimated kernel density at map location $\mathbf{x}$ |
| $n$ | Number of validated hot (or cold) pixel centroids |
| $h$ | Bandwidth in projected metres (ESRI:102022); set by Scott's rule |
| $K(\cdot)$ | Gaussian kernel: $K(u) = \frac{1}{2\pi}\exp(-\frac{1}{2}\lVert u\rVert^2)$ |
| $\mathbf{x}_i$ | Centroid coordinates of validated pixel $i$ |
| $\hat{\sigma}$ | Standard deviation of pixel centroid coordinates (pooled $x$ and $y$) |

**Contour levels extracted from $\hat{f}$:**

| Level | Label | Meaning |
|-------|-------|---------|
| 50 % density | L50 | Outer extent of the hotspot cluster |
| 75 % density | L75 | Inner zone — moderate density core |
| 90 % density | L90 | Dense core — highest KDE concentration |

Scott's rule is data-adaptive (bandwidth scales with the spread of validated pixels) and requires no cross-validation, making it computationally practical for 56 × 2 = 112 city-period combinations.

### Two-tier approach

| Tier | Pixel count | Method |
|------|------------|--------|
| **Tier 1** | $n \ge 10$ | Full KDE → L50 / L75 / L90 contour polygons |
| **Tier 2** | $1 \le n < 10$ | 1 km buffer square per pixel centroid; adjacent T2 squares merged; T2 adjacent to T1 absorbed into T1 |

Tier 2 exists because SEVIRI's ~3 km pixel spacing means small-area clusters (e.g., a single industrial hotspot) may cover only 1–3 pixels yet represent a genuine heat anomaly.  Discarding sub-10-pixel clusters would systematically exclude smaller cities where the UC is only a few km² in extent.

### §6d — Tier 2 proximity union

Adjacent Tier 2 spots (SEVIRI pixels ~3 km apart) are merged into a single polygon:

- **Connectivity test:** each original 500 m-radius geometry is buffered by 2000 m; two spots are adjacent if their buffered geometries overlap (buffer radius 2500 m > half the 3000 m pixel gap)
- **Merged geometry:** union of 1 km × 1 km squares centred on each connected pixel centroid
- **T2 absorbed by T1:** if any T2 square is within `_CONNECT_M` (2 km) of a Tier-1 L50 polygon, the T2 geometry is unioned into the T1 polygon and the T2 record is removed — capturing the isolated pixel without creating a separate spot
- **Stand-alone Tier-2 spots:** T2 squares not adjacent to any T1 polygon are retained as independent spots (L50 = 1 km² square; no L75 / L90)

> **Why 2000 m connectivity buffer?**  SEVIRI pixels at nadir are ~3000 m apart.  Two adjacent pixel geometries (each 500 m radius) have a ~2000 m gap between  their edges.  A 2000 m buffer exactly closes this gap for direct neighbours  while keeping second-order neighbours (~5000 m edge-to-edge) unconnected.

[↑ Table of Contents](#nb05-toc)


In [ ]:
def pixel_centroids(mask, tf):
    rows, cols = np.where(mask)
    if not len(rows):
        return np.empty((0, 2))
    xs, ys = rio_xy(tf, rows, cols)
    return np.column_stack([xs, ys])

def run_kde(coords, weights=None, grid_size=200):
    x, y = coords[:,0], coords[:,1]
    xmin, xmax, ymin, ymax = x.min(), x.max(), y.min(), y.max()
    pad = max(xmax-xmin, ymax-ymin)*0.2 + 1000
    xi, yi = np.meshgrid(np.linspace(xmin-pad, xmax+pad, grid_size),
                         np.linspace(ymin-pad, ymax+pad, grid_size))
    if weights is not None and KDE_WEIGHTED:
        w = np.where(weights > 0, weights, 0)
        w = w if w.sum() > 0 else None
    else:
        w = None
    kde = gaussian_kde(coords.T, weights=w)
    Z = kde(np.vstack([xi.ravel(), yi.ravel()])).reshape(xi.shape)
    Z_norm = Z / Z.max() if Z.max() > 0 else Z
    return Z_norm, xi, yi

def extract_contour_polys(Z_norm, xi, yi, levels):
    polys_by_level = []
    for lvl in levels:
        polys = []
        for c in sk_measure.find_contours(Z_norm, lvl):
            cf, rf = c[:,1], c[:,0]
            xc = xi[0,0] + cf/(Z_norm.shape[1]-1)*(xi[0,-1]-xi[0,0])
            yc = yi[0,0] + rf/(Z_norm.shape[0]-1)*(yi[-1,0]-yi[0,0])
            if len(xc) >= 4:
                p = Polygon(zip(xc, yc)).simplify(DP_TOL_M, preserve_topology=True)
                if p.is_valid and p.area > 0:
                    polys.append(p)
        polys_by_level.append(polys)
    return polys_by_level

def classify_zone(poly, uc, bra):
    c = poly.centroid
    if uc.contains(c):   return 'urban_core'
    if bra.contains(c):  return 'bra'
    return 'both'

print("KDE helpers defined.")

In [ ]:
# ── Per-city KDE delineation with cache ──────────────────────────────────────
# One record per spatial cluster. The 50% contour defines spot_id and geometry.
# 75%/90% inner zones are matched by centroid containment; their areas and WKT
# geometry strings are stored as attributes of the same record.
from shapely.prepared import prep as _prep
from shapely.geometry import Point as _Point

kde_records_all = {}
spot_counters   = {}

def next_spot_id(city, period, spot_type, tier):
    key = (city, period, spot_type, tier)
    spot_counters[key] = spot_counters.get(key, 0) + 1
    slug = city.lower().replace(' ', '_')
    return f"{slug}_{period[0]}{spot_type[0]}_T{tier}_{spot_counters[key]:03d}"

def _count_pts_in(poly, coords):
    if poly is None or len(coords) == 0: return 0
    pp = _prep(poly)
    return sum(1 for cx, cy in coords if pp.contains(_Point(cx, cy)))

def _match_inner(polys_outer, polys_inner):
    result = {i: [] for i in range(len(polys_outer))}
    for p_in in polys_inner:
        c = p_in.centroid
        for i, p_out in enumerate(polys_outer):
            if p_out.contains(c):
                result[i].append(p_in); break
    return result

for city in cities_with_sev:
    zi = zones_dict.get(city)
    if zi is None: continue
    gh = zi['hash']
    city_out = RESULTS_DIR / city
    city_out.mkdir(parents=True, exist_ok=True)
    kde_records_all[city] = {}

    for period in PERIODS:
        res = hotspot_results.get(city, {}).get(period)
        if res is None: continue

        cache_file = city_out / f'{city}_kde_records_{period}.geojson'

        if cache_valid(cache_file, gh):
            tmp = gpd.read_file(cache_file).to_crs('ESRI:102022')
            rlist = tmp.to_dict('records')
            kde_records_all[city][period] = rlist
            n_h = sum(1 for r in rlist if r.get('spot_type') == 'hot')
            n_c = sum(1 for r in rlist if r.get('spot_type') == 'cold')
            print(f"  ✓ cache  {city} [{period}]  hot_spots={n_h}  cold_spots={n_c}")
            continue

        thr = thresholds[city][period]
        arr = res['arr']
        tf  = RioAffine(*res['transform'][:6])
        records = []

        for spot_type, mask_key, bh_key, p_thresh in [
            ('hot',  'validated_hot',  'validated_hot_bh',  thr['p90']),
            ('cold', 'validated_cold', 'validated_cold_bh', thr['p10']),
        ]:
            mask   = res[mask_key]
            coords = pixel_centroids(mask, tf)
            n_pix  = coords.shape[0]
            bh_sig = bool(res.get(bh_key, np.zeros(1, bool)).sum() > 0)

            if n_pix >= MIN_PIXELS_TIER1:
                # Tier 1 — KDE contour delineation
                if spot_type == 'hot':
                    wts = np.array([max(0., float(arr[r,c]) - p_thresh)
                                    for r,c in zip(*np.where(mask))])
                else:
                    wts = np.array([max(0., p_thresh - float(arr[r,c]))
                                    for r,c in zip(*np.where(mask))])
                Z, xi, yi = run_kde(coords, wts)
                if Z is None: continue

                pl = extract_contour_polys(Z, xi, yi, KDE_CONTOURS)
                polys_50_raw, polys_75_raw, polys_90_raw = pl[0], pl[1], pl[2]

                # Clip all contour polygons to city-region boundary
                cr_geom = zi.get('cr')
                def _clip(polys):
                    result = []
                    for p in polys:
                        try:
                            clipped = p.intersection(cr_geom) if cr_geom else p
                            if clipped and not clipped.is_empty:
                                result.append(clipped)
                        except Exception:
                            result.append(p)
                    return result
                polys_50 = _clip(polys_50_raw)
                polys_75 = _clip(polys_75_raw)
                polys_90 = _clip(polys_90_raw)

                map_75 = _match_inner(polys_50, polys_75)
                ref_for_90 = polys_75 if polys_75 else polys_50
                map_90 = _match_inner(ref_for_90, polys_90) if ref_for_90 else {}

                for i, poly50 in enumerate(polys_50):
                    sid = next_spot_id(city, period, spot_type, 1)

                    inner_75  = map_75.get(i, [])
                    merged_75 = unary_union(inner_75) if inner_75 else None

                    all_90    = [p for lst in map_90.values() for p in lst
                                 if poly50.contains(p.centroid)]
                    merged_90 = unary_union(all_90) if all_90 else None

                    a50 = poly50.area / 1e6
                    a75 = merged_75.area / 1e6 if merged_75 else None
                    a90 = merged_90.area / 1e6 if merged_90 else None
                    dcp = round(a90 / a50 * 100, 2) if (a90 and a50 > 0) else None

                    records.append({
                        'spot_id':        sid,
                        'city':           city,
                        'period':         period,
                        'spot_type':      spot_type,
                        'zone':           classify_zone(poly50, zi['uc'], zi['bra']),
                        'tier':           1,
                        'cluster25':      city_to_cluster.get(city, -1),
                        'n_pixels':       int(n_pix),
                        'area_km2_50':    round(a50, 4),
                        'area_km2_75':    round(a75, 4) if a75 is not None else None,
                        'area_km2_90':    round(a90, 4) if a90 is not None else None,
                        'n_pixels_75':    _count_pts_in(merged_75, coords),
                        'n_pixels_90':    _count_pts_in(merged_90, coords),
                        'dense_core_pct': dcp,
                        'bh_significant': bh_sig,
                        'geom_75_wkt':    merged_75.wkt if merged_75 else None,
                        'geom_90_wkt':    merged_90.wkt if merged_90 else None,
                        'geometry':       poly50,
                    })

            elif n_pix > 0:
                # Tier 2 — buffer union
                bufs = [_Point(x, y).buffer(TIER2_BUFFER_M) for x, y in coords]
                buf  = unary_union(bufs).simplify(DP_TOL_M, preserve_topology=True)
                if buf and not buf.is_empty:
                    records.append({
                        'spot_id':        next_spot_id(city, period, spot_type, 2),
                        'city':           city,
                        'period':         period,
                        'spot_type':      spot_type,
                        'zone':           classify_zone(buf, zi['uc'], zi['bra']),
                        'tier':           2,
                        'cluster25':      city_to_cluster.get(city, -1),
                        'n_pixels':       int(n_pix),
                        'area_km2_50':    round(buf.area / 1e6, 4),
                        'area_km2_75':    None, 'area_km2_90': None,
                        'n_pixels_75':    0,    'n_pixels_90': 0,
                        'dense_core_pct': None,
                        'bh_significant': bh_sig,
                        'geom_75_wkt':    None, 'geom_90_wkt': None,
                        'geometry':       buf,
                    })

        if records:
            gdf_recs = gpd.GeoDataFrame(records, crs='ESRI:102022')
            gdf_recs.to_crs('EPSG:4326').to_file(cache_file, driver='GeoJSON')
            save_meta(cache_file, gh)
            n_h = sum(1 for r in records if r['spot_type'] == 'hot')
            n_c = sum(1 for r in records if r['spot_type'] == 'cold')
            print(f"  computed {city} [{period}]  hot_spots={n_h}  cold_spots={n_c}")
        else:
            print(f"  no polygons {city} [{period}]")
        kde_records_all[city][period] = records

print("\n§6 KDE delineation complete.")

In [ ]:
# §6d — Tier 2 Proximity Union  (1km² square expand + adjacent merge)
# Each Tier 2 spot geometry is replaced with a 1km × 1km square centred on
# its centroid.  Squares that overlap (centroids within 1km) are detected via
# a union-find and then unioned into one continuous polygon.
# Stand-alone spots keep their 1km² square as the new geometry.
# Tier 1 spots are never merged — they pass through unchanged.

from shapely.geometry import box as _box6d
from shapely.ops     import unary_union as _uu6d
from shapely.wkt     import loads as _wl6d
from collections     import defaultdict as _dd6d

_HALF6D    = 500.   # half-side of 1km² square (metres, ESRI:102022)
_CONNECT_M = 2000.  # adjacency buffer: SEVIRI pixels ~3km apart, 2km bridges the gap


def _make_sq6d(geom):
    # Replace geometry with a 1km x 1km square at its centroid.
    cx, cy = geom.centroid.x, geom.centroid.y
    return _box6d(cx - _HALF6D, cy - _HALF6D, cx + _HALF6D, cy + _HALF6D)


def _find_components6d(geoms, connect_m=_CONNECT_M):
    # Union-find: buffer each original geometry by connect_m, detect overlap.
    # This bridges the ~2km gap between adjacent SEVIRI pixel spot geometries.
    n = len(geoms)
    parent = list(range(n))
    def _find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def _uf_union(x, y):
        px, py = _find(x), _find(y)
        if px != py: parent[px] = py
    _buf = [g.buffer(connect_m) for g in geoms]
    for i in range(n):
        for j in range(i + 1, n):
            if _buf[i].intersects(_buf[j]):
                _uf_union(i, j)
    groups = _dd6d(list)
    for i in range(n): groups[_find(i)].append(i)
    return list(groups.values())


def _merge_group6d(recs, squares):
    # Union the 1km2 squares of a connected Tier 2 cluster.
    merged_geom = _uu6d([squares[i] for i in range(len(recs))])
    if merged_geom.is_empty:
        merged_geom = _uu6d([r['geometry'] for r in recs])

    geoms75 = [_wl6d(r['geom_75_wkt']) for r in recs
               if r.get('geom_75_wkt') and isinstance(r.get('geom_75_wkt'), str)]
    geoms90 = [_wl6d(r['geom_90_wkt']) for r in recs
               if r.get('geom_90_wkt') and isinstance(r.get('geom_90_wkt'), str)]
    geom75 = _uu6d(geoms75) if geoms75 else None
    geom90 = _uu6d(geoms90) if geoms90 else None

    a50 = merged_geom.area / 1e6
    a75 = geom75.area / 1e6 if geom75 else 0.
    a90 = geom90.area / 1e6 if geom90 else 0.
    base = max(recs, key=lambda r: r.get('area_km2_50', 0))
    is_merged = len(recs) > 1
    return {
        **base,
        'is_merged_t2':   is_merged,
        'spot_id':        (base['spot_id'] + f'_u{len(recs)}') if is_merged else base['spot_id'],
        'tier':           min(r.get('tier', 2) for r in recs),
        'n_pixels':       sum(r.get('n_pixels', 0) for r in recs),
        'n_pixels_75':    sum(r.get('n_pixels_75', 0) for r in recs),
        'n_pixels_90':    sum(r.get('n_pixels_90', 0) for r in recs),
        'area_km2_50':    round(a50, 4),
        'area_km2_75':    round(a75, 4) if a75 else None,
        'area_km2_90':    round(a90, 4) if a90 else None,
        'dense_core_pct': round(a90 / a50 * 100, 2) if a50 > 0 and a90 > 0 else 0.,
        'bh_significant': any(r.get('bh_significant', False) for r in recs),
        'geom_75_wkt':    geom75.wkt if geom75 else None,
        'geom_90_wkt':    geom90.wkt if geom90 else None,
        'geometry':       merged_geom,
    }


_stats6d = {'before': 0, 'after': 0, 'merged': 0}
for city in list(kde_records_all.keys()):
    for period in list(kde_records_all[city].keys()):
        recs_all = kde_records_all[city][period]
        if not recs_all: continue
        new_recs = []
        for st in {r['spot_type'] for r in recs_all}:
            recs = [r for r in recs_all if r['spot_type'] == st]
            _stats6d['before'] += len(recs)

            if len(recs) <= 1:
                for _r in recs: _r.setdefault('is_merged_t2', False)
                new_recs.extend(recs)
                _stats6d['after'] += len(recs)
                continue

            # Expand each to 1km² square
            squares6d = [_make_sq6d(r['geometry']) for r in recs]
            components6d = _find_components6d([r['geometry'] for r in recs])

            for comp_idx in components6d:
                grp    = [recs[i]     for i in comp_idx]
                sq_grp = [squares6d[i] for i in comp_idx]
                t1 = [(grp[i], sq_grp[i]) for i in range(len(grp)) if grp[i].get('tier', 2) == 1]
                t2 = [(grp[i], sq_grp[i]) for i in range(len(grp)) if grp[i].get('tier', 2) != 1]

                if t1 and t2:
                    # T2 spots adjacent to T1 → absorb into T1 geometry
                    _t2_union = _uu6d([_make_sq6d(r['geometry']) for r, _ in t2])
                    for r, _ in t1:
                        _ng = r['geometry'].union(_t2_union)
                        _ra = {**r,
                               'geometry':      _ng,
                               'area_km2_50':   round(_ng.area / 1e6, 4),
                               'is_merged_t2':  True,
                               'n_absorbed_t2': len(t2)}
                        new_recs.append(_ra)
                    _stats6d['merged'] += len(t2)
                elif t1:
                    for r, _ in t1:
                        r.setdefault('is_merged_t2', False)
                        r.setdefault('n_absorbed_t2', 0)
                        new_recs.append(r)
                else:
                    # Pure T2 component (no adjacent T1)
                    if len(t2) > 1:
                        _stats6d['merged'] += len(t2)
                        _mg = _merge_group6d([r for r, _ in t2], [sq for _, sq in t2])
                        _mg.setdefault('n_absorbed_t2', 0)
                        new_recs.append(_mg)
                    elif len(t2) == 1:
                        t2[0][0].setdefault('is_merged_t2', False)
                        t2[0][0].setdefault('n_absorbed_t2', 0)
                        new_recs.append(t2[0][0])
            _stats6d['after'] += len([r for r in new_recs if r['spot_type'] == st])
        kde_records_all[city][period] = new_recs

print(f"§6d Tier2 union: {_stats6d['before']} spots → {_stats6d['after']}  "
      f"({_stats6d['merged']} T2 spots merged/absorbed; "
      f"T2 adjacent to T1 → absorbed into T1 geometry)")


<a id="nb05-s6b"></a>

## §6b — Contour Completeness Diagnostic

### Tier classification summary

| Code | Full name | Condition | Method |
|------|-----------|-----------|--------|
| **T1** | Tier 1 — KDE delineation | ≥ 10 validated Gi★ pixels | KDE → 50/75/90% contours |
| **T2** | Tier 2 — buffer circle | 1–9 validated pixels | 500 m circle per pixel; adjacent spots merged to 1 km² squares |
| **T0** | No spot | 0 validated pixels | No polygon produced |

### Interpreting the diagnostic

Cities where **both Day and Night** produce Tier 1 hotspots have persistent, large-area heat anomalies (typically high-density commercial or industrial cores). Cities showing only Tier 2, or only one period, represent smaller or more localised thermal anomalies.

The completeness diagnostic flags three warning conditions for downstream users:
1. **Contour not closed** — KDE surface does not reach the target probability level (rare; indicates very diffuse validated pixel distribution)
2. **Single pixel / Tier 2** — fewer than 10 validated pixels; polygon should be interpreted as an approximate location rather than a precisely delineated boundary
3. **No spots** — zero validated pixels; city does not show a statistically significant thermal cluster in this period

> Tier classification is stored in the `tier` column of the GDF (§7) and is used to stratify analysis in §12.

[↑ Table of Contents](#nb05-toc)


In [ ]:
# ── §6b Contour completeness + totals ────────────────────────────────────────
print("\n§6b — Contour Completeness per City × Period × Spot Type")
print("=" * 90)
hdr = (f"  {'City':<22} {'Period':<7} {'Type':<6} {'Spots':>5} "
       f"{'T1':>4} {'T175':>5} {'T190':>5} {'T1L75%':>7} {'T1L90%':>7} "
       f"{'T2':>4} {'T275':>5} {'T290':>5} {'T2L75%':>7} {'T2L90%':>7}")
print(hdr)
print("  " + "-" * 88)

tot_spots  = 0
tot_t1     = 0; tot_t1_75 = 0; tot_t1_90 = 0
tot_t2     = 0; tot_t2_75 = 0; tot_t2_90 = 0
n_hot      = 0; n_cold    = 0

for city in cities_with_sev:
    for period in PERIODS:
        recs = kde_records_all.get(city, {}).get(period, [])
        for st in ('hot', 'cold'):
            sr = [r for r in recs if r.get('spot_type') == st]
            if not sr: continue

            t1_recs = [r for r in sr if r.get('tier') == 1]
            t2_recs = [r for r in sr if r.get('tier') != 1]
            n_t1   = len(t1_recs)
            n_t2   = len(t2_recs)

            # L75/L90 counts — separately per tier
            n_t1_75 = sum(1 for r in t1_recs if r.get('area_km2_75') is not None)
            n_t1_90 = sum(1 for r in t1_recs if r.get('area_km2_90') is not None)
            n_t2_75 = sum(1 for r in t2_recs if r.get('area_km2_75') is not None)
            n_t2_90 = sum(1 for r in t2_recs if r.get('area_km2_90') is not None)

            pct_t1_75 = f'{100*n_t1_75/n_t1:.0f}%' if n_t1 else '—'
            pct_t1_90 = f'{100*n_t1_90/n_t1:.0f}%' if n_t1 else '—'
            pct_t2_75 = f'{100*n_t2_75/n_t2:.0f}%' if n_t2 else '—'
            pct_t2_90 = f'{100*n_t2_90/n_t2:.0f}%' if n_t2 else '—'

            tot_spots  += len(sr)
            tot_t1     += n_t1;   tot_t1_75 += n_t1_75; tot_t1_90 += n_t1_90
            tot_t2     += n_t2;   tot_t2_75 += n_t2_75; tot_t2_90 += n_t2_90
            if st == 'hot': n_hot  += len(sr)
            else:             n_cold += len(sr)

            # Flag if T1 coverage is incomplete
            flag = ''
            if n_t1 > 0:
                if n_t1_90 < n_t1: flag = ' ← T1 L90 incomplete'
                elif n_t1_75 < n_t1: flag = ' ← T1 L75 incomplete'

            print(f"  {city:<22} {period:<7} {st:<6} {len(sr):>5} "
                  f"{n_t1:>4} {n_t1_75:>5} {n_t1_90:>5} {pct_t1_75:>7} {pct_t1_90:>7} "
                  f"{n_t2:>4} {n_t2_75:>5} {n_t2_90:>5} {pct_t2_75:>7} {pct_t2_90:>7}{flag}")

# ── Totals ────────────────────────────────────────────────────────────────────
print()
print(f"  TOTAL spots:  {tot_spots}  (Hot: {n_hot}  Cold: {n_cold})")
print(f"  T1 (KDE):   {tot_t1:>4}  "
      f"→ L75: {tot_t1_75}/{tot_t1} = {100*tot_t1_75//tot_t1 if tot_t1 else 0}%  "
      f"| L90: {tot_t1_90}/{tot_t1} = {100*tot_t1_90//tot_t1 if tot_t1 else 0}%")
print(f"  T2 (buffer): {tot_t2:>4}  "
      f"→ L75: {tot_t2_75}/{tot_t2} = {100*tot_t2_75//tot_t2 if tot_t2 else 0}%  "
      f"| L90: {tot_t2_90}/{tot_t2} = {100*tot_t2_90//tot_t2 if tot_t2 else 0}%")
_tot_75 = tot_t1_75 + tot_t2_75
_tot_90 = tot_t1_90 + tot_t2_90
_tot_all = tot_t1 + tot_t2
print(f"  ALL spots:   {_tot_all:>4}  "
      f"→ L75: {_tot_75}/{_tot_all} = {100*_tot_75//_tot_all if _tot_all else 0}%  "
      f"| L90: {_tot_90}/{_tot_all} = {100*_tot_90//_tot_all if _tot_all else 0}%")


<a id="nb05-s6c"></a>

## §6c — Comprehensive Spot Summary (§5b + §6 Combined)

One row per city × period × spot_type. Cluster25 is the primary grouping column so the table is ordered by physiographic landscape character, enabling direct comparison of hotspot metrics within climatically similar groups of cities.

### Column guide

| Column | Meaning |
|--------|---------|
| `Cluster` | Cluster25 of the city |
| `City` | City name |
| `Period` | `day` or `night` |
| `Spot type` | `hot` or `cold` |
| `N_pix_valid` | Validated Gi★ pixels in §5 mask |
| `Tier` | T1 (KDE) or T2 (buffer) |
| `Area_50 (km²)` | 50% KDE contour area |
| `Area_75 (km²)` | 75% KDE contour area |
| `Area_90 (km²)` | 90% KDE dense core area |
| `Dense_core_pct` | Area_90 / Area_50 × 100 — compactness index |
| `BH_significant` | ★ if spot also passes BH-FDR correction |

### Interpreting dense-core percentage

A high `Dense_core_pct` (> 50%) indicates a compact, concentrated heat anomaly — the hot zone has a well-defined core relative to its total extent. A low value (< 20%) suggests a diffuse, sprawling pattern, possibly driven by impervious-surface sprawl rather than a discrete heat source.

[↑ Table of Contents](#nb05-toc)


In [ ]:
# ── §6c — Comprehensive spot summary ─────────────────────────────────────────
import pandas as _pd6c

_s6c_rows = []
for city in cities_with_sev:
    cl = city_to_cluster.get(city, -1)
    for period in PERIODS:
        recs = kde_records_all.get(city, {}).get(period, [])
        res  = hotspot_results.get(city, {}).get(period, {})
        for st in ('hot', 'cold'):
            sr = [r for r in recs if r.get('spot_type') == st]
            if not sr and not res: continue

            n_px_z  = int(res.get(f'validated_{st}',    np.zeros(1)).sum()) if res else 0
            n_px_bh = int(res.get(f'validated_{st}_bh', np.zeros(1)).sum()) if res else 0
            n_t1  = sum(1 for r in sr if r.get('tier') == 1)
            n_t2  = sum(1 for r in sr if r.get('tier') == 2)
            n_75  = sum(1 for r in sr if r.get('area_km2_75') is not None)
            n_90  = sum(1 for r in sr if r.get('area_km2_90') is not None)
            a50 = round(sum((r.get('area_km2_50') or 0) for r in sr), 3)
            a75 = round(sum((r.get('area_km2_75') or 0) for r in sr), 3)
            a90 = round(sum((r.get('area_km2_90') or 0) for r in sr), 3)
            bh_day   = n_px_bh > 0 and period == 'day'
            bh_night = n_px_bh > 0 and period == 'night'

            # BH flag label
            bh_lbl = ''
            if n_px_bh > 0:
                bh_lbl = f'[★{"D" if period=="day" else "N"}-{st.capitalize()}]'

            _s6c_rows.append({
                'Cluster': int(cl) if cl != -1 else -1,
                'City': city + (' ' + bh_lbl if bh_lbl else ''),
                'Period': period, 'Type': st,
                'n_px_z': n_px_z, 'n_px_bh': n_px_bh,
                'T1': n_t1, 'T2': n_t2,
                'area_50_km2': a50, 'area_75_km2': a75, 'area_90_km2': a90,
                'area_75_pct': round(a75 / a50 * 100, 1) if a50 > 0 else 0.,
                'area_90_pct': round(a90 / a50 * 100, 1) if a50 > 0 else 0.,
                'L75_compl': f'{100*n_75//n_t1}%' if n_t1 else '—',
                'L90_compl': f'{100*n_90//n_t1}%' if n_t1 else '—',
            })

_s6c = _pd6c.DataFrame(_s6c_rows).sort_values(['Cluster', 'City', 'Period', 'Type'])
print("§6c — Comprehensive Spot Summary (Cluster → City → Period → Type)")
print("=" * 90)
print(_s6c.to_string(index=False))
print()
_s6c_out = RESULTS_DIR / 'spot_comprehensive_summary.csv'
_s6c.to_csv(_s6c_out, index=False)
print(f"Saved → {_s6c_out.name}")


<a id="nb05-s7"></a>

## §7 — Per-spot layer and enrichment

### Overview

This section builds `gdf`, one row per spot (the L50 KDE polygon), enriched with population,
LST, SUHII and the zone and typology attributes, and writes it as
`all_cities_spots.geojson`. That file is read by the dataset build (the `spots` layer and the
§9 hit rates) and by NB06. `all_cities_spots.csv` and `all_cities_spots.meta.json` are this
section's own cache (the CSV is also read by the dataset QA).

The per-contour-level table `gdf_master` and NB05's working GeoPackage are built in
[§10](#nb05-s10), not here.

### `gdf` schema (backward-compatible, one row per spot)

| Column | Type | Notes |
|--------|------|-------|
| `spot_id` | str | `{city_slug}_{p}{st}_T{tier}_{seq:03d}` |
| `city` / `period` / `spot_type` / `tier` | str/int | identification |
| `study_year` | int | from `config.STUDY_YEAR` — enables multi-year filter |
| `area_km2_50/75/90` | float | KDE contour areas (km²) |
| `dense_core_pct` | float | area_90 / area_50 × 100 |
| `population` / `pop_75` / `pop_90` | float | GHS-POP summed at native 100 m within the L50 / L75 / L90 contour; NaN where that level does not exist |
| `mean_lst` / `std_lst` / `max_lst` / `min_lst` | float | zonal LST stats (50% polygon) |
| `core_mean_lst` / `core_max_lst` | float | zonal LST stats for 90% polygon |
| `suhii` | float | Surface UHI Intensity (NB04) |
| `bh_significant` | bool | passes BH-FDR correction |
| `is_merged_t2` | bool | Tier-2 adjacent union |
| `persistent_dn` | bool | hot spot present in both day and night |
| `climate_class` / `landcover_class` / `elevation_class` / `geology_class` | str | |
| `city_area_km2` / `uc_area_km2` / `bra_area_km2` | float | zone areas |
| `geom_75_wkt` / `geom_90_wkt` | str | WKT of inner contours (for §11 maps) |
| `geometry` | Polygon | 50% KDE contour in ESRI:102022 |

[↑ Table of Contents](#nb05-toc)


In [ ]:
def zonal_stats(poly_102022, tif_path):
    # Mean/std/max/min LST within poly_102022. SEVIRI rasters stored in ESRI:102022.
    # Do NOT convert to 4326 — mask directly using 102022 coordinates.
    try:
        with rasterio.open(tif_path) as src:
            out, _ = rio_mask(src, [mapping(poly_102022)],
                              crop=True, nodata=np.nan, filled=True)
        vals = out[0].astype(float)
        vals[(vals < -50) | (vals > 100)] = np.nan
        valid = vals[np.isfinite(vals)]
        if not valid.size:
            return dict(mean_lst=np.nan, std_lst=np.nan, max_lst=np.nan, min_lst=np.nan)
        return dict(mean_lst=round(float(valid.mean()), 3),
                    std_lst =round(float(valid.std()),  3),
                    max_lst =round(float(valid.max()),  3),
                    min_lst =round(float(valid.min()),  3))
    except Exception:
        return dict(mean_lst=np.nan, std_lst=np.nan, max_lst=np.nan, min_lst=np.nan)


def pop_for(spot_id, level='L50'):
    """Population inside a spot's contour, from the native 100 m extraction.

    Looked up, never recomputed: the values come from GHS-POP summed server-side on
    its own 100 m grid (see the cache README). Returns NaN when the level does not
    exist for that spot -- tier-2 spots have no L75 or L90 -- so a missing contour
    stays distinguishable from a genuinely unpopulated one, which is 0.0.
    """
    return POP100M.get(f'{spot_id}_{level}', np.nan)


def pop_for_zone(city, zone='city_region'):
    """Population inside a city zone ('urban_core', 'bra', 'city_region').

    Use 'city_region' as the denominator for exposure shares. Spots are detected across
    the whole city region and do not respect the UC boundary: of the 185 L50 footprints
    only 16 lie wholly inside the urban core, 75 lie wholly inside the BRA, 36 straddle
    the two, and 8 sit mostly in neither. On average a spot is 20.9% UC, 69.2% BRA and
    9.9% elsewhere in the CR, so UC would be the wrong base and only CR covers all of it.
    """
    slug = re.sub(r'[^a-z0-9]+', '_', str(city).lower()).strip('_')
    return POPZONE.get(f'{slug}_{zone}', np.nan)


In [ ]:
# §7 — GDF Enrichment  (one row per spot — used by §9 through §12)
# gdf_master + GeoPackage export → §10

# ── STUDY_YEAR from config.py ─────────────────────────────────────────────────
# config exposes the study year twice: PROJECT_STUDY_YEAR (line ~26) and
# PROJECT_SETTINGS['study_year'], which STUDY_YEAR aliases. They are meant to agree; if they
# ever drift this raises rather than quietly analysing the wrong year, and there is no silent
# hard-coded fallback.
import importlib as _il7, sys as _sys7
if str(ROOT) not in _sys7.path:
    _sys7.path.insert(0, str(ROOT))
_cfg7 = _il7.import_module('config')
_sy_alias   = getattr(_cfg7, 'STUDY_YEAR', None)
_sy_project = getattr(_cfg7, 'PROJECT_STUDY_YEAR', None)
if _sy_alias is None and _sy_project is None:
    raise RuntimeError('config defines neither STUDY_YEAR nor PROJECT_STUDY_YEAR')
if None not in (_sy_alias, _sy_project) and int(_sy_alias) != int(_sy_project):
    raise RuntimeError(
        f'config study year disagrees: STUDY_YEAR={_sy_alias} vs '
        f'PROJECT_STUDY_YEAR={_sy_project}. Set both to the same year.')
STUDY_YEAR = int(_sy_alias if _sy_alias is not None else _sy_project)
print(f"Study year: {STUDY_YEAR}")

# ── Section A: Wide-format gdf ────────────────────────────────────────────────
_all_hashes  = '|'.join(sorted(zones_dict[c]['hash'] for c in cities_with_sev if c in zones_dict))
overall_hash = hashlib.sha256(_all_hashes.encode()).hexdigest()[:16]

master_csv     = RESULTS_DIR / 'all_cities_spots.csv'
master_geojson = RESULTS_DIR / 'all_cities_spots.geojson'

_land_lookup = {}
if 'cluster_df' in dir():
    import re as _re_land
    def _parse_cls_area(desc):
        m = _re_land.findall(r'about (\d+\.?\d*) sq', str(desc))
        return float(m[0]) if m else float('nan')
    for _, _lr in cluster_df.iterrows():
        _ia  = float(_lr.get('intersect_area_km2', 0) or 0)
        _ca  = _parse_cls_area(_lr.get('description', ''))
        _pct = round(_ca / _ia * 100, 2) if (_ia > 0 and _ca == _ca) else float('nan')
        _land_lookup[_lr['city_name']] = {
            'climate_class':      str(_lr.get('Climate_Class', '')),
            'landcover_class':    str(_lr.get('Landcover_Class', '')),
            'elevation_class':    str(_lr.get('Elevation_Class', '')),
            'geology_class':      str(_lr.get('Geology_Class', '')),
            'pct_dominant_class': _pct,
        }

# ── NB03 temporal trend lookup (Sen's slope + Mann-Kendall per city×period) ──
_nb03_trend = {}
try:
    _nb03_p = (ROOT / 'Results' / '03_temporal_analysis' /
               'all_cities_temporal_annual.csv')
    if _nb03_p.exists():
        _nb03_df = pd.read_csv(_nb03_p)
        _nb03_uc = _nb03_df[_nb03_df.get('zone', pd.Series(['urban_core']*len(_nb03_df))) == 'urban_core'] \
                   if 'zone' in _nb03_df.columns else _nb03_df
        for (_c, _p), _grp in _nb03_uc.groupby(['city', 'period']):
            _nb03_trend.setdefault(_c, {})[_p] = {
                'sen_slope_C_per_decade': float(_grp['sen_slope_per_decade'].dropna().iloc[0])
                    if 'sen_slope_per_decade' in _grp and _grp['sen_slope_per_decade'].notna().any() else float('nan'),
                'mk_trend':  str(_grp['mk_trend'].dropna().iloc[0])
                    if 'mk_trend' in _grp and _grp['mk_trend'].notna().any() else '',
                'mk_p_value': float(_grp['mk_p'].dropna().iloc[0])
                    if 'mk_p' in _grp and _grp['mk_p'].notna().any() else float('nan'),
            }
        print(f'NB03 trend lookup: {len(_nb03_trend)} cities')
except Exception as _e:
    print(f'  ⚠ NB03 trend lookup skipped: {_e}')

# ── NB04 heat event lookup (MODIS hot-day counts per city×period) ─────────
_nb04_heat = {}
try:
    _nb04_p = (ROOT / 'Results' / '04_temporal_analysis_daily' /
               'city_heat_event_summary.csv')
    if _nb04_p.exists():
        _nb04_df = pd.read_csv(_nb04_p)
        for _, _hr in _nb04_df.iterrows():
            _cn = str(_hr.get('city', '')).strip()
            _nb04_heat[_cn] = {
                'day':   {'n_persistent_events':    int(_hr.get('era5_day_n_persistent_events',      0) or 0),
                          'validation_tier':        str(_hr.get('day_sel_tier', '')),
                          'longest_event_days':     int(_hr.get('era5_day_longest_event_days',       0) or 0)},
                'night': {'n_persistent_events':    int(_hr.get('era5_night_n_persistent_events',    0) or 0),
                          'validation_tier':        str(_hr.get('night_sel_tier', '')),
                          'longest_event_days':     int(_hr.get('era5_night_longest_event_days',     0) or 0)},
            }
        print(f'NB04 heat lookup: {len(_nb04_heat)} cities')
except Exception as _e:
    print(f'  ⚠ NB04 heat lookup skipped: {_e}')

def _wide_schema_valid(path):
    try:
        s = gpd.read_file(path)  # read all rows to validate
        return (all(c in s.columns for c in
                   ('area_km2_50', 'climate_class', 'pop_75',
                    'city_area_km2', 'pct_dominant_class',
                    'is_merged_t2', 'study_year')) and
                int(s['study_year'].iloc[0]) == STUDY_YEAR and
                (s['population'].fillna(0) > 0).any())
    except Exception:
        return False

_skip_enrichment = False
if not FORCE_RERUN and master_csv.exists() and master_geojson.exists():
    _mc = master_csv.with_suffix('.meta.json')
    if _mc.exists():
        _m = json.loads(_mc.read_text())
        if (_m.get('source_version') == SOURCE_VERSION
                and _m.get('overall_hash') == overall_hash
                and _m.get('study_year', 0) == STUDY_YEAR
                and _wide_schema_valid(master_geojson)):
            gdf = gpd.read_file(master_geojson)
            for col in ('bh_significant', 'persistent_dn'):
                if col in gdf.columns:
                    gdf[col] = gdf[col].astype(bool)
            if 'study_year' not in gdf.columns:
                gdf['study_year'] = STUDY_YEAR
            print(f"✓ GDF loaded from cache  ({len(gdf)} spots × {len(gdf.columns)} cols)")
            _skip_enrichment = True

if not _skip_enrichment:
    from shapely.wkt import loads as _wkt_loads
    _CSV_OMIT = {'geom_75_wkt', 'geom_90_wkt'}

    all_records = []
    for city in cities_with_sev:
        _land = _land_lookup.get(city, {})
        for period in PERIODS:
            recs = kde_records_all.get(city, {}).get(period, [])
            tif  = seviri_inv.get(city, {}).get(period)
            for rec in recs:
                poly = (rec['geometry'] if hasattr(rec.get('geometry'), 'area')
                        else shape(rec['geometry']))
                stats = (zonal_stats(poly, tif) if tif else
                         dict(mean_lst=np.nan, std_lst=np.nan,
                              max_lst=np.nan,  min_lst=np.nan))
                wkt90 = rec.get('geom_90_wkt')
                if wkt90 and isinstance(wkt90, str):
                    try:
                        _p90 = _wkt_loads(wkt90)
                        cs = zonal_stats(_p90, tif) if tif else {}
                        core_mean_lst = cs.get('mean_lst', np.nan)
                        core_max_lst  = cs.get('max_lst',  np.nan)
                        pop_90 = pop_for(rec['spot_id'], 'L90')
                    except Exception:
                        core_mean_lst = core_max_lst = np.nan; pop_90 = np.nan
                else:
                    core_mean_lst = core_max_lst = np.nan; pop_90 = np.nan

                wkt75 = rec.get('geom_75_wkt')
                pop_75 = pop_for(rec['spot_id'], 'L75')

                all_records.append({
                    **{k: v for k, v in rec.items() if k != 'geometry'},
                    'population':    pop_for(rec['spot_id'], 'L50'),
                    'pop_75':        pop_75,
                    'pop_90':        pop_90,
                    'suhii':         suhii_lookup.get(city, {}).get(period, np.nan),
                    'core_mean_lst': core_mean_lst,
                    'core_max_lst':  core_max_lst,
                    'climate_class':      _land.get('climate_class', ''),
                    'landcover_class':    _land.get('landcover_class', ''),
                    'elevation_class':    _land.get('elevation_class', ''),
                    'geology_class':      _land.get('geology_class', ''),
                    'pct_dominant_class': _land.get('pct_dominant_class', float('nan')),
                    'city_area_km2':  round(zones_dict[city]['cr'].area  / 1e6, 4) if city in zones_dict else float('nan'),
                    'uc_area_km2':    round(zones_dict[city]['uc'].area  / 1e6, 4) if city in zones_dict else float('nan'),
                    'bra_area_km2':   round(zones_dict[city]['bra'].area / 1e6, 4) if city in zones_dict else float('nan'),
                    'study_year':     STUDY_YEAR,
                    # NB03 temporal trend
                    **{k: _nb03_trend.get(city, {}).get(period, {}).get(k, float('nan'))
                       for k in ('sen_slope_C_per_decade', 'mk_p_value')},
                    'mk_trend':       _nb03_trend.get(city, {}).get(period, {}).get('mk_trend', ''),
                    # NB04 heat events
                    **{k: _nb04_heat.get(city, {}).get(period, {}).get(k, 0)
                       for k in ('n_persistent_events', 'longest_event_days')},
                    'nb04_val_tier':  _nb04_heat.get(city, {}).get(period, {}).get('validation_tier', ''),
                    **stats,
                    'geometry':      poly,
                })

    if all_records:
        gdf = gpd.GeoDataFrame(all_records, crs='ESRI:102022')
        has_day   = set(gdf[(gdf.period=='day')   & (gdf.spot_type=='hot')]['city'])
        has_night = set(gdf[(gdf.period=='night') & (gdf.spot_type=='hot')]['city'])
        gdf['persistent_dn'] = gdf['city'].isin(has_day & has_night)

        csv_cols = [c for c in gdf.columns if c not in _CSV_OMIT and c != 'geometry']
        gdf[csv_cols].to_csv(master_csv, index=False)
        gdf.to_crs('EPSG:4326').to_file(master_geojson, driver='GeoJSON')
        master_csv.with_suffix('.meta.json').write_text(
            json.dumps({'source_version': SOURCE_VERSION,
                        'overall_hash': overall_hash,
                        'study_year': STUDY_YEAR}, indent=2))
        n_h = (gdf.spot_type == 'hot').sum()
        n_c = (gdf.spot_type == 'cold').sum()
        print(f"§7 GDF: {len(gdf)} spots ({n_h} hot / {n_c} cold) × {len(gdf.columns)} cols")
        print(f"  → master GDF and GeoPackage export in §10")
    else:
        gdf = gpd.GeoDataFrame()
        print("⚠ §7: no spot polygons — GDF is empty (check §5–§6)")

<a id="nb05-s8"></a>

## §8 — Distribution assessment

> **Not performed in this notebook.** Earlier versions of NB05 assessed the distribution of the
> spot variables here (CV, skewness, `log_recommended`) and wrote `distribution_assessment.csv`.
> No such code remains in NB05, no `distribution_assessment.csv` exists in
> `Results/05_hotspot_analysis/`, and NB06 does not read one. The heading is kept so the section
> numbers stay stable; the normality tests used by the association analysis are NB06's.

[↑ Table of Contents](#nb05-toc)

<a id="nb05-s9"></a>

## §9 — Stakeholder hit rates and buffer sensitivity

### Validation approach

Stakeholder GPS points represent locations where local informants independently perceived
thermal anomalies. Spatial hit-rate validation asks: *do the detected spots capture the
locations that people on the ground experience as anomalously hot or cold?* It is computed
separately for **day** and **night**, matching the diurnal structure of the SEVIRI analysis.

### Inputs: exactly what the dataset publishes

- **Spots:** NB05's L50 footprints from §7 (`all_cities_spots.geojson`), as the build reads them.
- **Points:** the cohort's 94 marked points (55 hot / 39 cold), from the 105 participants,
  rounded to three decimal degrees by the build's own function
  (`scripts/build_master_summary.prepare_cohort_points`). Both are imported, never copied.

### Hit-rate definition (row-based)

Each point is buffered by $d$ and joined to the footprints of the same spot type and period.
A point inside several footprints contributes one row per footprint, so

$$\mathrm{hit\ rate}(d) = \frac{\text{rows that fall inside a footprint}}{\text{all rows}},$$

and the denominator grows with $d$. The buffer runs from 0.5 to 10 km in 0.5 km steps; the
dotted line marks `config.STAKEHOLDER_VALIDATION_BUFFER_M` (1 km).

### Output

**No file.** The published hit rates (`hitrate_sensitivity`) belong to the dataset and are written
only by the build in [§14](#nb05-s14). This section shows the national, per-cluster and per-city
tables at 0.5, 1, 5 and 10 km and the sensitivity charts, and keeps the full table in
`HITRATE_NB05`, which §14b checks against the published layer.

### Interpretation caveats

1. Stakeholder points reflect *perceived* heat, which includes factors beyond LST (humidity,
   wind, proximity to heat sources). A perfect hit rate would be surprising even with a correct
   detection.
2. Cold-spot points are fewer (39 against 55), so cold rates move in larger steps.
3. Participants did not distinguish day from night when marking points; the same points are
   tested against both periods.

[↑ Table of Contents](#nb05-toc)

In [ ]:
# ── §9 — Stakeholder hit rates and buffer sensitivity ─────────────────────────
# compute_hitrate.hit_rate_table() on exactly the inputs the dataset build publishes:
#   - NB05's L50 spot footprints (all_cities_spots.geojson, written in §7), and
#   - the cohort points (55 hot / 39 cold), rounded by the build's own function.
# Both come from scripts/build_master_summary.py, imported, never copied. This section
# writes no file: the published hit rates are written only by the build (§14), which
# checks them against HITRATE_NB05.
import sys as _sys
from pathlib import Path as _Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

_ROOT9 = globals().get('ROOT') or (_Path.cwd() if (_Path.cwd() / 'scripts').is_dir()
                                   else _Path.cwd().parent)
for _p in (_ROOT9, _ROOT9 / 'src'):
    if str(_p) not in _sys.path:
        _sys.path.insert(0, str(_p))
import config as _cfg9
from scripts.build_master_summary import prepare_cohort_points, prepare_spot_footprints
from scripts.compute_hitrate import hit_rate_table

# In a full run the points come from this session (§2c), before §10 writes them to the
# working GeoPackage; run on its own, this section reads them back from that GeoPackage.
_src9 = ({'hot': hotspot_gps_df, 'cold': coldspot_gps_df}
         if 'hotspot_gps_df' in globals() and 'coldspot_gps_df' in globals() else None)
pts9 = prepare_cohort_points(sources=_src9)
spots9 = prepare_spot_footprints()
HITRATE_NB05 = hit_rate_table(pts9, spots9)
HITRATE_NB05['cluster25'] = HITRATE_NB05['city'].map(
    pts9.drop_duplicates('city').set_index('city')['cluster25'])

_nh, _nc = int((pts9.spot_type == 'hot').sum()), int((pts9.spot_type == 'cold').sum())
assert (len(pts9), _nh, _nc) == (94, 55, 39), f"cohort points {len(pts9)} ({_nh}/{_nc})"
BASIS9 = (f"Cohort basis: {len(pts9)} points ({_nh} hot / {_nc} cold) marked by the "
          f"105-participant cohort in {pts9.city.nunique()} cities and "
          f"{pts9.cluster25.nunique()} clusters, at the released 3-decimal coordinates, "
          f"against {len(spots9)} L50 spot footprints. Hit rates are row-based: a point lying "
          f"within several buffered footprints counts once per footprint, so the denominator "
          f"grows with the buffer radius.")
print(BASIS9)
print(f"HITRATE_NB05: {len(HITRATE_NB05):,} rows (city x spot type x period x buffer)")

# ── tables at 0.5, 1, 5 and 10 km ─────────────────────────────────────────────
_B9 = [500, 1000, 5000, 10000]


def _hr_table(by):
    t = (HITRATE_NB05[HITRATE_NB05.buffer_m.isin(_B9)]
         .groupby(by + ['spot_type', 'period', 'buffer_m'])[['n_hit', 'n_total']].sum()
         .reset_index())
    t['cell'] = [f"{h}/{n} ({100 * h / n:.1f}%)" if n else "-"
                 for h, n in zip(t.n_hit, t.n_total)]
    t['buffer'] = (t.buffer_m / 1000).map(lambda v: f"{v:g} km")
    out = t.pivot_table(index=by + ['spot_type', 'period'], columns='buffer',
                        values='cell', aggfunc='first')
    return out[[f"{b / 1000:g} km" for b in _B9]]


_nat9 = (HITRATE_NB05[HITRATE_NB05.buffer_m == 500]
         .groupby(['spot_type', 'period'])[['n_hit', 'n_total']].sum())
_want9 = {('hot', 'day'): (4, 55), ('hot', 'night'): (3, 55),
          ('cold', 'day'): (4, 39), ('cold', 'night'): (0, 39)}
_got9 = {k: (int(_nat9.loc[k, 'n_hit']), int(_nat9.loc[k, 'n_total'])) for k in _want9}
assert _got9 == _want9, f"500 m national hit rates {_got9}"

print("\nNational hit rates, hits / rows (%), at 0.5, 1, 5 and 10 km. " + BASIS9)
display(_hr_table([]))
print("\nPer cluster. " + BASIS9)
display(_hr_table(['cluster25']))
print("\nPer city. " + BASIS9)
display(_hr_table(['city']))

# ── buffer-sensitivity charts (day / night x hot / cold, 0.5-10 km) ───────────
# NB05's own chart definition, unchanged in style; it now reads HITRATE_NB05 and is
# shown here only (no file: Notebook 07 draws the published figure from the dataset).
sens_df = HITRATE_NB05
STAKE_BUFFER_M = getattr(_cfg9, 'STAKEHOLDER_VALIDATION_BUFFER_M', 1000)
CLUSTER25_COLORS = getattr(_cfg9, 'CLUSTER25_COLORS', {})
_DAY_FILL = '#FFD700'    # gold/yellow — daytime total hits
_NIGHT_FILL = '#AAAAAA'  # grey        — nighttime total hits
_FILL_A = 0.25           # alpha — both fills visible when overlapping


def _draw_sensitivity(chart_type):
    """chart_type='cluster': one line per Cluster25 group; 'national': one aggregate line."""
    fig_s, (ax_hot, ax_cold) = plt.subplots(1, 2, figsize=(14, 6.5), sharey=False)
    fig_s.subplots_adjust(bottom=0.22, wspace=0.35, top=0.80)
    for _st, _ax in (('hot', ax_hot), ('cold', ax_cold)):
        _sub = sens_df[sens_df.spot_type == _st]
        _clr_spot = '#CC2222' if _st == 'hot' else '#1155AA'
        _ax.set_xlabel('Buffer distance (km)', fontsize=10)
        _ax.set_ylabel('Hit-rate (%)', fontsize=10)
        _ax.set_title(f'{"Hot" if _st == "hot" else "Cold"} Spots',
                      fontsize=12, fontweight='bold', color=_clr_spot)
        _ax.grid(True, alpha=0.2, ls='--')
        _ax.tick_params(labelsize=9)
        _ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
        _ax.set_ylim(-3, 108)
        _ax.axvline(STAKE_BUFFER_M / 1000, color='#777', lw=0.9, ls=':', zorder=1)
        _ax2 = _ax.twinx()
        _ax2.set_ylabel('Total hit count', fontsize=9, color='#333')
        _ax2.tick_params(axis='y', labelsize=8, labelcolor='#333')
        for _per, _fill_clr in (('day', _DAY_FILL), ('night', _NIGHT_FILL)):
            _psub = _sub[_sub.period == _per]
            if not len(_psub):
                continue
            _tot = (_psub.groupby('buffer_m')[['n_hit', 'n_total']].sum()
                    .reset_index().sort_values('buffer_m'))
            _ax2.fill_between(_tot['buffer_m'] / 1000, 0, _tot['n_hit'],
                              alpha=_FILL_A, color=_fill_clr, zorder=1)
            _ax2.plot(_tot['buffer_m'] / 1000, _tot['n_hit'], color=_fill_clr, lw=0.7,
                      alpha=0.6, zorder=1)
        if not len(_sub):
            _ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                     transform=_ax.transAxes, color='#aaa', fontsize=11)
            continue
        if chart_type == 'cluster':
            _cl_agg = (_sub.dropna(subset=['cluster25'])
                       .groupby(['buffer_m', 'cluster25', 'period'])[['n_hit', 'n_total']]
                       .sum().reset_index())
            _cl_agg['hit_rate_cl'] = _cl_agg['n_hit'] / _cl_agg['n_total']
            for _cl in sorted(_cl_agg.groupby('cluster25')['n_hit'].sum()
                              .loc[lambda s: s > 0].index.tolist()):
                _clr_cl = CLUSTER25_COLORS.get(int(_cl), '#888')
                for _per, _ls in (('day', '-'), ('night', '--')):
                    _cs = (_cl_agg[(_cl_agg.cluster25 == _cl) & (_cl_agg.period == _per)]
                           .sort_values('buffer_m'))
                    if len(_cs):
                        _ax.plot(_cs['buffer_m'] / 1000, _cs['hit_rate_cl'] * 100,
                                 color=_clr_cl, lw=1.5, ls=_ls, zorder=3)
        else:
            for _per, _ls, _lw in (('day', '-', 2.5), ('night', '--', 2.0)):
                _ns = (_sub[_sub.period == _per].groupby('buffer_m')[['n_hit', 'n_total']]
                       .sum().reset_index().sort_values('buffer_m'))
                if len(_ns):
                    _ax.plot(_ns['buffer_m'] / 1000, _ns['n_hit'] / _ns['n_total'] * 100,
                             color=_clr_spot, lw=_lw, ls=_ls, zorder=4)
    _leg = [plt.Line2D([0], [0], ls='-', color='#555', lw=1.3, label='Daytime'),
            plt.Line2D([0], [0], ls='--', color='#555', lw=1.3, label='Nighttime'),
            plt.Rectangle((0, 0), 1, 1, fc=_DAY_FILL, alpha=_FILL_A + 0.1,
                          label='Daytime total hits'),
            plt.Rectangle((0, 0), 1, 1, fc=_NIGHT_FILL, alpha=_FILL_A + 0.1,
                          label='Nighttime total hits')]
    if chart_type == 'cluster':
        _leg = [plt.Line2D([0], [0], color=CLUSTER25_COLORS.get(int(_cl), '#888'), lw=1.5,
                           label=f'C{int(_cl)}')
                for _cl in sorted(sens_df.dropna(subset=['cluster25'])
                                  .groupby('cluster25')['n_hit'].sum()
                                  .loc[lambda s: s > 0].index.tolist())] + _leg
    fig_s.legend(handles=_leg, loc='lower center', ncol=min(len(_leg), 8), fontsize=8.5,
                 framealpha=0.92, edgecolor='#bbb', bbox_to_anchor=(0.5, 0.01))
    _what = {'cluster': 'per Cluster25 group (solid = day, dashed = night)',
             'national': 'national aggregate, sum of hits / sum of rows'}[chart_type]
    fig_s.suptitle(f'Stakeholder hit rate by buffer distance, {_what}\n'
                   f'{len(pts9)} cohort points ({_nh} hot / {_nc} cold); hit rates are '
                   f'row-based; shaded = total hit count (gold = day, grey = night)',
                   fontsize=10.5, fontweight='bold')
    plt.show()


_draw_sensitivity('national')
_draw_sensitivity('cluster')

<a id="nb05-s10"></a>

## §10 — NB05 working GeoPackage, per contour level

> **This is NB05's working file, not the published dataset.** `master_thermal_spots.gpkg` is an
> input to the dataset build (contours and the stakeholder points) and to
> `scripts/extract_population_100m.py`. The published dataset is assembled from all notebooks in
> [§14](#nb05-s14).

### Overview

This section assembles `gdf_master`, one row per contour level, and writes it with the city
zones and the stakeholder points to NB05's working GeoPackage. All analytical
computations in §9 and §11 use the per-spot `gdf` built in §7; §10 only concerns
data packaging and does not alter results.

### `gdf_master` schema — one row per contour level

Three rows are produced per spot (L50 / L75 / L90). Each row carries the full
attribute set inherited from `gdf` (§7) plus the following additional columns:

| Column | Description |
|--------|-------------|
| `contour_id` | `{spot_id}_L50`, `_L75`, or `_L90` — unique per row |
| `contour_level` | `L50` · `L75` · `L90` |
| `area_km2` | Area of **this** contour polygon (km²) |
| `mean_lst` / `std_lst` / `max_lst` / `min_lst` | Zonal LST stats for this contour |
| `population` | GHS-POP 2025 population within this contour, summed at native 100 m |
| `study_year` | From `config.STUDY_YEAR`; append-not-replace across years |

### Multi-year append logic

Keyed by `(city, spot_type, period, contour_level, study_year)`. Running with a
different `STUDY_YEAR` in `config.py` appends new rows without overwriting existing
years. Set `FORCE_RERUN = True` to rebuild the current year.

### Who reads it

| Layer | Read by |
|-------|---------|
| `hotspot_contours` | dataset build (`spot_contours`), `scripts/extract_population_100m.py` |
| `city_zones` | `scripts/extract_population_100m.py` |
| `stakeholder_hotspot_gps`, `stakeholder_coldspot_gps` | dataset build (`stakeholder_spots_gps`, hit rates) |

Persistence (`n_persistent_events`, `longest_event_days`) is taken from **ERA5-Land**, not MODIS:
the P90 definition fixes the number of hot days per year, so the MODIS hot-day count carried no
extra information and was dropped.

**GeoPackage layers**

| Layer | Contents |
|-------|---------|
| `hotspot_contours` | All L50/L75/L90 contour polygons — filter `contour_level = 'L50'` for main spots |
| `city_zones` | UC / BRA / City-Region boundaries with cluster25 |
| `stakeholder_hotspot_gps` | Hotspot GPS validation points |
| `stakeholder_coldspot_gps` | Coldspot GPS validation points |

[↑ Table of Contents](#nb05-toc)


In [ ]:
# §10 — Master GDF & Export
# Builds gdf_master (per-contour-level rows), then exports to
#   GeoPackage (4 layers: hotspot_contours, city_zones, stakeholder hot/cold GPS)

from shapely.wkt import loads as _wkt_loads_m

MASTER_GPKG_PATH = RESULTS_DIR / 'master_thermal_spots.gpkg'


def _build_master_rows(base_gdf, study_year, tif_inv, existing_keys):
    """Melt base_gdf into per-contour-level rows (L50 / L75 / L90)."""
    rows = []
    for _, row in base_gdf.iterrows():
        _base = {k: v for k, v in row.to_dict().items()
                 if k not in {'geometry', 'geom_75_wkt', 'geom_90_wkt'}}
        _base['study_year'] = study_year
        _tif = tif_inv.get(row['city'], {}).get(row['period'])

        def _as_pop(v):
            """0.0 means nobody lives there; only a genuinely absent value is NaN.

            Written out rather than `float(v or nan)`, which silently turned every
            unpopulated contour into NaN because 0.0 is falsy -- 12 of the 441 rows.
            """
            return float('nan') if v is None or pd.isna(v) else float(v)

        for _lvl, _src, _pop_key in [
            ('L50', row.geometry,            None),
            ('L75', row.get('geom_75_wkt'),  'pop_75'),
            ('L90', row.get('geom_90_wkt'),  'pop_90'),
        ]:
            _key = (row['city'], row['spot_type'], row['period'], _lvl, study_year)
            if _key in existing_keys:
                continue

            if _lvl == 'L50':
                _poly = row.geometry
            elif isinstance(_src, str) and _src:
                try:
                    _poly = _wkt_loads_m(_src)
                except Exception:
                    continue
            else:
                continue

            if _poly is None or _poly.is_empty:
                continue

            # Convert WGS84 polygon to ESRI:102022 for stats/population
            _poly_102022 = (
                gpd.GeoSeries([_poly], crs=base_gdf.crs)
                .to_crs('ESRI:102022').iloc[0]
                if base_gdf.crs and str(base_gdf.crs) != 'ESRI:102022'
                else _poly
            )

            if _lvl == 'L50':
                _st = {k: row.get(k, float('nan'))
                       for k in ('mean_lst', 'std_lst', 'max_lst', 'min_lst')}
                _pop = _as_pop(row.get('population'))
            else:
                _st  = (zonal_stats(_poly_102022, _tif) if _tif else
                        dict(mean_lst=float('nan'), std_lst=float('nan'),
                             max_lst=float('nan'),  min_lst=float('nan')))
                _pop = _as_pop(row.get(_pop_key))


            rows.append({
                **_base,
                'contour_id':    f"{row['spot_id']}_{_lvl}",
                'contour_level': _lvl,
                'area_km2':      round(_poly_102022.area / 1e6, 4),
                **_st,
                'population':    _pop,
                'geometry':      _poly,  # keep in source CRS (WGS84 if from cache)
            })
    return rows


# ── Load existing master GDF keys ─────────────────────────────────────────────
_ex_master = gpd.GeoDataFrame()
_ex_keys   = set()
if MASTER_GPKG_PATH.exists() and not FORCE_RERUN:
    try:
        _ex_master = gpd.read_file(MASTER_GPKG_PATH, layer='hotspot_contours')
        if {'study_year', 'contour_level', 'city', 'spot_type', 'period'}.issubset(
                _ex_master.columns):
            _ex_keys = set(zip(
                _ex_master['city'], _ex_master['spot_type'],
                _ex_master['period'], _ex_master['contour_level'],
                _ex_master['study_year'].astype(int)))
            print(f"  master GDF on disk: {len(_ex_master)} rows "
                  f"across {_ex_master['study_year'].nunique()} year(s)")
    except Exception as _em:
        print(f"  ⚠ master GDF read error: {_em}")

if len(gdf) > 0:
    _new_master_rows = _build_master_rows(gdf, STUDY_YEAR, seviri_inv, _ex_keys)
else:
    _new_master_rows = []

if _new_master_rows:
    _new_gdf_m = gpd.GeoDataFrame(_new_master_rows, crs=gdf.crs or 'EPSG:4326')
    gdf_master = (gpd.GeoDataFrame(
        pd.concat([_ex_master, _new_gdf_m], ignore_index=True),
        crs=gdf.crs or 'EPSG:4326')
        if len(_ex_master) else _new_gdf_m)
    print(f"  {len(_new_master_rows)} new contour rows → {len(gdf_master)} total")
elif len(_ex_master):
    gdf_master = _ex_master
    print(f"  No new rows for {STUDY_YEAR} — gdf_master: {len(gdf_master)} rows cached")
else:
    gdf_master = gpd.GeoDataFrame()
    print("  ⚠ gdf_master is empty")

# ── Save GeoPackage ───────────────────────────────────────────────────────────
if len(gdf_master) and _new_master_rows:
    _gdf_m_4326 = gdf_master.to_crs('EPSG:4326') if str(gdf_master.crs) != 'EPSG:4326' else gdf_master

    # ── GeoPackage (5 layers) ─────────────────────────────────────────────────
    _gpkg = MASTER_GPKG_PATH
    if _gpkg.exists(): _gpkg.unlink()

    _gdf_m_4326.to_file(_gpkg, layer='hotspot_contours', driver='GPKG')
    print(f"  GPKG 'hotspot_contours': {len(gdf_master)} rows (L50/L75/L90)")

    _z_rows = []
    for _c in cities_with_sev:
        _zi = zones_dict.get(_c)
        if _zi is None: continue
        for _zn, _zk in (('urban_core','uc'), ('bra','bra'), ('city_region','cr')):
            _zg = _zi.get(_zk)
            if _zg is not None:
                _z_rows.append({'city': _c, 'zone': _zn,
                                 'cluster25': city_to_cluster.get(_c, -1),
                                 'geometry': _zg})
    if _z_rows:
        (gpd.GeoDataFrame(_z_rows, crs='ESRI:102022')
         .to_crs('EPSG:4326').to_file(_gpkg, layer='city_zones', driver='GPKG'))
        print(f"  GPKG 'city_zones': {len(_z_rows)} features")

    for _sdf, _lname in ((hotspot_gps_df, 'stakeholder_hotspot_gps'),
                          (coldspot_gps_df, 'stakeholder_coldspot_gps')):
        if len(_sdf):
            _sg = gpd.GeoDataFrame(_sdf.copy(), crs='EPSG:4326',
                    geometry=gpd.points_from_xy(_sdf['long_spot'], _sdf['lat_spot']))
            _sg.to_file(_gpkg, layer=_lname, driver='GPKG')
            print(f"  GPKG '{_lname}': {len(_sg)} points")

    print(f"  GPKG saved: {_gpkg.name}")


elif not _new_master_rows and len(gdf_master):
    print(f"  No new rows for {STUDY_YEAR} — exports not re-written (already up-to-date)")

print(f"\n✓ §10 export complete")
print(f"  Filter contour_level='L50'  for main spot polygons  |  study_year={STUDY_YEAR}")

<a id="nb05-s11"></a>

## §11 — Visualisations

This section produces six complementary figure types covering different scales of
analysis. All figures are saved as both SVG (lossless, scalable for publication) and
PDF (print-ready), both at 500 DPI unless stated otherwise.

### Figure guide

| Sub-section | Figure type | Scale | What it shows |
|-------------|-------------|-------|---------------|
| **§11a** | Per-city maps | City | LST raster (YlOrRd) with KDE contour overlays (50/75/90%); UC boundary; stakeholder GPS stars/triangles |
| **§11a2** | A4 publication grid | All 56 cities | 7 × 8 mini-map grid, one per city; sorted north-to-south by UC centroid latitude; day and night separate |
| **§11b** | Gi★ Z-score distributions | City / National | Violin + box plots of Z-score distributions; hot/cold split; identifies cities with bimodal distributions |
| **§11c** | KDE contour area by Cluster25 | Cluster | Grouped bar charts of spot area (km²) by Cluster25; Tier1 + Tier2 split; hot & cold × day & night |
| **§11d** | Population exposure half-donuts | City / Cluster / National | Half-donut chart: left = hot, right = cold; Ring 1 = spot type, Ring 2 = day/night, Ring 3 = L90/L75/L50 contour zone population; composite 4-column panel of all cluster donuts |
| **§11e** | National summary map | Nigeria | Proportional population circles over Cluster25 choropleth; wedge split Day/Night; hot and cold separate |
| **§11f** | Stakeholder perception by Cluster25 | Cluster | Grouped bars of Q60 (diurnal severity) and Q61 (10-yr warming trend) responses; bars sorted by cluster |

### Output files (§11 figures)

| File | Sub-section |
|------|------------|
| `{city}_hotspot_map.svg` | §11a per-city maps |
| `a4_grid_{day\|night}.svg` | §11a2 publication grid |
| `gistar_distributions.svg` | §11b Z-score distributions |
| `cluster_kde_area_{hot\|cold}.svg` | §11c KDE area by cluster |
| `{city\|cluster}_population_halfdonut.svg` | §11d individual donuts |
| `cluster_halfdonuts_composite.svg` | §11d 4-column cluster composite |
| `national_population_halfdonut.svg` | §11d national donut |
| `national_{hot\|cold}_summary_map.svg` | §11e national summary maps |
| `stakeholder_perception_cluster.svg` | §11f perception bars |

### Colour conventions

| Element | Colour | Rationale |
|---------|--------|-----------|
| Hot spot outlines | `#FF2400` (bright red) → `#7B0000` (dense core) | Red family — universal UHI convention |
| Cold spot outlines | `#00BFFF` (sky blue) → `#003D7A` (dense core) | Blue family — thermal contrast with hot |
| Urban Core boundary | `#888888` (medium grey) | Neutral; does not compete with spot colours |
| Cluster25 palette | `CLUSTER25_COLORS` from `config.py` | Physiographic landscape groupings |
| Path-effect halo | White (2–4.5 pt stroke behind outline) | Ensures visibility on any LST background |

### Map drawing order (zorder)

| Element | zorder |
|---------|--------|
| LST raster | 0 |
| Water bodies | 1 |
| Roads | 2 |
| UC boundary | 5 |
| KDE contour outlines | 7 |
| Stakeholder GPS | 9 |

[↑ Table of Contents](#nb05-toc)


In [ ]:
# §11a — City Hotspot & Coldspot Maps
# Hot spot KDE boundaries: BLACK (visible against YlOrRd raster)
# Cold spot KDE boundaries: bright CYAN (#00E5FF → #004466)
# Colorbar: VERTICAL, right side of each period panel, city-specific LST range
# Roads: DASHED lines (all types)
# Legend: SEPARATE standalone figure (not embedded on map)
# Cluster25: corner annotation using city_to_cluster lookup

from shapely.wkt          import loads as _wl
from shapely.geometry     import box as _box
from matplotlib.patheffects import withStroke as _PE11a
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as _mpa


def _to_102022_wkt(wkt_str, src_crs=None):
    """Load WKT — always in ESRI:102022 metres (string attrs are never reprojected)."""
    return _wl(wkt_str)

# Hot = black family (visible on red LST raster)
HOT_COLORS  = {0.50: '#000000', 0.75: '#333333', 0.90: '#555555'}
# Cold = bright cyan family (visible on yellow-orange LST raster)
COLD_COLORS = {0.50: '#00E5FF', 0.75: '#0095AA', 0.90: '#004466'}

# Project stakeholder GPS to ESRI:102022
_col_city = 'matched_city_y'
_hot_gps = gpd.GeoDataFrame(
    hotspot_gps_df, crs='EPSG:4326',
    geometry=gpd.points_from_xy(hotspot_gps_df['long_spot'],
                                hotspot_gps_df['lat_spot'])
).to_crs('ESRI:102022') if len(hotspot_gps_df) else gpd.GeoDataFrame()

_cold_gps = gpd.GeoDataFrame(
    coldspot_gps_df, crs='EPSG:4326',
    geometry=gpd.points_from_xy(coldspot_gps_df['long_spot'],
                                coldspot_gps_df['lat_spot'])
).to_crs('ESRI:102022') if len(coldspot_gps_df) else gpd.GeoDataFrame()


def _load_ctx(city):
    folder = _folder.get(city, city.replace(' ', '_'))
    slug   = folder
    cdir   = AOI_DIR / folder
    odir   = cdir / 'osm'

    def _rd(path):
        try:
            g = gpd.read_file(path)
            return g.to_crs('ESRI:102022') if not g.empty else None
        except Exception:
            return None

    water = _rd(cdir / f'{slug}_water_bodies_102022.geojson')
    if water is None:
        water = _rd(cdir / f'{slug}_water_bodies_4326.geojson')
    roads  = _rd(odir / f'{slug}_roads_4326.geojson')
    places = _rd(odir / f'{slug}_places_4326.geojson')
    return water, roads, places


def _clip_to_extent(gdf, x0, x1, y0, y1):
    if gdf is None or gdf.empty:
        return None
    try:
        bbox    = _box(x0, y0, x1, y1)
        clipped = gdf[gdf.geometry.intersects(bbox)].copy()
        return clipped if not clipped.empty else None
    except Exception:
        return gdf


_ROAD_TYPES = ['motorway', 'trunk', 'primary', 'secondary', 'tertiary']
_ROAD_LW    = {'motorway': 1.0, 'trunk': 0.90, 'primary': 0.75,
               'secondary': 0.55, 'tertiary': 0.40}
_ROAD_CLR   = '#999999'


def _hwy_type(val):
    if isinstance(val, list):
        val = val[0] if val else ''
    return str(val).split(';')[0].strip()


def _add_north_arrow(ax):
    ax.annotate('', xy=(0.93, 0.86), xytext=(0.93, 0.78),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='black', lw=0.9))
    ax.text(0.93, 0.88, 'N', transform=ax.transAxes,
            ha='center', va='bottom', fontsize=6, fontweight='bold', color='black')


# ── Standalone legend (save once, not embedded on maps) ──────────────────────
def _save_hotspot_legend():
    fig_leg, ax_leg = plt.subplots(figsize=(4.5, 3.2))
    ax_leg.axis('off')
    _handles = [
        plt.Line2D([0], [0], color='#000000', lw=2.5,
                   label='Hot spot 50% KDE'),
        plt.Line2D([0], [0], color='#333333', lw=1.5, ls='--',
                   label='Hot spot 75% KDE (dashed)'),
        plt.Line2D([0], [0], color='#555555', lw=1.0, ls=':',
                   label='Hot spot 90% dense core (dotted)'),
        plt.Line2D([0], [0], color='#00E5FF', lw=2.5,
                   label='Cold spot 50% KDE'),
        plt.Line2D([0], [0], color='#0095AA', lw=1.5, ls='--',
                   label='Cold spot 75% KDE (dashed)'),
        plt.Line2D([0], [0], color='#004466', lw=1.0, ls=':',
                   label='Cold spot 90% dense core (dotted)'),
        plt.Line2D([0], [0], color='#888888', lw=2.5,
                   label='Urban Core boundary'),
        _mpa.Patch(facecolor='#C6E8FF', edgecolor='#7EC8E3', lw=0.5,
                   label='Water bodies'),
        plt.Line2D([0], [0], color='#999999', lw=0.9, ls='--',
                   label='Roads'),
        plt.Line2D([0], [0], marker='*', color='gold', ms=9,
                   mec='black', mew=0.7, ls='none',
                   label='Stakeholder hotspot GPS'),
        plt.Line2D([0], [0], marker='v', color='#00E5FF', ms=8,
                   mec='#0095AA', mew=0.7, ls='none',
                   label='Stakeholder coldspot GPS'),
    ]
    ax_leg.legend(handles=_handles, fontsize=8, loc='center',
                  frameon=True, framealpha=0.97, edgecolor='#ccc',
                  title='Map Legend', title_fontsize=9,
                  ncol=1, borderpad=0.8, labelspacing=0.55)
    fig_leg.tight_layout()
    for ext in ('svg', 'pdf'):
        fig_leg.savefig(FIGS_DIR / f'hotspot_map_legend.{ext}',
                        bbox_inches='tight', dpi=300)
    plt.close(fig_leg)
    print('  Standalone legend saved: hotspot_map_legend.svg')

_save_hotspot_legend()


def plot_city_hotspot_map(city, show=True):
    zi = zones_dict.get(city)
    if zi is None:
        return
    # gdf loaded from GeoJSON is in EPSG:4326; reproject to ESRI:102022 for plotting
    _gdf_p = gdf.to_crs('ESRI:102022') if len(gdf) and str(gdf.crs) != 'ESRI:102022' else gdf
    _gdf_src_crs = gdf.crs if len(gdf) else 'EPSG:4326'
    water_gdf, roads_gdf, _ = _load_ctx(city)

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.0), dpi=150)
    fig.suptitle(f'{city}  |  C{city_to_cluster.get(city, "?")}  — UHI Hot & Cold Spots',
                 fontsize=11, fontweight='bold')

    for ax, period in zip(axes, PERIODS):
        res = hotspot_results.get(city, {}).get(period)
        if res is None:
            ax.set_title(f'{period.capitalize()} — no data')
            ax.axis('off')
            continue

        arr = res['arr']
        tf  = RioAffine(*res['transform'][:6])
        nr, nc = arr.shape
        x0, y1 = tf * (0, 0)
        x1, y0 = tf * (nc, nr)

        # ── LST raster (zorder=0) — per-city, per-period colour range ─────────
        _valid = arr[np.isfinite(arr) & (arr > -50) & (arr < 100)]
        _vmin = float(np.percentile(_valid, 2))  if len(_valid) else 20.
        _vmax = float(np.percentile(_valid, 98)) if len(_valid) else 50.
        im = ax.imshow(arr, extent=[x0, x1, y0, y1], origin='upper',
                       cmap='YlOrRd', alpha=0.82, aspect='equal',
                       vmin=_vmin, vmax=_vmax, zorder=0)

        # Vertical colorbar — right side of this period's panel
        _div = make_axes_locatable(ax)
        _cax = _div.append_axes('right', size='4%', pad=0.04)
        _cb  = plt.colorbar(im, cax=_cax)
        _cb.set_label('LST (°C)', fontsize=8, labelpad=4)
        _cb.ax.tick_params(labelsize=7)
        _cb.ax.yaxis.set_major_formatter(
            plt.matplotlib.ticker.FormatStrFormatter('%.1f'))

        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)

        water_c = _clip_to_extent(water_gdf, x0, x1, y0, y1)
        roads_c = _clip_to_extent(roads_gdf, x0, x1, y0, y1)

        # ── Water (zorder=1) ──────────────────────────────────────────────────
        if water_c is not None:
            water_c.plot(ax=ax, color='#C6E8FF', edgecolor='#7EC8E3',
                         lw=0.5, zorder=1)

        # ── Roads — DASHED (zorder=2) ─────────────────────────────────────────
        if roads_c is not None and 'highway' in roads_c.columns:
            hwy_s = roads_c['highway'].apply(_hwy_type)
            for rtype in _ROAD_TYPES:
                seg = roads_c[hwy_s == rtype]
                if len(seg):
                    seg.plot(ax=ax, color=_ROAD_CLR,
                             lw=_ROAD_LW[rtype], ls='--',
                             zorder=2)

        # ── KDE contours (zorder=7) ───────────────────────────────────────────
        city_data = _gdf_p[(_gdf_p.city == city) & (_gdf_p.period == period)] \
                    if len(_gdf_p) else gpd.GeoDataFrame()
        for spot_type, lvl_colors in (('hot', HOT_COLORS), ('cold', COLD_COLORS)):
            spots = city_data[city_data.spot_type == spot_type] \
                    if len(city_data) else gpd.GeoDataFrame()
            for _, srow in spots.iterrows():
                # 50% contour — thick solid, no halo
                gpd.GeoSeries([srow.geometry], crs='ESRI:102022').boundary.plot(
                    ax=ax, edgecolor=lvl_colors[0.50], lw=2.8, zorder=7)
                # 75% and 90% contours
                for wkt_col, lvl, lw, ls in (
                    ('geom_75_wkt', 0.75, 1.8, '--'),
                    ('geom_90_wkt', 0.90, 1.2, ':'),
                ):
                    wkt = srow.get(wkt_col)
                    if wkt and isinstance(wkt, str):
                        try:
                            gpd.GeoSeries(
                                [_to_102022_wkt(wkt)],
                                crs='ESRI:102022').boundary.plot(
                                ax=ax, edgecolor=lvl_colors[lvl],
                                lw=lw, ls=ls, zorder=7)
                        except Exception:
                            pass

        # ── UC boundary (zorder=5) ────────────────────────────────────────────
        gpd.GeoSeries([zi['uc']], crs='ESRI:102022').boundary.plot(
            ax=ax, color='#888888', lw=3.2, zorder=5)

        # ── Stakeholder GPS (zorder=9) ────────────────────────────────────────
        if len(_hot_gps) and _col_city in _hot_gps.columns:
            pts = _hot_gps[_hot_gps[_col_city] == city]
            if len(pts):
                pts.plot(ax=ax, marker='*', color='gold', markersize=13,
                         edgecolors='black', linewidths=0.7, zorder=9)
        if len(_cold_gps) and _col_city in _cold_gps.columns:
            pts = _cold_gps[_cold_gps[_col_city] == city]
            if len(pts):
                pts.plot(ax=ax, marker='v', color='#00E5FF', markersize=9,
                         edgecolors='#0095AA', linewidths=0.7, zorder=9)

        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)

        # ── Cluster25 corner annotation (top-left) ────────────────────────────
        _cl_id = city_to_cluster.get(city, '?')
        ax.text(0.02, 0.98, f'C{_cl_id}', transform=ax.transAxes,
                fontsize=9, fontweight='bold', va='top', ha='left',
                color='#111', zorder=11,
                bbox=dict(boxstyle='round,pad=0.25', fc='white',
                          ec='#aaa', alpha=0.88, lw=0.8))

        p90 = thresholds.get(city, {}).get(period, {}).get('p90', float('nan'))
        ax.set_title(f'{period.capitalize()}  (P90 = {p90:.1f} °C)', fontsize=9)
        _add_north_arrow(ax)
        ax.set_axis_off()

    plt.tight_layout()
    fig.savefig(FIGS_DIR / f'{city}_hotspot_map.svg', bbox_inches='tight', dpi=500)
    fig.savefig(FIGS_DIR / f'{city}_hotspot_map.pdf', bbox_inches='tight', dpi=500)
    if show:
        plt.show()
    else:
        plt.close(fig)


# Sort north-to-south by UC centroid latitude
_city_lat = {}
for _c in cities_with_sev:
    _zi = zones_dict.get(_c)
    if _zi is None:
        _city_lat[_c] = 0.0
        continue
    try:
        _lat = (gpd.GeoSeries([_zi['uc']], crs='ESRI:102022')
                .to_crs('EPSG:4326').centroid.iloc[0].y)
    except Exception:
        _lat = 0.0
    _city_lat[_c] = _lat
_cities_by_lat = sorted(cities_with_sev, key=lambda c: -_city_lat.get(c, 0.0))
shown = 0
for city in _cities_by_lat:
    plot_city_hotspot_map(city, show=(shown == 0))
    shown += 1
print(f"§11a maps: {shown} cities saved  →  {FIGS_DIR}")
print("  Hot KDE=black · Cold KDE=cyan · Roads=dashed · Cbar=vertical-right")


<a id="nb05-s11a2"></a>

## §11a2 — Publication A4 Grid: All 56 Cities

### Purpose

The A4 grid provides a **synoptic overview** of hotspot patterns across all cities for a
single page in the thesis or appendix. At 7 × 8 = 56 panels on an A4 sheet
(8.27 × 11.69 inches), each panel is approximately 1.18 × 1.46 inches. Design choices
reflect this extreme space constraint.

### Sorting order

Cities are sorted **north-to-south by UC centroid latitude** (EPSG:4326). This geographic
ordering allows the reader to scan column by column and perceive latitudinal gradients in
hotspot characteristics (e.g., larger cold spots in northern arid cities, larger heat cores
in southern humid cities).

### Contextual layers

Only **primary roads** (motorway, trunk, primary) are shown — no water bodies, no secondary
streets. At this scale, minor roads would obscure the KDE contours rather than add context.

### Font sizes (A4-optimised)

All text elements are tuned for the 1.18 × 1.46 inch panel size:

| Element | Font size |
|---------|-----------|
| Panel title (`{city} C{cluster}`) | 5.5 pt |
| Period footnote | 4.5 pt |
| LST colorbar label | 6 pt |
| Figure legend | 6 pt |
| North arrow label | 5.5 pt |
| Figure super-title | 8 pt |

### Legend and scale bar

A single figure-level legend (bottom centre) is shared across all 56 panels to avoid
wasting space on repeated per-panel legends. The legend shows:
- Hot spot 50% contour sample
- Cold spot 50% contour sample
- Primary road sample
- Urban Core boundary sample
- Stakeholder GPS markers (hot ★ / cold ▼)

Each panel shows its city name and Cluster25 ID in the title. A north arrow (top right
of figure) orients the reader.

[↑ Table of Contents](#nb05-toc)


In [ ]:
# §11a2 — A4 publication grid: all 56 cities
from matplotlib.patheffects import withStroke as _PE11a2
# Layout: 7 columns × 8 rows = 56 slots; sorted by SUHII descending per period
# Layout: 7 columns × 8 rows = 56 slots; sorted by Cluster25 then city name
# Hot KDE: BLACK borders (same as §11a) | Cold KDE: bright CYAN
# Contextual: primary roads only (faint grey, zorder=2) — no water, no places
# UC boundary zorder=5 | KDE contours zorder=7 | GPS zorder=9
# LST colorbar: period-specific YlOrRd scale (day vs night computed separately)
# One figure-level north arrow (top-right) | One legend (bottom centre)

# Sort north-to-south by UC centroid latitude
_city_lat_a2 = {}
for _c in cities_with_sev:
    _zi = zones_dict.get(_c)
    if _zi is None:
        _city_lat_a2[_c] = 0.0
        continue
    try:
        _city_lat_a2[_c] = (gpd.GeoSeries([_zi['uc']], crs='ESRI:102022')
                            .to_crs('EPSG:4326').centroid.iloc[0].y)
    except Exception:
        _city_lat_a2[_c] = 0.0
_city_order = sorted(cities_with_sev, key=lambda c: -_city_lat_a2.get(c, 0.0))

# Pre-load contextual layers (roads only needed for A4)
_ctx_cache = {}
for _c in _city_order:
    _ctx_cache[_c] = _load_ctx(_c)   # (water_gdf, roads_gdf, places_gdf)

_PRIMARY_ROADS = {'motorway', 'trunk', 'primary'}


def plot_a4_grid(period='day', show=True):
    # Reproject gdf to ESRI:102022 for metric plotting
    _gdf_a2 = gdf.to_crs('ESRI:102022') if len(gdf) and str(gdf.crs) != 'ESRI:102022' else gdf
    _gdf_a2_crs = gdf.crs if len(gdf) else 'EPSG:4326'
    # Sort: cluster groups ordered by northernmost member; within each cluster north-to-south
    # Step 1: for each cluster find the latitude of its most-northern city
    _cl_north_lat = {}
    for _c in cities_with_sev:
        _cl = city_to_cluster.get(_c, 999)
        _lat = _city_lat_a2.get(_c, 0.0)
        if _cl not in _cl_north_lat or _lat > _cl_north_lat[_cl]:
            _cl_north_lat[_cl] = _lat
    # Step 2: sort cities — primary key = cluster-group position (north first),
    #         secondary = within-cluster north-to-south latitude
    _city_order_p = sorted(
        cities_with_sev,
        key=lambda c: (
            -_cl_north_lat.get(city_to_cluster.get(c, 999), 0.0),  # cluster group order
            city_to_cluster.get(c, 999),                            # stable cluster tiebreak
            -_city_lat_a2.get(c, 0.0),                             # within cluster: N→S
        )
    )
    # A4 publication grid of all 56 city hotspot maps for one period
    NCOLS, NROWS = 7, 8
    FIG_W, FIG_H = 8.27, 11.69

    fig, axes = plt.subplots(NROWS, NCOLS,
                              figsize=(FIG_W, FIG_H),
                              gridspec_kw=dict(hspace=0.10, wspace=0.05))
    axes_flat = axes.ravel()

    # Period-specific shared LST colour scale (YlOrRd, 2nd-98th pct)
    _all_vals = []
    for city in _city_order_p:
        res = hotspot_results.get(city, {}).get(period, {})
        if res:
            arr  = res.get('arr', np.array([]))
            valid = arr[np.isfinite(arr) & (arr > -50) & (arr < 100)]
            if valid.size:
                _all_vals.extend(valid.tolist())
    vmin = np.percentile(_all_vals, 2)  if _all_vals else 20
    vmax = np.percentile(_all_vals, 98) if _all_vals else 50

    HOT_C  = '#000000'  # black — matches §11a per-city maps
    COLD_C = '#00E5FF'  # bright cyan — matches §11a

    for idx, city in enumerate(_city_order_p):
        ax  = axes_flat[idx]
        zi  = zones_dict.get(city)
        res = hotspot_results.get(city, {}).get(period)
        _, roads_gdf, _ = _ctx_cache.get(city, (None, None, None))

        cl = city_to_cluster.get(city, '?')
        ax.set_title(f'{city[:13]}  C{cl}', fontsize=5.5, pad=2,
                     fontweight='bold', color='#111')

        if res is None or zi is None:
            ax.set_facecolor('#EEEEEE')
            ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                    fontsize=4.5, color='grey', transform=ax.transAxes)
            ax.axis('off')
            continue

        arr = res['arr']
        tf  = RioAffine(*res['transform'][:6])
        nr, nc = arr.shape
        x0, y1 = tf * (0, 0)
        x1, y0 = tf * (nc, nr)

        # LST raster — yellow→orange→red (zorder=0)
        ax.imshow(arr, extent=[x0, x1, y0, y1], origin='upper',
                  cmap='YlOrRd', vmin=vmin, vmax=vmax,
                  alpha=0.85, aspect='equal', zorder=0)

        # Lock axes to raster extent
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)

        # Primary roads only — faint grey (zorder=2)
        if roads_gdf is not None and len(roads_gdf) and 'highway' in roads_gdf.columns:
            hwy_s = roads_gdf['highway'].apply(_hwy_type)
            prim  = roads_gdf[hwy_s.isin(_PRIMARY_ROADS)]
            # Clip to raster bbox
            prim  = _clip_to_extent(prim, x0, x1, y0, y1)
            if prim is not None and len(prim):
                prim.plot(ax=ax, color='#DDDDDD', lw=0.2, alpha=0.7, zorder=2)

        # KDE contours L50 / L75 / L90 (solid lines, thinner inward)
        if len(gdf):
            from shapely.wkt import loads as _wkt_l_a2
            city_spots = _gdf_a2[(_gdf_a2.city == city) & (_gdf_a2.period == period)]
            for st, clr in (('hot', HOT_C), ('cold', COLD_C)):
                for _, row in city_spots[city_spots.spot_type == st].iterrows():
                    try:
                        # L50 — thickest solid, no halo
                        gpd.GeoSeries([row.geometry], crs='ESRI:102022').boundary.plot(
                            ax=ax, edgecolor=clr, lw=1.5, alpha=0.95, zorder=7)
                        # L75 — medium
                        _w75 = row.get('geom_75_wkt')
                        if _w75 and isinstance(_w75, str):
                            try:
                                _g75 = gpd.GeoSeries([_wkt_l_a2(_w75)], crs=_gdf_a2_crs).to_crs('ESRI:102022')
                                _g75.boundary.plot(ax=ax, edgecolor=clr, lw=0.9, alpha=0.80, zorder=7)
                            except Exception: pass
                        # L90 — thinnest
                        _w90 = row.get('geom_90_wkt')
                        if _w90 and isinstance(_w90, str):
                            try:
                                _g90 = gpd.GeoSeries([_wkt_l_a2(_w90)], crs=_gdf_a2_crs).to_crs('ESRI:102022')
                                _g90.boundary.plot(ax=ax, edgecolor=clr, lw=0.55, alpha=0.65, zorder=7)
                            except Exception: pass
                    except Exception:
                        pass

        # UC boundary (zorder=5)
        gpd.GeoSeries([zi['uc']], crs='ESRI:102022').boundary.plot(
            ax=ax, color='#888888', lw=1.5,               zorder=5)

        # Stakeholder GPS (zorder=9)
        for gps_df, mkr, mclr, mecol, msize in (
            (_hot_gps,  '*', 'gold', 'black',   4.5),
            (_cold_gps, 'v', '#00E5FF', '#0095AA', 3.5),
        ):
            if len(gps_df) and _col_city in gps_df.columns:
                pts = gps_df[gps_df[_col_city] == city]
                if len(pts):
                    pts.plot(ax=ax, marker=mkr, color=mclr, markersize=msize,
                             edgecolors=mecol, linewidths=0.4, zorder=9)

        # Restore axes limits after GeoDataFrame.plot calls
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)
        ax.axis('off')

    # Hide unused slots
    for idx in range(len(_city_order_p), NCOLS * NROWS):
        axes_flat[idx].axis('off')

    # Shared LST colorbar — horizontal, bottom of figure
    _sm = plt.cm.ScalarMappable(cmap='YlOrRd',
                                norm=plt.Normalize(vmin=vmin, vmax=vmax))
    _sm.set_array([])
    _cbar_ax = fig.add_axes([0.18, 0.015, 0.65, 0.012])
    cbar = fig.colorbar(_sm, cax=_cbar_ax, orientation='horizontal')
    cbar.set_label('LST (°C)', fontsize=7)
    cbar.ax.tick_params(labelsize=5.5)

    # Legend (bottom centre) — roads + KDE + UC + GPS
    _leg_h = [
        mpatches.Patch(edgecolor=HOT_C,  facecolor='none', lw=1.5,
                       label='Hot spot (KDE 50%)'),
        mpatches.Patch(edgecolor=COLD_C, facecolor='none', lw=1.5,
                       label='Cold spot (KDE 50%)'),
        plt.Line2D([0], [0], color='#DDDDDD', lw=1.2,
                   label='Primary roads'),
        mpatches.Patch(edgecolor='black', facecolor='none', lw=0.6,
                       label='Urban Core'),
        plt.Line2D([0], [0], marker='*', color='gold', ms=5, ls='',
                   markeredgecolor='black', markeredgewidth=0.6,
                   label='Stakeholder hotspot'),
        plt.Line2D([0], [0], marker='v', color='#00E5FF', ms=4, ls='',
                   markeredgecolor='#0095AA', markeredgewidth=0.6,
                   label='Stakeholder coldspot'),
    ]
    # ── Save §11a2 standalone legend ─────────────────────────────────────────
    if period == 'day':   # save once (legend is period-agnostic)
        _fig_leg2, _ax_leg2 = plt.subplots(figsize=(7.5, 1.0))
        _ax_leg2.axis('off')
        _ax_leg2.legend(handles=_leg_h,
                        loc='center', fontsize=7, frameon=True,
                        framealpha=0.97, edgecolor='#ccc',
                        title='Map legend (§11a2)', title_fontsize=8,
                        ncol=len(_leg_h))
        _fig_leg2.tight_layout()
        _fig_leg2.savefig(FIGS_DIR / 'a4_grid_legend.svg', bbox_inches='tight', dpi=300)
        plt.close(_fig_leg2)
        print('  §11a2 standalone legend saved.')
    # (not embedded — uncomment below to add back)
    # fig.legend(handles=_leg_h, loc='lower center', ncol=3, fontsize=6,
    #            bbox_to_anchor=(0.46, 0.001), framealpha=0.95,
    #            edgecolor='#aaa', handlelength=1.5)

    # North arrow — compact figure-level inset at top-right
    _na = fig.add_axes([0.932, 0.962, 0.020, 0.020])
    _na.annotate('', xy=(0.5, 0.75), xytext=(0.5, 0.25),
                 xycoords='axes fraction', textcoords='axes fraction',
                 arrowprops=dict(arrowstyle='->', color='black', lw=0.8))
    _na.text(0.5, 0.92, 'N', ha='center', va='top', fontsize=5.5,
             fontweight='bold', transform=_na.transAxes, color='black')
    _na.axis('off')

    fig.suptitle(
        f'Urban Heat Island Hot & Cold Spots — All 56 Nigerian Cities '
        f'({period.capitalize()} Period)\n'
        f'KDE 50% contours · Cluster25 colour legend in titles · '
        f'★ = stakeholder-indicated hotspot',
        fontsize=8.0, fontweight='bold', y=0.998)

    plt.subplots_adjust(left=0.01, right=0.92, top=0.97, bottom=0.06)

    stem    = f'all_cities_hotspot_A4_{period}'
    pdf_out = FIGS_DIR / f'{stem}.pdf'
    png_out = FIGS_DIR / f'{stem}.png'
    fig.savefig(pdf_out, format='pdf', dpi=300, bbox_inches='tight')
    fig.savefig(png_out, format='png', dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    print(f"§11a2 A4 grid ({period}) saved:")
    print(f"  PDF → {pdf_out.name}  (vector, print-ready)")
    print(f"  PNG → {png_out.name}  (300 dpi)")

# Generate Day and Night A4 grids
plot_a4_grid('day',   show=True)
plot_a4_grid('night', show=False)


In [ ]:
# §11b — Gi★ Z-score distributions (matching old notebook format)
from scipy import stats as _sp_stats

def plot_gistar_dist(city, show=True):
    city_data = hotspot_results.get(city, {})
    if not city_data: return

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    fig.suptitle(f'{city}: Gi★ Z-Score Distributions',
                 fontsize=13, fontweight='bold', y=0.98)

    for idx, (ax, period) in enumerate(zip(axes, PERIODS)):
        res = city_data.get(period)
        if res is None:
            ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                    fontsize=13, transform=ax.transAxes)
            ax.set_title(period.capitalize(), fontsize=12, fontweight='bold')
            continue

        z_grid = res.get('z_grid')
        if z_grid is None: ax.axis('off'); continue
        z_scores = z_grid[np.isfinite(z_grid)].ravel()
        if not len(z_scores): ax.axis('off'); continue

        n        = len(z_scores)
        mean_z   = float(np.mean(z_scores))
        std_z    = float(np.std(z_scores))
        skewness = float(_sp_stats.skew(z_scores))
        kurtosis = float(_sp_stats.kurtosis(z_scores))
        hot_95   = int(np.sum(z_scores >  1.96)); hot_99  = int(np.sum(z_scores >  2.58))
        cold_95  = int(np.sum(z_scores < -1.96)); cold_99 = int(np.sum(z_scores < -2.58))

        # Density histogram
        ax.hist(z_scores, bins=50, density=True,
                alpha=0.65, color='#C8D8E8', edgecolor='#4a4a4a',
                linewidth=0.5, label='Observed distribution')

        # Theoretical N(0,1) curve
        x_norm = np.linspace(z_scores.min(), z_scores.max(), 300)
        ax.plot(x_norm, _sp_stats.norm.pdf(x_norm, 0, 1),
                'k--', lw=2, label='N(0,1) Theoretical')

        # Threshold lines ±1.96 and ±2.58
        ax.axvline( 1.96, color='#FF8C00', ls='--', lw=2.0, alpha=0.95, label='±1.96 (95% CI)')
        ax.axvline(-1.96, color='#FF8C00', ls='--', lw=2.0, alpha=0.95)
        ax.axvline( 2.58, color='#CC0000', ls=':',  lw=2.0, alpha=0.95, label='±2.58 (99% CI)')
        ax.axvline(-2.58, color='#0000CC', ls=':',  lw=2.0, alpha=0.95)

        # Shaded tails
        ax.axvspan( 1.96, float(z_scores.max()), alpha=0.22, color='#FF4500', label='Hot spots (Z > 1.96)')
        ax.axvspan(float(z_scores.min()), -1.96, alpha=0.22, color='#1565C0', label='Cold spots (Z < −1.96)')

        ax.set_xlabel('Gi★ Z-Score', fontsize=11)
        if idx == 0:
            ax.set_ylabel('Probability Density', fontsize=11)
        ax.set_title(period.capitalize(), fontsize=12, fontweight='bold', pad=8)
        ax.grid(True, alpha=0.25, ls='--')
        ax.legend(loc='upper right', fontsize=7.5)

        stats_txt = (
            f'Mean:  {mean_z:.3f}\n'
            f'Std:   {std_z:.3f}\n'
            f'Skew:  {skewness:.3f}\n'
            f'Kurt:  {kurtosis:.3f}\n\n'
            f'Hot spots:\n'
            f'  Z > 1.96 : {100*hot_95/n:.1f}% ({hot_95:,})\n'
            f'  Z > 2.58 : {100*hot_99/n:.1f}% ({hot_99:,})\n\n'
            f'Cold spots:\n'
            f'  Z < −1.96 : {100*cold_95/n:.1f}% ({cold_95:,})\n'
            f'  Z < −2.58 : {100*cold_99/n:.1f}% ({cold_99:,})'
        )
        ax.text(0.02, 0.98, stats_txt, transform=ax.transAxes,
                fontsize=7.5, va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                          alpha=0.88, edgecolor='gray'))

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(FIGS_DIR / f'{city}_gistar_dist.svg', bbox_inches='tight', dpi=500)
    fig.savefig(FIGS_DIR / f'{city}_gistar_dist.pdf', bbox_inches='tight', dpi=500)
    if show: plt.show()
    else: plt.close(fig)

shown = 0
for city in cities_with_sev:
    plot_gistar_dist(city, show=(shown == 0)); shown += 1
print(f"§11b Z-score distribution plots saved for {shown} cities")

# §11c — Hotspot Area by Contour Level & Cluster25
# All clusters shown (union across both periods); total area (sum), not median
if len(gdf) > 0:
    # Union of clusters across both periods so empty bars appear
    all_hot_clusters = sorted(gdf[gdf.spot_type == 'hot']['cluster25'].dropna().unique().astype(int))

    fig, axes = plt.subplots(1, 2, figsize=(11.69, 5.85), sharey=True)
    CLRS = ['#E07B00', '#c45a00', '#7b3200']

    for ax, period in zip(axes, PERIODS):
        sub = gdf[(gdf.spot_type == 'hot') & (gdf.period == period) & (gdf.tier == 1)]
        x = np.arange(len(all_hot_clusters)); w = 0.25

        for i, (col, lbl) in enumerate([('area_km2_50', '50%'),
                                         ('area_km2_75', '75%'),
                                         ('area_km2_90', '90%')]):
            vals = []
            for cl in all_hot_clusters:
                cl_sub = sub[sub.cluster25 == cl]
                vals.append(float(cl_sub[col].sum()) if len(cl_sub) else 0.0)
            ax.bar(x + i * w, vals, width=w * 0.9, color=CLRS[i], alpha=0.85, label=f'KDE {lbl}')

        ax.set_xticks(x + w)
        ax.set_xticklabels([f'C{c}' for c in all_hot_clusters], rotation=45, ha='right', fontsize=8)
        ax.set_title(f'{period.capitalize()} (Tier-1 spots only)', fontsize=10)
        ax.legend(fontsize=8)
        ax.set_ylabel('Total area (km²)')

    fig.suptitle('Hotspot Area by Contour Level & Cluster25'
                 '(total km²; all clusters shown, empty = 0)', fontsize=11)
    plt.tight_layout()
    fig.savefig(FIGS_DIR / 'hotspot_area_contour_cluster.svg', bbox_inches='tight', dpi=500)
    fig.savefig(FIGS_DIR / 'hotspot_area_contour_cluster.pdf', bbox_inches='tight', dpi=500)
    plt.show()
    print("§11c saved")


In [ ]:
# §11c — KDE Contour Area by Cluster25  (Tier1 + Tier2 | Hot & Cold | Day & Night)
# Stacked bars: L90 dense core / L75 inner zone / L50 outer extent
# Reference palette: cluster25_reference_palette.csv

import matplotlib.patches as _mp11c

_CLUST_CLR_11c = CLUSTER25_COLORS

if len(gdf) == 0:
    print('  GDF empty — skipping §11c')
else:
    _all_cl = sorted(gdf['cluster25'].dropna().unique().astype(int))

    def _build_area_df(spot_type_filter, period_filter):
        rows = []
        for cl in _all_cl:
            sub = gdf[(gdf.cluster25==cl) & (gdf.spot_type==spot_type_filter)
                      & (gdf.period==period_filter)]
            a50 = float(sub['area_km2_50'].sum()) if len(sub) else 0.
            a75 = float(sub['area_km2_75'].fillna(0).sum()) if len(sub) else 0.
            a90 = float(sub['area_km2_90'].fillna(0).sum()) if len(sub) else 0.
            rows.append({'cluster': cl, 'a90': a90,
                          'a75_ex': max(0, a75 - a90),
                          'a50_ex': max(0, a50 - a75)})
        return pd.DataFrame(rows)

    for spot_type, fig_name in (('hot', 'hotspot_area_contour_cluster.svg'),
                                 ('cold', 'coldspot_area_contour_cluster.svg')):
        fig11c, axes11c = plt.subplots(1, 2, figsize=(11.69, 5.5),
                                        sharey=False)
        fig11c.suptitle(
            f'{"Hot" if spot_type=="hot" else "Cold"}spot KDE Area by Cluster25 '
            f'(Tier1 + Tier2 combined)',
            fontsize=11, fontweight='bold')

        for ax11c, period in zip(axes11c, PERIODS):
            df_c = _build_area_df(spot_type, period)
            # Sort highest→lowest by total area within this panel
            df_c = (df_c.assign(_tot=lambda d: d.a90+d.a75_ex+d.a50_ex)
                        .sort_values('_tot', ascending=False)
                        .reset_index(drop=True))
            _pan_cl = df_c['cluster'].tolist()
            x    = np.arange(len(_pan_cl))
            bar_clrs = [_CLUST_CLR_11c.get(int(cl), '#BAB0AC') for cl in _pan_cl]

            left = np.zeros(len(_all_cl))
            for col, htch, alpha_v, lbl in [
                ('a90',    'oo', 0.85, 'L90 dense core'),
                ('a75_ex', 'xx', 0.70, 'L75 inner zone'),
                ('a50_ex', '//', 0.55, 'L50 outer extent'),
            ]:
                vals = df_c[col].values
                ax11c.bar(x, vals, bottom=left, color=bar_clrs, hatch=htch,
                          alpha=alpha_v, edgecolor='black', lw=0.4,
                          label=lbl, width=0.6)
                left += vals

            ax11c.set_xticks(x)
            ax11c.set_xticklabels([f'C{cl}' for cl in _pan_cl], fontsize=8,
                                   rotation=45, ha='right')
            ax11c.set_xlabel('Cluster25', fontsize=9)
            ax11c.set_ylabel('Total area (km²)', fontsize=9)
            ax11c.set_title(f'{period.capitalize()}', fontsize=10, fontweight='bold')
            ax11c.grid(axis='y', alpha=0.25, lw=0.5)

            if ax11c is axes11c[0]:
                ax11c.legend(fontsize=8, loc='upper right')

        # Cluster colour legend
        cl_h = [_mp11c.Patch(color=_CLUST_CLR_11c.get(cl, '#BAB0AC'),
                              label=f'C{cl}', edgecolor='black', lw=0.4)
                for cl in _all_cl]
        fig11c.legend(handles=cl_h, title='Cluster25', fontsize=7.5,
                      title_fontsize=9, loc='lower center',
                      ncol=min(len(cl_h), 8),
                      bbox_to_anchor=(0.5, -0.02), framealpha=0.9)
        plt.tight_layout(rect=[0, 0.06, 1, 1])
        fig11c.savefig(FIGS_DIR / fig_name, bbox_inches='tight')
        plt.show()
        print(f'  §11c saved: {fig_name}')

    # ── §11c Composite: Hot (Day/Night) | Cold (Day/Night) ──────────────────
    fig11cx, axes11cx = plt.subplots(1, 4, figsize=(22, 6.0), sharey=False)
    fig11cx.suptitle(
        'KDE Contour Area by Cluster25  (Tier1 + Tier2)  —  Hot & Cold · Day & Night',
        fontsize=12, fontweight='bold')

    _panels = [('hot','day'), ('hot','night'), ('cold','day'), ('cold','night')]
    _DAY_EDGE   = '#FFD700'   # yellow outline for day bars
    _NIGHT_EDGE = '#000000'   # black outline for night bars
    _OUTLINE_LW = 2.0

    for ax11cx, (spot_type, period) in zip(axes11cx, _panels):
        df_cx  = _build_area_df(spot_type, period)
        # Sort highest→lowest
        df_cx  = (df_cx.assign(_tot=lambda d: d.a90+d.a75_ex+d.a50_ex)
                       .sort_values('_tot', ascending=False)
                       .reset_index(drop=True))
        _pan_clx = df_cx['cluster'].tolist()
        x      = np.arange(len(_pan_clx))
        bar_clrs = [_CLUST_CLR_11c.get(int(cl), '#BAB0AC') for cl in _pan_clx]
        edgecolor = _DAY_EDGE if period == 'day' else _NIGHT_EDGE

        left = np.zeros(len(_all_cl))
        for col, htch, alpha_v, lbl in [
            ('a90',    'oo', 0.88, 'L90 dense core'),
            ('a75_ex', 'xx', 0.72, 'L75 inner zone'),
            ('a50_ex', '//', 0.56, 'L50 outer extent'),
        ]:
            vals = df_cx[col].values
            ax11cx.bar(x, vals, bottom=left, color=bar_clrs, hatch=htch,
                       alpha=alpha_v, edgecolor=edgecolor, lw=_OUTLINE_LW,
                       label=lbl, width=0.65)
            left += vals

        ax11cx.set_xticks(x)
        ax11cx.set_xticklabels([f'C{cl}' for cl in _pan_clx], fontsize=7,
                                rotation=50, ha='right')
        ax11cx.set_xlabel('Cluster25', fontsize=8)
        ax11cx.set_ylabel('Area (km²)', fontsize=8)
        lbl_period = '☀ Day' if period == 'day' else '☾ Night'
        lbl_type   = 'Hotspot' if spot_type == 'hot' else 'Coldspot'
        edge_lbl   = f'({"yellow" if period=="day" else "black"} outline)'
        ax11cx.set_title(f'{lbl_type}\n{lbl_period} {edge_lbl}',
                         fontsize=9, fontweight='bold')
        ax11cx.grid(axis='y', alpha=0.2, lw=0.4)

    # Shared legend at bottom
    import matplotlib.patches as _mp11cx
    hatch_h = [
        _mp11cx.Patch(facecolor='#AAAAAA', hatch='oo', edgecolor='black',
                      label='L90 dense core'),
        _mp11cx.Patch(facecolor='#AAAAAA', hatch='xx', edgecolor='black',
                      label='L75 inner zone'),
        _mp11cx.Patch(facecolor='#AAAAAA', hatch='//', edgecolor='black',
                      label='L50 outer extent'),
        _mp11cx.Patch(facecolor='none', edgecolor=_DAY_EDGE,   lw=2, label='Day (yellow border)'),
        _mp11cx.Patch(facecolor='none', edgecolor=_NIGHT_EDGE, lw=2, label='Night (black border)'),
    ]
    cl_h = [_mp11cx.Patch(color=_CLUST_CLR_11c.get(cl, '#BAB0AC'),
                           label=f'C{cl}', edgecolor='black', lw=0.4)
            for cl in _all_cl]
    fig11cx.legend(
        handles=hatch_h + [_mp11cx.Patch(visible=False)] + cl_h,
        title='Contour zone  |  Day/Night  |  Cluster25',
        fontsize=7.5, title_fontsize=8.5,
        loc='lower center', ncol=min(len(hatch_h) + 1 + len(cl_h), 12),
        bbox_to_anchor=(0.5, -0.01), framealpha=0.92)
    plt.tight_layout(rect=[0, 0.10, 1, 1])
    fig11cx.savefig(FIGS_DIR / 'hotcold_area_contour_cluster_composite.svg', bbox_inches='tight', dpi=500)
    fig11cx.savefig(FIGS_DIR / 'hotcold_area_contour_cluster_composite.pdf', bbox_inches='tight', dpi=500)
    plt.show()
    print('  §11c composite saved: hotcold_area_contour_cluster_composite.svg/.pdf')


In [ ]:
# §11d — Population Exposure Half-Donuts  (3 levels: City | Cluster | National)
# Half-donut: flat edge down, arc up.
# Left half = Hot spots | Right half = Cold spots
# Ring 1 (inner)  = Spot type, proportional to total population
# Ring 2 (middle) = Day vs Night within each spot type
# Ring 3 (outer)  = Contour zone (L90 / L75-excl / L50-excl)
# Labels: white-halo bbox; only for arcs ≥ MIN_ARC°; spacing enforced per side

import matplotlib.colors as _mc11d
from matplotlib.patches import Wedge as _W11d, Patch as _P11d

HOT_BASE  = '#E07B00'
COLD_BASE = '#2B6CB0'
MIN_ARC   = 8.0     # minimum arc (degrees) to draw a label


def _fmt11d(v):
    if v >= 1e6: return f'{v/1e6:.1f}M'
    if v >= 1e3: return f'{v/1e3:.1f}k'
    return f'{int(v)}'


def _blend11d(hex_c, fade):
    rgb = np.array(_mc11d.to_rgb(hex_c))
    return tuple(rgb + fade * (1 - rgb))


def _draw_halfdonut(sub_gdf, title):
    if not len(sub_gdf):
        return None

    fig, ax = plt.subplots(figsize=(10, 6.5), subplot_kw=dict(aspect='equal'))

    r_i = [0.32, 0.60, 0.92]
    r_w = [0.25, 0.28, 0.28]

    def _pop(st, per):
        s = sub_gdf[(sub_gdf.spot_type == st) & (sub_gdf.period == per)]
        return float(s['population'].sum()) if len(s) else 0.

    def _pop75(st, per):
        s = sub_gdf[(sub_gdf.spot_type == st) & (sub_gdf.period == per)]
        if 'pop_75' in s.columns and s['pop_75'].notna().any():
            return float(s['pop_75'].sum()) if len(s) else 0.
        a50 = float(s['area_km2_50'].sum()) or 1
        a75 = float(s['area_km2_75'].fillna(0).sum())
        return _pop(st, per) * (a75 / a50)

    def _pop90(st, per):
        s = sub_gdf[(sub_gdf.spot_type == st) & (sub_gdf.period == per)]
        if 'pop_90' in s.columns and s['pop_90'].notna().any():
            return float(s['pop_90'].sum()) if len(s) else 0.
        a50 = float(s['area_km2_50'].sum()) or 1
        a90 = float(s['area_km2_90'].fillna(0).sum())
        return _pop(st, per) * (a90 / a50)

    total_hot_pop  = _pop('hot', 'day')  + _pop('hot', 'night')
    total_cold_pop = _pop('cold', 'day') + _pop('cold', 'night')
    total_pop      = total_hot_pop + total_cold_pop

    total_all  = (total_hot_pop + total_cold_pop) or 1
    hot_sweep  = min(180., 180. * total_hot_pop  / total_all)
    cold_sweep = min(180., 180. * total_cold_pop / total_all)

    raw_labels = []   # collect before drawing (to apply spacing first)

    for st, base_ang, ang_span, base_clr in (
        ('hot',  180 - hot_sweep,  hot_sweep,  HOT_BASE),
        ('cold', 0,                cold_sweep, COLD_BASE),
    ):
        side = 'left' if st == 'hot' else 'right'
        if ang_span < 0.5:
            continue

        # Ring 1: spot-type band
        ax.add_patch(_W11d((0, 0), r_i[0] + r_w[0], base_ang, base_ang + ang_span,
                           width=r_w[0], facecolor=_blend11d(base_clr, 0.0),
                           edgecolor='white', lw=2.0))
        if ang_span >= MIN_ARC:
            mid_ang = base_ang + ang_span / 2
            mid_r   = r_i[0] + r_w[0] / 2
            raw_labels.append({'ang': mid_ang, 'r': mid_r, 'side': side,
                               'text': f'{"Hot" if st=="hot" else "Cold"}: '
                                       f'{_fmt11d(total_hot_pop if st=="hot" else total_cold_pop)}',
                               'fs': 9, 'fw': 'bold'})

        # Ring 2: Day / Night
        ang2    = base_ang
        tot_st  = (total_hot_pop if st == 'hot' else total_cold_pop) or 1
        for per, fade, ico in (('day', 0.0, '☀'), ('night', 0.30, '☾')):
            pp  = _pop(st, per)
            sw2 = ang_span * (pp / tot_st)
            if pp > 0 and sw2 > 0.5:
                ax.add_patch(_W11d((0, 0), r_i[1] + r_w[1], ang2, ang2 + sw2,
                                   width=r_w[1], facecolor=_blend11d(base_clr, fade),
                                   edgecolor='white', lw=1.4))
                if sw2 >= MIN_ARC:
                    raw_labels.append({'ang': ang2 + sw2 / 2, 'r': r_i[1] + r_w[1] / 2,
                                       'side': side,
                                       'text': f'{ico} {per.capitalize()}: {_fmt11d(pp)}',
                                       'fs': 7.5, 'fw': 'normal'})

                # Ring 3: contour zones
                p90    = min(_pop90(st, per), pp)
                p75    = min(_pop75(st, per), pp)
                p75_ex = max(0, p75 - p90)
                p50_ex = max(0, pp  - p75)
                ang3   = ang2
                for pop_seg, fade3, htch, zone_lbl in (
                    (p90,    0.0,  'oo', 'L90'),
                    (p75_ex, 0.15, 'xx', 'L75'),
                    (p50_ex, 0.30, '//', 'L50'),
                ):
                    sw3 = sw2 * (pop_seg / pp) if pp > 0 else 0
                    if pop_seg > 0 and sw3 > 0.3:
                        ax.add_patch(_W11d((0, 0), r_i[2] + r_w[2], ang3, ang3 + sw3,
                                          width=r_w[2],
                                          facecolor=_blend11d(base_clr, 0.55 + fade3),
                                          hatch=htch, edgecolor='white', lw=0.8))
                        if sw3 >= MIN_ARC:
                            raw_labels.append({
                                'ang':  ang3 + sw3 / 2,
                                'r':    r_i[2] + r_w[2] / 2,
                                'side': side,
                                'text': f'{zone_lbl}: {_fmt11d(pop_seg)}',
                                'fs':   6.5, 'fw': 'normal'})
                    ang3 += sw3
            ang2 += sw2

    # Centre label
    ax.text(0,  0.10, _fmt11d(total_pop),   ha='center', va='center',
            fontsize=13, fontweight='bold', color='#333')
    ax.text(0, -0.08, 'Total\npop',          ha='center', va='center',
            fontsize=7, color='#666')
    ax.text(-0.7, -0.08, '← Hot spots',  ha='center', va='top',
            fontsize=9, fontweight='bold', color=HOT_BASE)
    ax.text( 0.7, -0.08, 'Cold spots →', ha='center', va='top',
            fontsize=9, fontweight='bold', color=COLD_BASE)

    ax.set_xlim(-2.2, 2.2)
    ax.set_ylim(-0.6, 1.80)

    # Build annotate positions; enforce minimum y-spacing per side
    MIN_DY = 0.14
    annots = []
    for sp in raw_labels:
        ang_r = np.radians(sp['ang'])
        rx = sp['r'] * np.cos(ang_r)
        ry = sp['r'] * np.sin(ang_r)
        lx = -1.90 if sp['side'] == 'left' else 1.90
        annots.append({**sp, 'rx': rx, 'ry': ry, 'lx': lx, 'y_text': max(0.04, ry)})

    for side in ('left', 'right'):
        side_a = sorted([a for a in annots if a['side'] == side],
                        key=lambda a: a['y_text'])
        for i in range(1, len(side_a)):
            if side_a[i]['y_text'] - side_a[i - 1]['y_text'] < MIN_DY:
                side_a[i]['y_text'] = side_a[i - 1]['y_text'] + MIN_DY

    for sp in annots:
        ax.annotate(
            sp['text'],
            xy=(sp['rx'], sp['ry']),
            xytext=(sp['lx'], sp['y_text']),
            ha='right' if sp['side'] == 'left' else 'left',
            va='center', fontsize=sp['fs'], fontweight=sp['fw'],
            color='#222',
            bbox=dict(boxstyle='round,pad=0.18', fc='white', ec='none', alpha=0.88),
            arrowprops=dict(arrowstyle='-', color='#bbb', lw=0.5))

    ax.axis('off')
    ax.set_title(title, fontsize=9, fontweight='bold', pad=10)

    leg_h = [
        _P11d(facecolor=HOT_BASE,  label='Hot spots'),
        _P11d(facecolor=COLD_BASE, label='Cold spots'),
        _P11d(facecolor='#AAAAAA', hatch='oo', edgecolor='white', label='L90 dense core'),
        _P11d(facecolor='#AAAAAA', hatch='xx', edgecolor='white', label='L75 inner zone'),
        _P11d(facecolor='#AAAAAA', hatch='//', edgecolor='white', label='L50 outer extent'),
    ]
    ax.legend(handles=leg_h, loc='lower center', ncol=5, fontsize=7.5,
              bbox_to_anchor=(0.5, -0.04), framealpha=0.9)
    plt.tight_layout()
    return fig


def plot_city_halfdonuts(show_first=True):
    if not len(gdf):
        print('  ⚠ GDF empty — skipping city half-donuts')
        return
    city_order = sorted(
        gdf['city'].unique(),
        key=lambda c: (city_to_cluster.get(c, 99), c))
    shown = 0
    for city in city_order:
        sub = gdf[gdf.city == city]
        if not len(sub):
            continue
        cl    = city_to_cluster.get(city, '?')
        title = (f'City: {city}  |  Cluster {cl} — Population Exposure to UHI Spots\n'
                 f'Left=Hot  Right=Cold  ·  Ring1=Type  Ring2=Day/Night  Ring3=Contour')
        fig = _draw_halfdonut(sub, title)
        if fig is None:
            continue
        out = FIGS_DIR / f'{city}_population_halfdonut.svg'
        fig.savefig(out, bbox_inches='tight')
        if show_first and shown == 0:
            plt.show()
        else:
            plt.close(fig)
        shown += 1
    print(f'  §§11d city half-donuts: {shown} saved → {FIGS_DIR}')


def plot_cluster_halfdonuts(show_first=False):
    if not len(gdf):
        print('  ⚠ GDF empty — skipping cluster half-donuts')
        return
    all_clusters = sorted(gdf['cluster25'].dropna().unique().astype(int))
    shown = 0
    for cl in all_clusters:
        sub = gdf[gdf.cluster25 == cl]
        if not len(sub):
            continue
        title = (f'Cluster {cl} — Population Exposure to UHI Hot & Cold Spots\n'
                 f'Left=Hot  Right=Cold  ·  Ring1=Type  Ring2=Day/Night  Ring3=Contour')
        fig = _draw_halfdonut(sub, title)
        if fig is None:
            continue
        out = FIGS_DIR / f'cluster{cl}_population_halfdonut.svg'
        fig.savefig(out, bbox_inches='tight')
        if show_first and shown == 0:
            plt.show()
        else:
            plt.close(fig)
        shown += 1
    print(f'  §§11d cluster half-donuts: {shown} saved → {FIGS_DIR}')


def plot_national_halfdonut(show=False):
    if not len(gdf):
        print('  ⚠ GDF empty — skipping national half-donut')
        return
    title = ('Nigeria — National Population Exposure to UHI Hot & Cold Spots\n'
             'Left=Hot  Right=Cold  ·  Ring1=Type  Ring2=Day/Night  Ring3=Contour zone')
    fig = _draw_halfdonut(gdf, title)
    if fig is None:
        return
    out = FIGS_DIR / 'national_population_halfdonut.svg'
    fig.savefig(out, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    print(f'  §§11d national half-donut saved → {out.name}')


plot_city_halfdonuts(show_first=True)
plot_cluster_halfdonuts(show_first=False)
plot_national_halfdonut(show=False)

# ── §11d composite: 4-column panel of all cluster half-donuts ────────────────────
def plot_cluster_halfdonuts_composite(ncols=4, show=False, dpi=150):
    if not len(gdf):
        print('  GDF empty — composite skipped')
        return
    all_clusters = sorted(gdf['cluster25'].dropna().unique().astype(int))
    imgs, labels = [], []
    for cl in all_clusters:
        sub = gdf[gdf.cluster25 == cl]
        if not len(sub):
            continue
        fig_cl = _draw_halfdonut(sub, f'Cluster {cl}')
        if fig_cl is None:
            continue
        fig_cl.canvas.draw()
        w, h = fig_cl.canvas.get_width_height()
        buf = np.asarray(fig_cl.canvas.buffer_rgba())[..., :3]
        imgs.append(buf)
        labels.append(f'Cluster {cl}')
        plt.close(fig_cl)
    if not imgs:
        print('  No cluster panels rendered — composite skipped')
        return
    nrows = (len(imgs) + ncols - 1) // ncols
    fig_comp, axes_comp = plt.subplots(nrows, ncols,
                                       figsize=(ncols * 5.5, nrows * 3.5),
                                       gridspec_kw={'hspace': 0.08, 'wspace': 0.04})
    axes_flat = np.array(axes_comp).ravel()
    for i, (img, lbl) in enumerate(zip(imgs, labels)):
        axes_flat[i].imshow(img)
        axes_flat[i].axis('off')
        axes_flat[i].set_title(lbl, fontsize=7, fontweight='bold', pad=2)
    for j in range(len(imgs), len(axes_flat)):
        axes_flat[j].axis('off')
    fig_comp.suptitle(
        'Cluster25 — Population Exposure Half-Donuts\n'
        'Left = Hot spots  |  Right = Cold spots  '
        '·  Ring1 = Type  Ring2 = Day/Night  Ring3 = Contour zone',
        fontsize=9, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    out_comp = FIGS_DIR / 'cluster_halfdonuts_composite.svg'
    fig_comp.savefig(out_comp, bbox_inches='tight', dpi=dpi)
    fig_comp.savefig(str(out_comp).replace('.svg', '.pdf'), bbox_inches='tight', dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig_comp)
    print(f'  §11d composite saved: {out_comp.name}  ({len(imgs)} clusters)')


plot_cluster_halfdonuts_composite(ncols=4, show=False)


# Half-donut summary tables (city-level and cluster-level CSVs)
if len(gdf):
    _hd_base = ['spot_type', 'period', 'area_km2_50']
    _hd_base += ['area_km2_75'] if 'area_km2_75' in gdf.columns else []
    _hd_base += ['area_km2_90'] if 'area_km2_90' in gdf.columns else []
    _hd_pop  = ['population']
    _hd_pop  += ['pop_75'] if 'pop_75' in gdf.columns else []
    _hd_pop  += ['pop_90'] if 'pop_90' in gdf.columns else []
    # City-level
    _city_hd = (gdf[['city'] + _hd_base + _hd_pop]
                .groupby(['city', 'spot_type', 'period'])
                .sum(numeric_only=True).reset_index())
    _city_hd.rename(columns={'population': 'pop_50'}, inplace=True)
    _city_hd.to_csv(RESULTS_DIR / 'halfdonut_city_summary.csv', index=False)
    print(f'  Saved: {RESULTS_DIR / "halfdonut_city_summary.csv"}')
    # Cluster-level
    if 'cluster' in gdf.columns:
        _cl_hd = (gdf[['cluster'] + _hd_base + _hd_pop]
                  .groupby(['cluster', 'spot_type', 'period'])
                  .sum(numeric_only=True).reset_index())
        _cl_hd.rename(columns={'population': 'pop_50'}, inplace=True)
        _cl_hd.to_csv(RESULTS_DIR / 'halfdonut_cluster_summary.csv', index=False)
        print(f'  Saved: {RESULTS_DIR / "halfdonut_cluster_summary.csv"}')
else:
    print('  GDF empty — half-donut CSVs skipped')

# §11e — National Summary Maps: Hot & Cold Spot Wedge Circles
# Background: Cluster25 choropleth (city-region polygons, low alpha)
# Circles: proportional wedge — Day (orange) | Night (dark grey) for hot
#                                Day (cyan) | Night (blue) for cold
# Waterbodies overlaid from per-city AOI folders

from matplotlib.patches import Wedge as _Wedge5e
from shapely.ops import unary_union as _uu5e

_CLUST_CLR_5e = CLUSTER25_COLORS

def _fmt_pop5e(v):
    if v >= 1e6: return f'{v/1e6:.1f}M'
    if v >= 1e3: return f'{v/1e3:.1f}k'
    return f'{int(v)}'

# Pre-load landscape GDF once at cell level — avoids two expensive
# gpd.read_file() calls (one per hot/cold map) and eliminates the
# per-polygon iterrows() loop that caused >20-minute render times.
try:
    _land_gdf_national = gpd.read_file(config.LANDSCAPE_DATA_PATH).to_crs('EPSG:4326')
    _land_gdf_national['_clr'] = _land_gdf_national['Cluster25'].apply(
        lambda c: _CLUST_CLR_5e.get(int(c), '#DDDDDD'))
    print(f'  Landscape cache: {len(_land_gdf_national)} polygons ready for national map')
except Exception as _e_land_cache:
    _land_gdf_national = None
    print(f'  Warning: landscape GDF not pre-loaded: {_e_land_cache}')


def plot_national_map_v2(spot_type_filter='hot', show=True):
    DAY_CLR   = '#FF9500' if spot_type_filter == 'hot' else '#00CED1'
    NIGHT_CLR = '#4A4A4A' if spot_type_filter == 'hot' else '#1565C0'
    spot_label = 'Hotspot' if spot_type_filter == 'hot' else 'Coldspot'

    _nigeria_path = ROOT / 'data' / 'inputs' / 'geospatial_features' / 'boundary_Nigeria.geojson'
    try:
        nigeria_gdf = gpd.read_file(_nigeria_path)
    except Exception as e:
        print(f'  Nigeria boundary not found: {e}'); return

    fig, ax = plt.subplots(figsize=(10, 10))

    # ── Background: Cluster25 landscape (single vectorised call) ─────────────
    if _land_gdf_national is not None:
        _land_gdf_national.plot(ax=ax, color=_land_gdf_national['_clr'],
                                alpha=0.30, edgecolor='none', zorder=1)
    else:
        nigeria_gdf.plot(ax=ax, facecolor='#F7F4F0', edgecolor='#555', lw=1.2, zorder=1)

    # National boundary outline (on top of landscape)
    nigeria_gdf.boundary.plot(ax=ax, edgecolor='#1A1A1A', lw=1.8, zorder=6)

    # ── Per-city wedge circles ────────────────────────────────────────────────
    city_stats = {}
    for city in cities_with_sev:
        zi = zones_dict.get(city)
        if zi is None: continue
        centroid_4326 = gpd.GeoSeries([zi['uc']], crs='ESRI:102022').to_crs('EPSG:4326').centroid.iloc[0]
        stats = {'lon': centroid_4326.x, 'lat': centroid_4326.y}
        for period in ('day', 'night'):
            sub = (gdf[(gdf.city==city) & (gdf.period==period) &
                       (gdf.spot_type==spot_type_filter)] if len(gdf) else gpd.GeoDataFrame())
            stats[f'{period}_area'] = float(sub['area_km2_50'].sum()) if len(sub) else 0.
            stats[f'{period}_pop']  = float(sub['population'].sum())  if len(sub) else 0.
        stats['total_area'] = stats['day_area'] + stats['night_area']
        stats['total_pop']  = stats['day_pop']  + stats['night_pop']
        stats['bh'] = bool(gdf[(gdf.city==city)&(gdf.spot_type==spot_type_filter)]['bh_significant'].any()) if len(gdf) else False
        city_stats[city] = stats

    max_area = max((s['total_area'] for s in city_stats.values() if s['total_area'] > 0), default=1)
    max_pop  = max((s['total_pop']  for s in city_stats.values() if s['total_pop']  > 0), default=1)
    SCALE = 0.55

    total_day_pop   = sum(s['day_pop']   for s in city_stats.values())
    total_night_pop = sum(s['night_pop'] for s in city_stats.values())
    n_with_spots    = sum(1 for s in city_stats.values() if s['total_area'] > 0)
    total_area_km2  = sum(s['total_area'] for s in city_stats.values())

    for city, s in city_stats.items():
        lon, lat = s['lon'], s['lat']
        total = s['total_area'] or 0
        total_pop_city = s['total_pop'] or 0
        if total <= 0 or total_pop_city <= 0:
            ax.plot(lon, lat, 'o', color='#CCCCCC', ms=4, zorder=5, alpha=0.6)
            continue
        radius  = SCALE * (max(total_pop_city, 1) / max_pop) ** 0.5
        day_f   = s['day_area']   / total
        night_f = s['night_area'] / total
        day_start = 90; day_end = 90 + day_f * 360
        if s['day_area'] > 0:
            ax.add_patch(_Wedge5e((lon, lat), radius, day_start, day_end,
                                  facecolor=DAY_CLR, edgecolor='white', lw=0.8,
                                  alpha=0.92, zorder=6))
        night_start = day_end; night_end = 450
        if s['night_area'] > 0:
            ax.add_patch(_Wedge5e((lon, lat), radius, night_start, night_end,
                                  facecolor=NIGHT_CLR, edgecolor='white', lw=0.8,
                                  alpha=0.92, zorder=6))
        ax.plot(lon, lat, 'ko', ms=2.5, zorder=7)
        ax.text(lon, lat - radius - 0.08,
                f"{city}\n({s['total_area']:.0f} km²)",
                ha='center', va='top', fontsize=5.5, color='#222', zorder=7,
                linespacing=0.9,
                bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none', alpha=0.65))

    bounds = nigeria_gdf.total_bounds
    ax.set_xlim(bounds[0]-0.5, bounds[2]+0.5)
    ax.set_ylim(bounds[1]-1.0, bounds[3]+0.8)
    ax.grid(True, alpha=0.12, ls='--', lw=0.4, zorder=0)
    ax.set_xlabel('Longitude', fontsize=10)
    ax.set_ylabel('Latitude',  fontsize=10)
    ax.set_aspect('equal')
    ax.set_title(f'Nigeria UHI {spot_label} Summary — Area & Population Exposure',
                 fontsize=13, fontweight='bold', pad=12)

    # Legend — Day / Night / No spots only (no Cluster25 entries)
    import matplotlib.patches as _mp5e
    import matplotlib.lines as _ml5e_leg
    legend_els = [
        _mp5e.Patch(facecolor=DAY_CLR,   edgecolor='white',
                    label=f'Day {spot_label.lower()} area',   alpha=0.92),
        _mp5e.Patch(facecolor=NIGHT_CLR, edgecolor='white',
                    label=f'Night {spot_label.lower()} area', alpha=0.92),
        _mp5e.Patch(facecolor='#CCCCCC', edgecolor='none',
                    label='No spots detected'),
        _ml5e_leg.Line2D([0], [0], color='#1A1A1A', lw=1.8,
                         label='Nigeria boundary'),
    ]
    ax.legend(handles=legend_els, loc='lower left', fontsize=8.5, framealpha=0.95,
              edgecolor='#999', title='Legend', title_fontsize=10, ncol=1)

    # North arrow
    ax.annotate('N', xy=(0.97, 0.96), xytext=(0.97, 0.91),
                xycoords='axes fraction', fontsize=12, ha='center', fontweight='bold',
                arrowprops=dict(arrowstyle='->', lw=2, color='black'), zorder=10)

    # Scale bar — bottom centre
    _deg_100km = 100 / 111.0   # degrees ≈ 100km at equator
    sc_lon = (bounds[0] + bounds[2]) / 2 - _deg_100km / 2
    sc_lat = bounds[1] - 0.55
    ax.plot([sc_lon, sc_lon+0.9], [sc_lat, sc_lat], 'k-', lw=3, zorder=8)
    ax.plot([sc_lon, sc_lon], [sc_lat-0.1, sc_lat+0.1], 'k-', lw=2, zorder=8)
    ax.plot([sc_lon+0.9, sc_lon+0.9], [sc_lat-0.1, sc_lat+0.1], 'k-', lw=2, zorder=8)
    ax.text(sc_lon+0.45, sc_lat+0.15, '~100 km',
            ha='center', va='bottom', fontsize=8, fontweight='bold', zorder=8)

    # Stats box
    stats_box = (
        f'National Summary\n'
        f'\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n'
        f'Cities analysed : {len(cities_with_sev)}\n'
        f'Cities w/ {spot_label.lower()}s: {n_with_spots}\n'
        f'Total area (km²): {total_area_km2:.0f}\n'
        f'Day   pop exp  : {_fmt_pop5e(total_day_pop)}\n'
        f'Night pop exp  : {_fmt_pop5e(total_night_pop)}'
    )
    ax.text(0.98, 0.01, stats_box, transform=ax.transAxes,
            fontsize=8, va='bottom', ha='right',
            bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='#AAA', alpha=0.9),
            fontfamily='monospace', zorder=10)

    # Proportional circle legend (population scale) — bottom right, above stats box
    _prop_ins = ax.inset_axes([0.77, 0.14, 0.20, 0.20])
    _prop_ins.set_xlim(0, 10); _prop_ins.set_ylim(0, 10)
    _prop_ins.set_facecolor('white'); _prop_ins.patch.set_alpha(0.92)
    _prop_ins.patch.set_edgecolor('#AAAAAA'); _prop_ins.patch.set_linewidth(0.8)
    _prop_ins.axis('off')
    _prop_ins.text(5, 9.3, 'Population\nExposure', ha='center', va='top',
                   fontsize=6, fontweight='bold')
    from matplotlib.patches import Wedge as _WLeg5e
    _ref_pops5e = [max_pop, max_pop * 0.35, max_pop * 0.08]
    _ref_lbl5e  = [_fmt_pop5e(p) for p in _ref_pops5e]
    for _ri, (_rp, _rl) in enumerate(zip(_ref_pops5e, _ref_lbl5e)):
        _rr = 2.0 * (_rp / max_pop) ** 0.5
        _yc = 7.0 - _ri * 3.0
        _prop_ins.add_patch(_WLeg5e((3, _yc), _rr, 0, 360,
                                     facecolor=DAY_CLR, edgecolor='white',
                                     lw=0.6, alpha=0.88))
        _prop_ins.text(3 + _rr + 0.4, _yc, _rl, va='center', ha='left', fontsize=6)

    plt.tight_layout()
    out_name = f'national_{spot_type_filter}_summary_map.svg'
    fig.savefig(FIGS_DIR / out_name, bbox_inches='tight', dpi=500)
    fig.savefig(FIGS_DIR / out_name.replace('.svg', '.pdf'), bbox_inches='tight', dpi=500)
    if show: plt.show()
    else:    plt.close(fig)
    print(f'  §11e saved: {out_name}')


plot_national_map_v2(spot_type_filter='hot',  show=True)
plot_national_map_v2(spot_type_filter='cold', show=False)


In [ ]:
# §11f — Stakeholder Perception by Cluster25 (Q60 / Q61)
# Bars sorted by number of participants, highest at top.
if len(workshop_df) == 0:
    print("⚠ No workshop attendee data — skipping §11f")
else:
    _prc = (workshop_df
            .groupby('pid').agg(
                city=('matched_city_y', 'first'),
                warming=('61_temp_increase_10yrs', 'first'),
                diurnal=('60_diurnal_severity_heat', 'first'),
            ).reset_index())
    _prc['warming'] = _prc['warming'].fillna("I'd rather not say")
    _prc['diurnal'] = _prc['diurnal'].fillna("I'd rather not say")
    _prc['cluster25'] = _prc['city'].map(city_to_cluster)
    _prc['cluster_label'] = _prc['cluster25'].apply(
        lambda x: f'C{int(x)}' if pd.notna(x) else None)
    _prc = _prc[_prc['cluster_label'].notna()]  # drop unmatched clusters

    _WARM_COLORS = {'Yes': '#d62728', 'No': '#2ca02c',
                    "I'm not sure": '#7f7f7f', "I'd rather not say": '#c7c7c7'}
    _DURN_COLORS = {'During the day': '#ff7f0e', 'At night': '#1f77b4',
                    'Both': '#9467bd', "I'd rather not say": '#c7c7c7'}
    _WARM_ORDER = ['Yes', 'No', "I'm not sure", "I'd rather not say"]
    _DURN_ORDER = ['During the day', 'At night', 'Both', "I'd rather not say"]

    # Build cluster order (numeric clusters only — C* excluded)
    _cl_all = sorted(_prc['cluster_label'].unique(),
                     key=lambda x: int(x[1:]))

    # Build cross-tabs
    _w_ctab = pd.crosstab(_prc['cluster_label'], _prc['warming']).reindex(_cl_all).fillna(0)
    _d_ctab = pd.crosstab(_prc['cluster_label'], _prc['diurnal']).reindex(_cl_all).fillna(0)
    for col in _WARM_ORDER:
        if col not in _w_ctab.columns: _w_ctab[col] = 0.0
    for col in _DURN_ORDER:
        if col not in _d_ctab.columns: _d_ctab[col] = 0.0
    _w_ctab = _w_ctab[_WARM_ORDER]
    _d_ctab = _d_ctab[_DURN_ORDER]

    # ── Sort by total participants (descending) so highest is index-0 ──────────
    # With invert_yaxis(), index-0 → top of chart → highest count appears at top.
    _cl_totals = _w_ctab.sum(axis=1)
    _cl_sorted = list(_cl_totals.sort_values(ascending=False).index)
    # (C* already excluded — no need to sort it to the bottom)
    _cl_order = _cl_sorted
    _w_ctab = _w_ctab.reindex(_cl_order).fillna(0)
    _d_ctab = _d_ctab.reindex(_cl_order).fillna(0)

    _fig_h = max(len(_cl_order) * 0.55 + 2.5, 5.0)
    fig_prc, (_axw, _axd) = plt.subplots(1, 2, figsize=(13, _fig_h),
                                          gridspec_kw={'wspace': 0.08})

    for ax, ctab, colors, order, title in [
        (_axw, _w_ctab, _WARM_COLORS, _WARM_ORDER,
         'A) City Warming Over 10 Years (Q61)'),
        (_axd, _d_ctab, _DURN_COLORS, _DURN_ORDER,
         'B) Worst Diurnal Heat Period (Q60)'),
    ]:
        left = np.zeros(len(ctab))
        for col in order:
            vals = ctab[col].values
            ax.barh(range(len(ctab)), vals, left=left, color=colors[col],
                    label=col, height=0.70)
            left += vals
        ax.set_yticks(range(len(ctab)))
        ax.set_yticklabels(ctab.index, fontsize=9)
        ax.invert_yaxis()    # index-0 (highest count) appears at top
        ax.set_xlabel('Number of Participants', fontsize=9)
        ax.set_title(title, fontsize=10, fontweight='bold', loc='left')
        _xlim = max(float(ctab.sum(axis=1).max()) * 1.35, 8.0)
        ax.set_xlim(0, _xlim)
        for i, cl in enumerate(ctab.index):
            _n = int(ctab.loc[cl].sum())
            ax.text(_n + _xlim * 0.01, i, f'n={_n}',
                    va='center', fontsize=7, color='gray')
            # (C* excluded — no italic styling needed)
        ax.legend(loc='lower right', fontsize=8, framealpha=0.9)

    _axd.set_yticklabels([])
    fig_prc.suptitle('Stakeholder Perception by Physiographic Cluster\n'
                     '(workshop attendees; matched_city_y → Cluster25; sorted by n)',
                     fontsize=11, fontweight='bold')
    plt.tight_layout()
    fig_prc.savefig(FIGS_DIR / 'stakeholder_perception_cluster_Q60_Q61.svg',
                    bbox_inches='tight')
    plt.show()
    print(f"§11f: {len(_cl_order)} clusters, {len(_prc)} unique participants "
          f"(sorted highest→lowest)")


<a id="nb05-s12"></a>

## §12 — Aggregation Hierarchy & Summary Tables

### Aggregation rationale

The wide-format GDF (§7) contains one row per spot polygon. Aggregation up the spatial
hierarchy produces progressively coarser summaries suitable for different reporting contexts:

```
Individual spots → City subtotals → Cluster25 subtotals → National total
```

No `is_primary` filter is applied — all validated spot polygons are included. Where a city
has both Tier 1 and Tier 2 spots in the same period/type combination, both contribute to
the city total.

### Exported CSV files

| File | Level | Read by |
|------|-------|---------|
| `cluster_hotspot_summary.csv` | Cluster × period × type | Notebook 07 (Table 2) |
| `national_hotspot_summary.csv` | National × period × type | Notebook 07 (Table 2) |

The city-level table is built in memory only (the two files above are aggregated from it).
Related inputs written elsewhere: `spot_comprehensive_summary.csv` (§6c) and
`halfdonut_city_summary.csv` (§11d) are read by the dataset build and Notebook 07.
`master_city_composite.csv` is no longer read by anything: the build derives `validation_tier`
and `bh_any` itself, and the file has been moved to `Results/_to_delete/nb05_unused/` (see §13).

### Interpreting Day vs Night asymmetry

Many cities show markedly different hotspot areas between Day and Night. Day-dominant
profiles indicate **radiant-heat** hotspots driven by solar absorption (impervious
surfaces, rooftops). Night-dominant profiles indicate **stored-heat** release from dense
urban fabric with high thermal capacity. Cities where Day and Night are nearly equal
reflect a pervasive urban heat island present through the full diurnal cycle.

### BH-FDR flag interpretation

`bh_significant = True` marks spots that pass the stricter Benjamini-Hochberg FDR
correction (Z > ~4 in practice; see the §4 note). These are the most robust hotspot
signals in the dataset — their locations can be defended even under conservative
multiple-testing assumptions. `pct_cities_bh` in the cluster summary reports the
fraction of a cluster's cities where at least one spot is BH-significant.

[↑ Table of Contents](#nb05-toc)


In [ ]:
if len(gdf) > 0:
    # Wide-format GDF: one row per spot (50% polygon). No is_primary filter needed.

    AGG_CITY = dict(
        n_spots           = ('spot_id',        'count'),
        total_area_50_km2 = ('area_km2_50',    'sum'),
        mean_area_50_km2  = ('area_km2_50',    'mean'),
        std_area_50_km2   = ('area_km2_50',    'std'),
        total_area_90_km2 = ('area_km2_90',    'sum'),
        mean_dense_core   = ('dense_core_pct', 'mean'),
        mean_lst          = ('mean_lst',        'mean'),
        max_lst           = ('max_lst',         'max'),
        total_pop         = ('population',      'sum'),
        mean_pop_per_spot = ('population',      'mean'),
        suhii             = ('suhii',           'first'),
        bh_significant    = ('bh_significant',  'any'),
        persistent_dn     = ('persistent_dn',   'first'),
    )

    # City level
    city_per = (gdf.groupby(['city','cluster25','period','spot_type'])
                .agg(**AGG_CITY).reset_index())
    city_sub = (gdf.groupby(['city','cluster25','spot_type'])
                .agg(**AGG_CITY).reset_index().assign(period='All periods'))
    city_tbl = (pd.concat([city_per, city_sub], ignore_index=True)
                .sort_values(['cluster25','city','spot_type','period']))
    # per_city_hotspot_summary.csv is no longer written: nothing reads it (the city table
    # is still built, because the cluster and national tables are aggregated from it)
    print(f"City summary    : {len(city_tbl)} rows (in memory only)")

    # Cluster level
    AGG_CLUST = dict(
        n_cities          = ('city',             'nunique'),
        n_spots           = ('n_spots',          'sum'),
        total_area_50_km2 = ('total_area_50_km2','sum'),
        mean_area_50_km2  = ('mean_area_50_km2', 'mean'),
        std_area_50_km2   = ('std_area_50_km2',  'mean'),
        total_area_90_km2 = ('total_area_90_km2','sum'),
        mean_dense_core   = ('mean_dense_core',  'mean'),
        mean_lst          = ('mean_lst',         'mean'),
        max_lst           = ('max_lst',          'max'),
        total_pop         = ('total_pop',        'sum'),
        mean_suhii        = ('suhii',            'mean'),
        std_suhii         = ('suhii',            'std'),
        pct_cities_bh     = ('bh_significant',   'mean'),
    )
    cl_per = (city_per.groupby(['cluster25','period','spot_type'])
              .agg(**AGG_CLUST).reset_index())
    cl_per['pct_cities_bh'] = (cl_per['pct_cities_bh'] * 100).round(1)
    cl_sub = (city_per.groupby(['cluster25','spot_type'])
              .agg(**AGG_CLUST).reset_index().assign(period='All periods'))
    cl_sub['pct_cities_bh'] = (cl_sub['pct_cities_bh'] * 100).round(1)
    cl_tbl = (pd.concat([cl_per, cl_sub], ignore_index=True)
              .sort_values(['cluster25','spot_type','period']))
    cl_tbl.to_csv(RESULTS_DIR / 'cluster_hotspot_summary.csv', index=False)
    print(f"Cluster summary : {len(cl_tbl)} rows  →  cluster_hotspot_summary.csv")

    # National level
    AGG_NAT = dict(
        n_clusters        = ('cluster25',        'nunique'),
        n_cities          = ('n_cities',         'sum'),
        n_spots           = ('n_spots',          'sum'),
        total_area_50_km2 = ('total_area_50_km2','sum'),
        mean_area_50_km2  = ('mean_area_50_km2', 'mean'),
        total_area_90_km2 = ('total_area_90_km2','sum'),
        mean_dense_core   = ('mean_dense_core',  'mean'),
        mean_lst          = ('mean_lst',         'mean'),
        total_pop         = ('total_pop',        'sum'),
        mean_suhii        = ('mean_suhii',       'mean'),
        pct_cities_bh     = ('pct_cities_bh',    'mean'),
    )
    nat_per = cl_per.groupby(['period','spot_type']).agg(**AGG_NAT).reset_index()
    nat_sub = (cl_per.groupby('spot_type').agg(**AGG_NAT).reset_index()
               .assign(period='All periods'))
    nat_tbl = pd.concat([nat_per, nat_sub], ignore_index=True)
    nat_tbl.to_csv(RESULTS_DIR / 'national_hotspot_summary.csv', index=False)
    print(f"National summary: {len(nat_tbl)} rows  →  national_hotspot_summary.csv")

    print("\n── National totals ──────────────────────────────────────────────────")
    _show_cols = ['spot_type','n_cities','n_spots','total_area_50_km2',
                  'total_pop','mean_lst','mean_suhii']
    print(nat_tbl[nat_tbl.period=='All periods'][_show_cols].round(2).to_string(index=False))
else:
    print("⚠ GDF empty — skipping §12 aggregation")


<a id="nb05-s12b"></a>

## §12b — Descriptive Statistical Charts

Seven charts summarising aggregation results across the spatial hierarchy:

| Chart | Type | X-axis | Stratification |
|-------|------|--------|----------------|
| **A** | Box plot | Area (km²) by Cluster25 | hot vs cold × day/night |
| **B** | Grouped bar | Population exposure by Cluster25 | day/night |
| **C** | Violin plot | Dense-core compactness (%) by Cluster25 | hot spots |
| **D** | Ranked horizontal bar | Hotspot area per city | BH-FDR flag overlay |
| **E** | Ranked horizontal bar | Coldspot area per city | BH-FDR flag overlay |
| **F** | Ranked horizontal bar | Hotspot area per Cluster25 | L90/L75/L50 hatch zones |
| **G** | Ranked horizontal bar | Coldspot area per Cluster25 | L90/L75/L50 hatch zones |

### Why Cluster25 as the stratification variable?

Cluster25 groups cities by their underlying physiographic and climatic landscape character (terrain, vegetation, rainfall). Using Cluster25 as the primary axis in §12b reveals whether thermal anomaly size and intensity are driven by landscape context (expected) or are idiosyncratic to individual cities (anomalous). Large inter-cluster differences validate the Cluster25 approach as a meaningful spatial stratification.

### Dense-core compactness (Chart C)

The dense-core percentage (area_90 / area_50 × 100) measures how concentrated the thermal anomaly is. A compact core (high %) suggests a discrete heat source (industrial site, major road junction); a diffuse pattern (low %) suggests extended impervious-surface heating. Violin plots show the full distribution, not just the mean, to reveal whether compactness is consistent across cities in each cluster or highly variable.

[↑ Table of Contents](#nb05-toc)


In [ ]:
if len(gdf) > 0:
    import matplotlib.patches as mpatches
    import matplotlib.cm as _cm12b
    # Publication minimum font sizes
    plt.rcParams.update({
        'font.size':       7.0,
        'axes.titlesize':  8.0,
        'axes.labelsize':  7.0,
        'xtick.labelsize': 6.5,
        'ytick.labelsize': 6.5,
        'legend.fontsize': 6.5,
        'figure.titlesize': 9.0,
    })

    # ── Cluster25 palette from reference (cluster25_reference_palette.csv) ────
    CLUST_CLR = CLUSTER25_COLORS
    _all_cl25 = sorted({int(c) for c in gdf['cluster25'].dropna().unique()})

    # ── Chart A: Area box plots by Cluster25 ──────────────────────────────────
    fig_a, axes_a = plt.subplots(1, 2, figsize=(11.69, 5))
    for ax, period in zip(axes_a, PERIODS):
        sub = gdf[gdf.period == period]
        clusters = sorted(sub.cluster25.unique())
        x = np.arange(len(clusters))
        d_hot  = [sub[(sub.cluster25==cl)&(sub.spot_type=='hot') ]['area_km2_50'].dropna().values for cl in clusters]
        d_cold = [sub[(sub.cluster25==cl)&(sub.spot_type=='cold')]['area_km2_50'].dropna().values for cl in clusters]
        kw_hot  = dict(patch_artist=True, widths=0.3,
                       boxprops=dict(facecolor='#FFDAB9',color='#E07B00'),
                       medianprops=dict(color='#E07B00',lw=2),
                       whiskerprops=dict(color='#E07B00'), capprops=dict(color='#E07B00'),
                       flierprops=dict(marker='o',ms=3,color='#E07B00',alpha=0.5))
        kw_cold = dict(patch_artist=True, widths=0.3,
                       boxprops=dict(facecolor='#B0C4DE',color='#4A6FA5'),
                       medianprops=dict(color='#4A6FA5',lw=2),
                       whiskerprops=dict(color='#4A6FA5'), capprops=dict(color='#4A6FA5'),
                       flierprops=dict(marker='o',ms=3,color='#4A6FA5',alpha=0.5))
        ax.boxplot(d_hot,  positions=x-0.18, **kw_hot)
        ax.boxplot(d_cold, positions=x+0.18, **kw_cold)
        ax.set_xticks(x); ax.set_xticklabels([f'C{c}' for c in clusters], fontsize=9)
        ax.set_title(f'{period.capitalize()} — Spot Area by Cluster25')
        ax.set_ylabel('area_km2_50'); ax.set_xlabel('Cluster25')
        ax.legend(handles=[mpatches.Patch(facecolor='#FFDAB9',edgecolor='#E07B00',label='Hot'),
                            mpatches.Patch(facecolor='#B0C4DE',edgecolor='#4A6FA5',label='Cold')],
                  fontsize=8)
    fig_a.suptitle('Spot Area Distribution by Cluster25', fontsize=11)
    plt.tight_layout()
    fig_a.savefig(FIGS_DIR / 'area_boxplot_cluster.svg', bbox_inches='tight', dpi=500)
    fig_a.savefig(FIGS_DIR / 'area_boxplot_cluster.pdf', bbox_inches='tight', dpi=500)
    plt.show()

    # ── Chart B: Population exposure grouped bars ──────────────────────────────
    _cl_pop = cl_tbl[cl_tbl.period != 'All periods'].copy() if 'cl_tbl' in dir() else pd.DataFrame()
    if len(_cl_pop):
        fig_b, axes_b = plt.subplots(1, 2, figsize=(11.69, 5))
        for ax, st in zip(axes_b, ('hot','cold')):
            sub  = _cl_pop[_cl_pop.spot_type == st]
            clus = sorted(sub.cluster25.unique())
            x = np.arange(len(clus)); w = 0.35
            for shift, period, clr in ((0,'day','#E07B00'),(w,'night','#4A6FA5')):
                vals = [sub[(sub.cluster25==cl)&(sub.period==period)]['total_pop'].values for cl in clus]
                vals = [v[0]/1000 if len(v) else 0 for v in vals]
                ax.bar(x+shift, vals, width=w*0.9, color=clr, alpha=0.85, label=period.capitalize())
            ax.set_xticks(x+w/2); ax.set_xticklabels([f'C{c}' for c in clus], fontsize=9)
            ax.set_title(f"{'Hot' if st=='hot' else 'Cold'} Spot Population")
            ax.set_ylabel('Population (thousands)'); ax.set_xlabel('Cluster25')
            ax.legend(fontsize=8)
        fig_b.suptitle('Population in Hotspot / Coldspot Zones by Cluster25', fontsize=11)
        plt.tight_layout()
        fig_b.savefig(FIGS_DIR / 'population_exposure_cluster.svg', bbox_inches='tight', dpi=500)
        fig_b.savefig(FIGS_DIR / 'population_exposure_cluster.pdf', bbox_inches='tight', dpi=500)
        plt.show()

    # ── Chart C: Dense-core compactness violin ─────────────────────────────────
    _core = gdf[(gdf.spot_type=='hot') & gdf['dense_core_pct'].notna()].copy()
    if len(_core):
        clusters = sorted(_core.cluster25.unique())
        fig_c, ax_c = plt.subplots(figsize=(8, 5))
        parts = ax_c.violinplot([_core[_core.cluster25==cl]['dense_core_pct'].values
                                  for cl in clusters],
                                 positions=range(len(clusters)),
                                 showmedians=True, showextrema=True)
        for i, (pc, cl) in enumerate(zip(parts['bodies'], clusters)):
            pc.set_facecolor(CLUST_CLR.get(int(cl), '#BAB0AC')); pc.set_alpha(0.7)
        parts['cmedians'].set_color('black'); parts['cmedians'].set_lw(2)
        ax_c.set_xticks(range(len(clusters)))
        ax_c.set_xticklabels([f'C{cl}' for cl in clusters], fontsize=9)
        ax_c.set_ylabel('Dense core %  (area_90 / area_50 x 100)')
        ax_c.set_title('Hotspot Core Compactness by Cluster25\n'
                       '(high % = tightly concentrated heat island)')
        ax_c.axhline(50, ls='--', color='grey', lw=0.8, label='50% line')
        ax_c.legend(fontsize=8)
        plt.tight_layout()
        fig_c.savefig(FIGS_DIR / 'dense_core_pct_violin.svg', bbox_inches='tight', dpi=500)
        fig_c.savefig(FIGS_DIR / 'dense_core_pct_violin.pdf', bbox_inches='tight', dpi=500)
        plt.show()

    # ── Shared helper: ranked horizontal bar chart (hot OR cold) ─────────────
    def _chart_ranked(spot_type_filter, out_name, chart_title):
        _bh_h = max(0.28, 5.5 / max(len(cities_with_sev), 1))
        fig_d, axes_d = plt.subplots(1, 2,
            figsize=(17, max(7, len(cities_with_sev)*_bh_h + 2.5)),
            sharey=False)

        for ax_d, period in zip(axes_d, PERIODS):
            _rows_d = []
            for city in cities_with_sev:
                sub = (gdf[(gdf.city==city) & (gdf.period==period) &
                           (gdf.spot_type==spot_type_filter)]
                       if len(gdf) else gpd.GeoDataFrame())
                a50 = float(sub['area_km2_50'].sum()) if len(sub) else 0.
                a75 = float(sub['area_km2_75'].fillna(0).sum()) if len(sub) else 0.
                a90 = float(sub['area_km2_90'].fillna(0).sum()) if len(sub) else 0.
                bh  = bool(sub['bh_significant'].any()) if len(sub) else False
                cl  = int(city_to_cluster.get(city, -1))
                _rows_d.append({'city': city, 'a50': a50, 'a75': a75, 'a90': a90,
                                'bh': bh, 'cluster': cl})

            # Exclude cities with no spots for this period
            _dat_d = (pd.DataFrame(_rows_d)
                      .query('a50 > 0')
                      .sort_values('a50', ascending=True))

            if _dat_d.empty:
                ax_d.set_title(f'{period.capitalize()} — no {spot_type_filter} spots')
                ax_d.axis('off')
                continue

            left = np.zeros(len(_dat_d))
            for seg_vals, htch, alpha_v, seg_lbl in [
                (_dat_d['a90'].values,                          'oo', 0.85, 'L90 dense core'),
                ((_dat_d['a75']-_dat_d['a90']).clip(0).values, 'xx', 0.70, 'L75 inner zone'),
                ((_dat_d['a50']-_dat_d['a75']).clip(0).values, '//', 0.55, 'L50 outer extent'),
            ]:
                bar_clrs = [CLUST_CLR.get(int(_dat_d.iloc[i]['cluster']), '#BAB0AC')
                            for i in range(len(_dat_d))]
                ax_d.barh(range(len(_dat_d)), seg_vals, left=left,
                          color=bar_clrs, hatch=htch, alpha=alpha_v,
                          edgecolor='black', lw=0.4, label=seg_lbl)
                left += seg_vals

            ylabels = [f'{r.city}{"*" if r.bh else ""}' for _, r in _dat_d.iterrows()]
            ax_d.set_yticks(range(len(_dat_d)))
            ax_d.set_yticklabels(ylabels, fontsize=9)
            ax_d.set_xlabel('Area (km²)', fontsize=9)
            ax_d.set_title(f'{period.capitalize()} — {spot_type_filter.capitalize()}spot Area\n'
                           f'(* = BH-FDR  |  colour = Cluster25  |  hatch = contour zone)',
                           fontsize=9, fontweight='bold')

        # Legend
        cl_handles = [mpatches.Patch(color=CLUST_CLR.get(cl, '#BAB0AC'),
                                     label=f'C{cl}', edgecolor='black', lw=0.5)
                      for cl in sorted(_all_cl25)]
        hatch_handles = [
            mpatches.Patch(facecolor='#AAAAAA', hatch='oo', edgecolor='black', label='L90 dense core'),
            mpatches.Patch(facecolor='#AAAAAA', hatch='xx', edgecolor='black', label='L75 inner zone'),
            mpatches.Patch(facecolor='#AAAAAA', hatch='//', edgecolor='black', label='L50 outer extent'),
        ]
        fig_d.legend(handles=cl_handles + [mpatches.Patch(visible=False)] + hatch_handles,
                     title='Cluster25  |  Contour zone', fontsize=7.0, title_fontsize=8,
                     loc='lower center', ncol=min(len(cl_handles)+4, 10),
                     bbox_to_anchor=(0.5, -0.02), framealpha=0.9)
        fig_d.suptitle(chart_title, fontsize=11, fontweight='bold')
        plt.tight_layout(rect=[0, 0.07, 1, 1])
        fig_d.savefig(FIGS_DIR / out_name, bbox_inches='tight')
        plt.show()
        print(f'  Saved: {out_name}')

    # ── Chart D: Hotspot ranked area ──────────────────────────────────────────
    _chart_ranked(
        'hot', 'city_hotspot_area_ranked.svg',
        'Hotspot Area per City — Day & Night  (cities with spots only)\n'
        '* = BH-FDR  |  colour = Cluster25  |  hatch = contour zone'
    )
    # ── Chart E: Coldspot ranked area ─────────────────────────────────────────
    _chart_ranked(
        'cold', 'city_coldspot_area_ranked.svg',
        'Coldspot Area per City — Day & Night  (cities with spots only)\n'
        '* = BH-FDR  |  colour = Cluster25  |  hatch = contour zone'
    )
    # ── Chart F/G: Cluster-level ranked area ──────────────────────────────────
    def _chart_ranked_cluster(spot_type_filter, out_name, chart_title):
        _bh_h = max(0.35, 5.5 / max(len(CLUST_CLR), 1))
        fig_cl, axes_cl = plt.subplots(1, 2,
            figsize=(17, max(6, len(CLUST_CLR) * _bh_h + 2.5)),
            sharey=False)

        for ax_cl, period in zip(axes_cl, PERIODS):
            _rows_cl = {}
            for city in cities_with_sev:
                cl = int(city_to_cluster.get(city, -1))
                sub = (gdf[(gdf.city==city) & (gdf.period==period) &
                           (gdf.spot_type==spot_type_filter)]
                       if len(gdf) else gpd.GeoDataFrame())
                a50 = float(sub['area_km2_50'].sum()) if len(sub) else 0.
                a75 = float(sub['area_km2_75'].fillna(0).sum()) if len(sub) else 0.
                a90 = float(sub['area_km2_90'].fillna(0).sum()) if len(sub) else 0.
                if cl not in _rows_cl:
                    _rows_cl[cl] = {'a50': 0., 'a75': 0., 'a90': 0., 'n_cities': 0}
                _rows_cl[cl]['a50'] += a50
                _rows_cl[cl]['a75'] += a75
                _rows_cl[cl]['a90'] += a90
                if a50 > 0:
                    _rows_cl[cl]['n_cities'] += 1

            _dat_cl = (pd.DataFrame([
                    {'cluster': cl, 'a50': v['a50'], 'a75': v['a75'],
                     'a90': v['a90'], 'n_cities': v['n_cities']}
                    for cl, v in _rows_cl.items()
                ])
                .query('a50 > 0')
                .sort_values('a50', ascending=True))

            if _dat_cl.empty:
                ax_cl.set_title(f'{period.capitalize()} — no {spot_type_filter} spots')
                ax_cl.axis('off')
                continue

            left = np.zeros(len(_dat_cl))
            for seg_vals, htch, alpha_v, seg_lbl in [
                (_dat_cl['a90'].values,                           'oo', 0.85, 'L90 dense core'),
                ((_dat_cl['a75']-_dat_cl['a90']).clip(0).values,  'xx', 0.70, 'L75 inner zone'),
                ((_dat_cl['a50']-_dat_cl['a75']).clip(0).values,  '//', 0.55, 'L50 outer extent'),
            ]:
                bar_clrs = [CLUST_CLR.get(int(_dat_cl.iloc[i]['cluster']), '#BAB0AC')
                            for i in range(len(_dat_cl))]
                ax_cl.barh(range(len(_dat_cl)), seg_vals, left=left,
                           color=bar_clrs, hatch=htch, alpha=alpha_v,
                           edgecolor='black', lw=0.4, label=seg_lbl)
                left += seg_vals

            ylabels = [f"C{int(r.cluster)} ({int(r.n_cities)} cit{'y' if r.n_cities==1 else 'ies'})"
                       for _, r in _dat_cl.iterrows()]
            ax_cl.set_yticks(range(len(_dat_cl)))
            ax_cl.set_yticklabels(ylabels, fontsize=9.5)
            ax_cl.set_xlabel('Area (km²)', fontsize=9)
            ax_cl.set_title(f'{period.capitalize()} — {spot_type_filter.capitalize()}spot Area\n'
                            f'by Cluster25  (sorted by total area)',
                            fontsize=10, fontweight='bold')
            ax_cl.grid(axis='x', alpha=0.25, lw=0.5)

        # Shared legend
        import matplotlib.patches as _mp12cl
        hatch_h = [
            _mp12cl.Patch(facecolor='#AAAAAA', hatch='oo', edgecolor='black', label='L90 dense core'),
            _mp12cl.Patch(facecolor='#AAAAAA', hatch='xx', edgecolor='black', label='L75 inner zone'),
            _mp12cl.Patch(facecolor='#AAAAAA', hatch='//', edgecolor='black', label='L50 outer extent'),
        ]
        cl_h = [_mp12cl.Patch(color=CLUST_CLR.get(cl, '#BAB0AC'),
                               label=f'C{cl}', edgecolor='black', lw=0.4)
                for cl in sorted(CLUST_CLR.keys())]
        fig_cl.legend(handles=hatch_h + cl_h,
                      title='Contour zone  |  Cluster25',
                      fontsize=7.5, title_fontsize=8.5,
                      loc='lower center', ncol=min(len(hatch_h) + len(cl_h), 12),
                      bbox_to_anchor=(0.5, -0.02), framealpha=0.92)
        fig_cl.suptitle(chart_title, fontsize=11, fontweight='bold')
        plt.tight_layout(rect=[0, 0.07, 1, 1])
        fig_cl.savefig(FIGS_DIR / out_name, bbox_inches='tight', dpi=500)
        fig_cl.savefig(FIGS_DIR / out_name.replace('.svg', '.pdf'), bbox_inches='tight', dpi=500)
        plt.show()
        print(f'  §12b cluster chart saved: {out_name}')

    _chart_ranked_cluster(
        'hot', 'cluster_hotspot_area_ranked.svg',
        'Hotspot Area per Cluster25 — Day & Night  (clusters with spots only)\n'
        'colour = Cluster25  |  hatch = contour zone'
    )
    _chart_ranked_cluster(
        'cold', 'cluster_coldspot_area_ranked.svg',
        'Coldspot Area per Cluster25 — Day & Night  (clusters with spots only)\n'
        'colour = Cluster25  |  hatch = contour zone'
    )

else:
    print("GDF empty — skipping §12b charts")


<a id="nb05-s12c"></a>

## §12c — Publication Figure: Hotspot Core Compactness (≥3 cities)

A journal-ready, single-column (≈90 mm) version of Chart C, restricted to
**Cluster25 groups with ≥3 cities** — the minimum for a violin to describe a
distribution rather than a point/line. All text is ≥8 pt; violins, spacing and
figure are proportionally reduced but kept legible at print size.

**Outputs:** `dense_core_pct_violin_pub.svg` + `dense_core_pct_violin_pub.pdf`
(the PDF is written with an explicit `format='pdf'` so it is a true vector PDF,
not the SVG-in-`.pdf` produced by the default `svg_only` export hook).

[↑ Table of Contents](#nb05-toc)

In [ ]:
# ── §12c — Compact publication violin: Hotspot Core Compactness (≥3 cities) ───
# Journal-ready single-column figure. Only Cluster25 groups with ≥3 cities are
# shown (a violin needs several cities to describe a distribution). All text ≥8 pt.
if len(gdf) > 0:
    _corep = gdf[(gdf.spot_type == 'hot') & gdf['dense_core_pct'].notna()].copy()
    _corep['cluster25'] = _corep['cluster25'].astype(int)
    _ncity_d = {int(k): int(v) for k, v in
                _corep.groupby('cluster25')['city'].nunique().items()}
    _keep = sorted([c for c, n in _ncity_d.items() if n >= 3])
    _cp = _corep[_corep.cluster25.isin(_keep)]

    with plt.rc_context({
        'font.size': 8, 'axes.titlesize': 9, 'axes.labelsize': 8,
        'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
        'axes.linewidth': 0.6, 'pdf.fonttype': 42, 'ps.fonttype': 42,
        'svg.fonttype': 'none',
    }):
        fig_cpub, ax = plt.subplots(figsize=(3.54, 2.7))   # ~90 mm single column
        _data = [_cp[_cp.cluster25 == cl]['dense_core_pct'].values for cl in _keep]
        parts = ax.violinplot(_data, positions=range(len(_keep)),
                              widths=0.78, showmedians=True, showextrema=True)
        for pc, cl in zip(parts['bodies'], _keep):
            pc.set_facecolor(CLUSTER25_COLORS.get(int(cl), '#BAB0AC'))
            pc.set_alpha(0.78); pc.set_edgecolor('black'); pc.set_linewidth(0.4)
        for _k in ('cbars', 'cmins', 'cmaxes'):
            parts[_k].set_color('#555555'); parts[_k].set_linewidth(0.6)
        parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(1.3)

        ax.axhline(50, ls='--', color='grey', lw=0.7)
        ax.text(len(_keep) - 0.55, 50.5, '50%', fontsize=8, color='grey',
                ha='right', va='bottom')

        ax.set_xticks(range(len(_keep)))
        ax.set_xticklabels([f'C{cl}\n(n={_ncity_d[cl]})' for cl in _keep])
        ax.set_ylabel('Dense core (%)\n(area$_{90}$/area$_{50}$ × 100)')
        ax.set_xlabel('Cluster25')
        ax.set_title('Hotspot Core Compactness by Cluster25\n'
                     '(clusters with ≥3 cities)')
        ax.margins(x=0.07)
        ax.tick_params(length=2.5, width=0.6)
        for _sp in ('top', 'right'):
            ax.spines[_sp].set_visible(False)
        fig_cpub.tight_layout(pad=0.4)

        _out = FIGS_DIR / 'dense_core_pct_violin_pub'
        # explicit format= bypasses the svg_only export hook so the PDF is a real PDF
        fig_cpub.savefig(_out.with_suffix('.svg'), format='svg', bbox_inches='tight')
        fig_cpub.savefig(_out.with_suffix('.pdf'), format='pdf', bbox_inches='tight')
        plt.show()
        print(f'  Clusters kept (≥3 cities): {["C%d" % c for c in _keep]}')
        print(f'  ✓ Saved {_out.name}.svg / .pdf → {FIGS_DIR}')
else:
    print('GDF empty — skipping compact violin')


<a id="nb05-s13"></a>

## §13 — Summary and carry-forward to NB06

### Completion status

| Step | Status | Key output |
|------|--------|-----------|
| §1 Setup, version control, cache helpers | ✅ | `SOURCE_VERSION`, `FORCE_RERUN` |
| §2 Data loading (zones, SEVIRI, SUHII, stakeholder ABOVE_300K, population) | ✅ | `zones_dict`, `hotspot_results`, `suhii_lookup` |
| §3 P90/P10 thresholds (city-specific, Day/Night) | ✅ | `thresholds` dict |
| §4 Gi★ + BH-FDR (Queen contiguity, `esda.G_Local`; Z≥1.96 primary, BH stored) | ✅ | `compute_gistar()` |
| §5 Validated masks with NPZ cache | ✅ | `validated_masks`, `validated_*_bh` |
| §6 Two-Tier KDE (Scott bandwidth, wide-format, §6d proximity union) | ✅ | `kde_records_all`, `spot_comprehensive_summary.csv` (§6c) |
| §7 Per-spot layer and enrichment | ✅ | `gdf`, `all_cities_spots.geojson` (+ `.csv`, `.meta.json`: §7's own cache) |
| §8 Distribution assessment | — | not performed in this notebook (see §8) |
| §9 Stakeholder hit rates and buffer sensitivity | ✅ | `HITRATE_NB05` (no file); charts and national / cluster / city tables |
| §10 NB05 working GeoPackage, per contour level | ✅ | `master_thermal_spots.gpkg` (4 layers) |
| §11 Visualisations (a, a2, b, c, d, f) + §11d half-donut tables | ✅ | figures; `halfdonut_city_summary.csv`, `halfdonut_cluster_summary.csv` |
| §12 Aggregation (+ §12b, §12c charts) | ✅ | `cluster_hotspot_summary.csv`, `national_hotspot_summary.csv` |
| §14 Build the dataset (14a build, 14b summary) | ✅ | the published dataset, via `scripts/build_master_summary.py` |

### What reads NB05's files

| File | Written by | Read by |
|------|-----------|---------|
| `all_cities_spots.geojson` | §7 | dataset build, NB06, §9 |
| `all_cities_spots.csv`, `all_cities_spots.meta.json` | §7 | §7 (its cache); the CSV also by the dataset QA |
| `master_thermal_spots.gpkg` | §10 | dataset build, `scripts/extract_population_100m.py`, §9 when run on its own |
| `spot_comprehensive_summary.csv` | §6c | dataset build, dataset QA, Notebook 07 |
| `halfdonut_city_summary.csv` | §11d | dataset build, Notebook 07 |
| `halfdonut_cluster_summary.csv` | §11d | nothing |
| `cluster_hotspot_summary.csv`, `national_hotspot_summary.csv` | §12 | Notebook 07 (Table 2) |
| `master_city_composite.csv` | no cell of this notebook (a file from May 2026) | nothing: the build derives `validation_tier` and `bh_any` itself; moved to `Results/_to_delete/nb05_unused/` |

### NB06 carry-forward

NB06 reads one NB05 output:

```python
import geopandas as gpd
gdf = gpd.read_file('Results/05_hotspot_analysis/all_cities_spots.geojson')   # one row per spot (L50)
```

### Output tree

```
Results/05_hotspot_analysis/
├── all_cities_spots.geojson / .csv / .meta.json   §7   per-spot layer (+ cache)
├── master_thermal_spots.gpkg                      §10  NB05 working GeoPackage, not the dataset
├── spot_comprehensive_summary.csv                 §6c
├── halfdonut_city_summary.csv / _cluster_         §11d
├── cluster_hotspot_summary.csv                    §12
├── national_hotspot_summary.csv                   §12
├── population_100m/                               scripts/extract_population_100m.py
├── figures/                                       §11, §12b, §12c
├── surface_thermal_anomaly_stakeholder_composite/ §14  the published dataset (deposit)
├── dataset_build/                                 §14  build report, changes reports
├── _superseded_build_<date>/                      §14  a deposit replaced by an update
└── _build_staging/                                §14  every build lands here first
```

[↑ Table of Contents](#nb05-toc)

<a id="nb05-s14"></a>

## §14 Dataset Build

This section and [§10](#nb05-s10) do different jobs: §10 writes NB05's own working file; §14
assembles the **published dataset** (`surface_thermal_anomaly_stakeholder_composite`) from the
outputs of NB01–NB05 with `scripts/build_master_summary.py`. That script holds the only copy of
the build logic. This section calls it exactly as the command line does
(`python scripts/build_master_summary.py`), so the two routes always build the same thing.

**How a build is decided.** Every build is written to
`Results/05_hotspot_analysis/_build_staging/` first and compared with the deposit by its
**content fingerprint**: a SHA-256 over every layer, the scenario configuration and the data
dictionary, excluding the build date, commit and checksums. MD5s still verify a download; the
fingerprint decides whether two builds hold the same data, because a GeoPackage can differ
byte-wise while its content is identical.

| Case | Meaning | What happens |
|---|---|---|
| **Same fingerprint** | Nothing changed. | *"Dataset reproduced (fingerprint identical)"*. Nothing is touched and staging is cleared. |
| **Different content** (values, or a new `scenario_id`) | An input, the code or a setting changed. | QA runs on the staging build. **If it passes**, the current deposit is kept whole in `_superseded_build_<date>/` and replaced by the staging build with all its files: GeoPackage, data dictionary (CSV and MD), README, CHANGELOG, LICENSE and the checksums, written last. **If QA fails**, the deposit is not touched and staging is kept for inspection. |

No version is named here: versions are named on Zenodo only. A deposit is identified by its
content fingerprint; its CHANGELOG entry is headed by date and fingerprint and says which earlier
deposit it is compared with. The field-by-field record of each update is
`dataset_build/CHANGES_REPORT_<date>.md`. Before anything is built, a pre-flight check confirms
that every input exists and names the notebook or script that produces any missing one, so a
partial dataset is never built.

[↑ Table of Contents](#nb05-toc)

<a id="nb05-s14a"></a>

### 14a — Build

Calls `build_dataset()`: staging, the fingerprint decision, and, when the content differs, QA on the staging build, which must pass before the deposit is updated.

In [ ]:
# ── §14a — Build ──────────────────────────────────────────────────────────────
# One build code, two entry points: this cell and `python scripts/build_master_summary.py`
# both call build_dataset(). None of the build logic lives in this notebook.
import sys
import pandas as pd
from pathlib import Path
from IPython.display import display

_ROOT14 = Path.cwd() if (Path.cwd() / 'scripts').is_dir() else Path.cwd().parent
if str(_ROOT14) not in sys.path:
    sys.path.insert(0, str(_ROOT14))
DEPOSIT_DIR = (_ROOT14 / 'Results' / '05_hotspot_analysis'
               / 'surface_thermal_anomaly_stakeholder_composite')

BUILD_DATASET = True        # set False to skip
import importlib
for _m in ('build_master_summary', 'qa_master_summary',
           'scripts.build_master_summary', 'scripts.qa_master_summary'):
    if _m in sys.modules:                  # a live kernel picks up edits to the scripts
        importlib.reload(sys.modules[_m])
from scripts.build_master_summary import build_dataset
report = build_dataset(out_dir=DEPOSIT_DIR) if BUILD_DATASET else None
QA_RESULT = report.qa if report is not None and report.qa else None
if report is not None:
    display(report.summary())   # case, fingerprints, QA, update, changes
    print(report.message)
assert QA_RESULT is None or QA_RESULT['ok'], \
    f"QA failed on the staging build ({QA_RESULT['passed']}/{QA_RESULT['total']}): deposit not touched"


<a id="nb05-s14b"></a>

### 14b — Summary of the whole dataset

Read from the deposit after the build; nothing is written. (1) identity, (2) one row per layer, (3) key counts, each asserted, (4) the published hit rates against §9, (5) changes in this build relative to the deposit, (6) the MD5 table.

[↑ Table of Contents](#nb05-toc)

In [ ]:
# ── §14b — Summary of the whole dataset ──────────────────────────────────────
# Read from the deposit after the build; nothing is written. Every count is asserted.
from scripts.build_master_summary import summarise_package

S14 = summarise_package(DEPOSIT_DIR)
_rep14 = globals().get('report')
_qa14 = globals().get('QA_RESULT')

# 1. Identity
_id14 = S14['identity'].copy()
_id14.loc[len(_id14)] = ['decision taken by this build', _rep14.case if _rep14 else 'build skipped']
_id14.loc[len(_id14)] = ['this build: fingerprint',
                         f"sha256:{_rep14.fingerprint}" if _rep14 else '-']
_id14.loc[len(_id14)] = ['QA on the staging build',
                         (f"{_qa14['passed']}/{_qa14['total']} checks passed" if _qa14
                          else 'not run: the content did not change')]
_id14.loc[len(_id14)] = ['deposit updated by this build',
                         ('yes; the previous deposit is kept in '
                          + str(_rep14.superseded_to) if _rep14 and _rep14.deposit_changed
                          else 'no')]
print("1. Identity (the deposit)")
display(_id14)

# 2. One row per layer
print("2. Layers")
display(S14['layers'])
assert len(S14['layers']) == 18, f"{len(S14['layers'])} layers, expected 18"

# 3. Key counts, asserted
_c14 = S14['counts']
_expect14 = {'cities': 56, 'clusters': 16, 'candidates': 67, 'absorbed': 11,
             'spots': 185, 'contours': 441, 'heat_events': 807,
             'participants': 105, 'points': 94, 'points_hot': 55, 'points_cold': 39,
             'composite_trend_rows': 136}
print("3. Key counts")
for _k, _v in _expect14.items():
    assert _c14[_k] == _v, f"{_k}: {_c14[_k]} in the deposit, expected {_v}"
    print(f"  [OK] {_k:22} {_c14[_k]}")

# 4. Hit-rate check: the published layer equals §9's result, row for row
if 'HITRATE_NB05' not in globals():
    # §9 not run in this session: recompute with exactly the inputs §9 uses
    from scripts.build_master_summary import prepare_cohort_points, prepare_spot_footprints
    from scripts.compute_hitrate import hit_rate_table
    HITRATE_NB05 = hit_rate_table(prepare_cohort_points(), prepare_spot_footprints())
    print("   (§9 was not run in this session; HITRATE_NB05 recomputed with the same inputs)")
_pub14 = S14['frames']['hitrate_sensitivity']
_k14 = ['city', 'spot_type', 'period', 'buffer_m']
_m14 = _pub14.merge(HITRATE_NB05, on=_k14, how='outer', suffixes=('_pub', '_nb05'),
                    indicator=True)
_nd14 = int((_m14['_merge'] != 'both').sum()
            + (_m14['n_hit_pub'] != _m14['n_hit_nb05']).sum()
            + (_m14['n_total_pub'] != _m14['n_total_nb05']).sum()
            + ((_m14['hit_rate_pub'] - _m14['hit_rate_nb05']).abs() > 1e-12).sum())
assert len(_pub14) == len(HITRATE_NB05) == 1160 and _nd14 == 0, \
    f"published {len(_pub14)} rows, §9 {len(HITRATE_NB05)} rows, {_nd14} differences"
print(f"4. Hit rates: published hitrate_sensitivity equals HITRATE_NB05 from §9 "
      f"({len(_pub14):,} rows, {_nd14} differences)")

# 5. Changes in this build relative to the deposit
print("5. Changes in this build against the deposit it replaces (field by field: "
      + (str(_rep14.changes_report) if _rep14 and _rep14.changes_report else "none written") + ")")
if _rep14 and _rep14.changes_table:
    display(pd.DataFrame(_rep14.changes_table))
    for _l in _rep14.data_changes:
        print("   data:     " + _l[2:])
    for _l in _rep14.metadata_changes:
        print("   metadata: " + _l[2:][:160])
else:
    print("   none")

# 6. MD5 table of the deposit files
print("6. Deposit files")
display(S14['md5'])
assert S14['md5'][S14['md5'].file != 'checksums_md5.txt']['matches manifest'].all(), \
    "a deposit file does not match checksums_md5.txt"